# Sprint04 Evaluation Notebook
### Stage 1: Environment, Configuration & Artifact/Dataset Discovery

Read-only evaluation of the Sprint04-trained model. This notebook never
modifies training code or re-runs training logic. All artifacts are
consumed exactly as a production inference system would.

In [2]:
# =============================================================================
# Sprint04_Evaluation.ipynb
# STAGE 1 — Environment, Configuration, Logging, Seeds, Device, Discovery
# -----------------------------------------------------------------------------
# READ-ONLY notebook. Never recreates training/optimizer/scheduler/AMP/
# checkpoint managers. This stage only prepares the environment and locates
# the three mounted Kaggle datasets. Deep artifact file validation happens
# in Stage 2.
# =============================================================================

import os
import sys
import json
import time
import random
import logging
import platform
import fnmatch
import importlib.metadata as importlib_metadata
from pathlib import Path
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
from typing import Optional, Dict, List, Tuple

# -----------------------------------------------------------------------------
# 1.1  Third-party imports — fail loudly and descriptively, never silently
# -----------------------------------------------------------------------------
_IMPORT_ERRORS: List[str] = []

def _require(module_name: str):
    try:
        return __import__(module_name)
    except ImportError:
        _IMPORT_ERRORS.append(module_name)
        return None

np = _require("numpy")
torch = _require("torch")
pd = _require("pandas")
sklearn = _require("sklearn")
matplotlib = _require("matplotlib")
PIL = _require("PIL")
timm = _require("timm")          # needed later to reconstruct the backbone
torchvision = _require("torchvision")

if _IMPORT_ERRORS:
    raise ImportError(
        "Stage 1 aborted — required packages are missing: "
        f"{_IMPORT_ERRORS}. Install them before proceeding "
        "(e.g. `pip install torch torchvision timm scikit-learn`)."
    )


# =============================================================================
# 1.2  Configuration
# =============================================================================

@dataclass(frozen=True)
class EvalConfig:
    notebook_name: str = "Sprint04_Evaluation.ipynb"
    run_id: str = field(default_factory=lambda: datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
    random_seed: int = 42
    kaggle_input_root: Path = Path("/kaggle/input/datasets")
    kaggle_working_root: Path = Path("/kaggle/working")
    local_input_env_var: str = "EVAL_INPUT_ROOT"
    local_output_env_var: str = "EVAL_OUTPUT_ROOT"
    output_dirname: str = "sprint04_evaluation"

    @property
    def is_kaggle(self) -> bool:
        return self.kaggle_input_root.is_dir()

    def resolve_input_root(self) -> Path:
        if self.is_kaggle:
            return self.kaggle_input_root
        env_path = os.environ.get(self.local_input_env_var)
        if env_path and Path(env_path).is_dir():
            return Path(env_path)
        fallback = Path.cwd() / "kaggle_input"
        if fallback.is_dir():
            return fallback
        raise FileNotFoundError(
            "No Kaggle input root found at /kaggle/input, and no local "
            f"fallback found. Set the {self.local_input_env_var} environment "
            "variable to the directory containing the three mounted datasets, "
            f"or create {fallback}."
        )

    def resolve_output_root(self) -> Path:
        if self.is_kaggle:
            root = self.kaggle_working_root / self.output_dirname
        else:
            env_path = os.environ.get(self.local_output_env_var)
            root = Path(env_path) if env_path else Path.cwd() / self.output_dirname
        root.mkdir(parents=True, exist_ok=True)
        return root


CFG = EvalConfig()
INPUT_ROOT = CFG.resolve_input_root()
OUTPUT_ROOT = CFG.resolve_output_root()
STAGE_DIR = OUTPUT_ROOT / "stage01_environment"
STAGE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = OUTPUT_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# 1.3  Logging
# =============================================================================

def setup_logging(log_dir: Path, stage_name: str) -> logging.Logger:
    logger = logging.getLogger(f"sprint04_eval.{stage_name}")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False

    fmt = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    console = logging.StreamHandler(sys.stdout)
    console.setFormatter(fmt)
    logger.addHandler(console)

    file_handler = logging.FileHandler(log_dir / f"{stage_name}.log", mode="a")
    file_handler.setFormatter(fmt)
    logger.addHandler(file_handler)

    return logger


logger = setup_logging(LOG_DIR, "stage01_environment")


class StageTimer:
    """Reusable context manager for start/finish/duration logging.
    Reused as-is in Stages 2-8 for consistent, comparable stage logs."""

    def __init__(self, stage_name: str, log: logging.Logger):
        self.stage_name = stage_name
        self.log = log
        self.t0: Optional[float] = None
        self.warnings: List[str] = []

    def warn(self, message: str) -> None:
        self.warnings.append(message)
        self.log.warning(message)

    def __enter__(self):
        self.t0 = time.perf_counter()
        self.log.info(f"START  stage={self.stage_name}")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        duration = time.perf_counter() - self.t0
        status = "FAILED" if exc_type else "FINISH"
        self.log.info(
            f"{status} stage={self.stage_name} duration_sec={duration:.2f} "
            f"warnings={len(self.warnings)}"
        )
        return False  # never swallow exceptions


# =============================================================================
# 1.4  Reproducibility — random seeds
# =============================================================================

def set_global_seed(seed: int) -> Dict[str, bool]:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    cuda_seeded = False
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        cuda_seeded = True
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    return {"cuda_seeded": cuda_seeded, "cudnn_deterministic": True, "cudnn_benchmark": False}


# =============================================================================
# 1.5  Device selection
# =============================================================================

def select_device() -> Tuple["torch.device", Dict[str, object]]:
    info: Dict[str, object] = {}
    if torch.cuda.is_available():
        device = torch.device("cuda")
        idx = torch.cuda.current_device()
        props = torch.cuda.get_device_properties(idx)
        info.update({
            "backend": "cuda",
            "device_name": props.name,
            "total_memory_gb": round(props.total_memory / (1024 ** 3), 2),
            "cuda_capability": f"{props.major}.{props.minor}",
            "cuda_version": torch.version.cuda,
            "cudnn_version": torch.backends.cudnn.version(),
        })
    elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        device = torch.device("mps")
        info.update({"backend": "mps", "device_name": "Apple MPS"})
    else:
        device = torch.device("cpu")
        info.update({"backend": "cpu", "device_name": platform.processor() or "unknown-cpu"})
    return device, info


# =============================================================================
# STAGE 1 — PATCH: Section 1.6 replacement
# Fixes: Kaggle mounts datasets by OWNER, not one-folder-per-dataset.
# anupsharma1730/ contains BOTH visionserveai-sprint04-artifacts/ and
# visionserveai-training-artifacts-v1/ as siblings. organizations/ contains
# nih-chest-xrays/ nested by org slug. Discovery must therefore assign a
# role to whichever directory contains the anchor, not to the top-level
# owner folder it happens to sit under.
# =============================================================================

import re

# Directories we must never descend into during discovery — these hold
# thousands of PNGs each and contribute nothing to role discovery.
_HEAVY_DIR_PATTERN = re.compile(r"^images(_\d+)?$", re.IGNORECASE)


def _bounded_walk(root: Path, max_depth: int):
    """os.walk with depth pruning AND heavy-directory pruning, so
    discovery never touches the NIH image payload itself."""
    root = root.resolve()
    base_depth = len(root.parts)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = len(Path(dirpath).parts) - base_depth
        # Prune traversal into large image-payload directories.
        dirnames[:] = [d for d in dirnames if not _HEAVY_DIR_PATTERN.match(d)]
        if depth >= max_depth:
            dirnames[:] = []
        yield Path(dirpath), dirnames, filenames


def discover_datasets(input_root: Path, log: logging.Logger, max_depth: int = 8) -> Dict[str, Dict[str, str]]:
    """
    Role-anchored discovery. Each role is located independently by the
    directory that actually contains its anchor, not by which owner-level
    folder (e.g. 'anupsharma1730', 'organizations') it happens to be
    nested under. This correctly handles Kaggle mounting multiple
    same-owner datasets under one shared parent folder.
    """
    if not input_root.is_dir():
        raise FileNotFoundError(f"Input root does not exist: {input_root}")

    found: Dict[str, Path] = {}

    def _claim(role: str, dataset_root: Path):
        if role in found and found[role] != dataset_root:
            raise RuntimeError(
                f"Ambiguous dataset discovery for role={role!r}: both "
                f"{found[role]} and {dataset_root} match. Discovery "
                "signatures need tightening."
            )
        found[role] = dataset_root

    for dirpath, dirnames, filenames in _bounded_walk(input_root, max_depth):
        # --- artifacts: .../visionserveai/sprint04/ -----------------------
        if dirpath.name == "sprint04" and dirpath.parent.name == "visionserveai":
            if (dirpath / "training_summary.json").is_file():
                _claim("artifacts", dirpath.parent.parent)

        # --- registry_manifests: .../visionserveai/sprint03/ --------------
        if dirpath.name == "sprint03" and dirpath.parent.name == "visionserveai":
            has_registry = any(fnmatch.fnmatch(f, "disease_registry*.json") for f in filenames) or \
                            (dirpath / "registry").is_dir()
            has_manifest = any(fnmatch.fnmatch(f, "test_manifest*.csv") for f in filenames) or \
                            (dirpath / "manifests").is_dir()
            if has_registry or has_manifest:
                _claim("registry_manifests", dirpath.parent.parent)

        # --- images: dir containing Data_Entry_2017.csv --------------------
        if "Data_Entry_2017.csv" in filenames:
            _claim("images", dirpath)

        if {"artifacts", "registry_manifests", "images"} <= found.keys():
            break  # early exit — no need to keep walking once all 3 are found

    for role, path in found.items():
        try:
            rel = path.relative_to(input_root)
        except ValueError:
            rel = path
        log.info(f"dataset_found role={role} path={rel}")

    required_roles = {"artifacts", "registry_manifests", "images"}
    missing = required_roles - found.keys()
    if missing:
        raise FileNotFoundError(
            f"Could not locate required dataset role(s): {sorted(missing)}. "
            f"Roles found so far: { {k: str(v) for k, v in found.items()} }. "
            "Check that all three Kaggle datasets are attached to this notebook."
        )

    return {
        role: {"top_level_name": str(path.relative_to(input_root)), "path": str(path)}
        for role, path in found.items()
    }

# =============================================================================
# 1.7  Version reporting
# =============================================================================

def collect_versions() -> Dict[str, str]:
    def _v(pkg_module, dist_name):
        try:
            return getattr(pkg_module, "__version__", None) or importlib_metadata.version(dist_name)
        except Exception:
            return "unknown"

    return {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": _v(torch, "torch"),
        "torchvision": _v(torchvision, "torchvision"),
        "numpy": _v(np, "numpy"),
        "pandas": _v(pd, "pandas"),
        "scikit_learn": _v(sklearn, "scikit-learn"),
        "matplotlib": _v(matplotlib, "matplotlib"),
        "pillow": _v(PIL, "pillow"),
        "timm": _v(timm, "timm"),
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
    }


# =============================================================================
# 1.8  Requirements validation
# =============================================================================

REQUIRED_HARD = ["torch", "torchvision", "numpy", "pandas", "sklearn", "matplotlib", "PIL", "timm"]
SOFT_RECOMMENDED = ["pyarrow"]  # needed for the .parquet manifest variants

import importlib

def validate_requirements(log):

    hard_missing = []

    for module in REQUIRED_HARD:

        try:
            importlib.import_module(module)

        except ImportError:

            hard_missing.append(module)

    if hard_missing:

        raise ImportError(
            f"Hard requirement(s) missing: {hard_missing}"
        )

    soft_status = {}

    for module in SOFT_RECOMMENDED:

        try:
            importlib.import_module(module)
            soft_status[module] = "present"

        except ImportError:

            soft_status[module] = "missing"

            log.warning(
                f"Optional dependency {module} not installed."
            )

    return {

        "hard_required": REQUIRED_HARD,

        "hard_missing": hard_missing,

        "soft_recommended": soft_status,

    }

# =============================================================================
# 1.9  Run Stage 1
# =============================================================================

def run_stage1() -> Dict[str, object]:
    with StageTimer("stage01_environment", logger) as timer:
        seed_info = set_global_seed(CFG.random_seed)
        logger.info(f"seed_set seed={CFG.random_seed} info={seed_info}")

        device, device_info = select_device()
        logger.info(f"device_selected={device} info={device_info}")

        req_report = validate_requirements(logger)
        logger.info(f"requirements_validated hard_ok=True soft={req_report['soft_recommended']}")

        datasets = discover_datasets(INPUT_ROOT, logger)
        logger.info(f"datasets_discovered={ {k: v['top_level_name'] for k, v in datasets.items()} }")

        versions = collect_versions()

        summary = {
            "notebook": CFG.notebook_name,
            "run_id": CFG.run_id,
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "execution_mode": "kaggle" if CFG.is_kaggle else "local",
            "paths": {
                "input_root": str(INPUT_ROOT),
                "output_root": str(OUTPUT_ROOT),
            },
            "reproducibility": {"random_seed": CFG.random_seed, **seed_info},
            "device": {"selected": str(device), **device_info},
            "versions": versions,
            "requirements_validation": req_report,
            "dataset_discovery": datasets,
            "warnings": timer.warnings,
        }

        out_path = STAGE_DIR / "environment_summary.json"
        with open(out_path, "w") as f:
            json.dump(summary, f, indent=2, default=str)

        # Read back to confirm the artifact is valid, non-corrupted JSON.
        with open(out_path) as f:
            reloaded = json.load(f)
        assert reloaded["dataset_discovery"].keys() == {"artifacts", "registry_manifests", "images"}, \
            "environment_summary.json failed round-trip validation"

        logger.info(f"artifact_written path={out_path}")
        summary["_artifact_path"] = str(out_path)
        return summary


# -----------------------------------------------------------------------
# Re-run Stage 1 with the patched discovery logic
# -----------------------------------------------------------------------
ENV_SUMMARY = run_stage1()

print("\n" + "=" * 70)
print("STAGE 1 — ENVIRONMENT SUMMARY (patched)")
print("=" * 70)
print(f"Execution mode : {ENV_SUMMARY['execution_mode']}")
print(f"Device         : {ENV_SUMMARY['device']['selected']} "
      f"({ENV_SUMMARY['device'].get('device_name', 'n/a')})")
print("Datasets found :")
for role, d in ENV_SUMMARY["dataset_discovery"].items():
    print(f"  - {role:20s} -> {d['top_level_name']}")
print(f"Artifact       : {ENV_SUMMARY['_artifact_path']}")
print("Stage 1 — Environment & Discovery : OK")


2026-07-05 05:30:03 | INFO    | sprint04_eval.stage01_environment | START  stage=stage01_environment
2026-07-05 05:30:03 | INFO    | sprint04_eval.stage01_environment | seed_set seed=42 info={'cuda_seeded': True, 'cudnn_deterministic': True, 'cudnn_benchmark': False}
2026-07-05 05:30:03 | INFO    | sprint04_eval.stage01_environment | device_selected=cuda info={'backend': 'cuda', 'device_name': 'Tesla T4', 'total_memory_gb': 14.56, 'cuda_capability': '7.5', 'cuda_version': '12.8', 'cudnn_version': 91002}
2026-07-05 05:30:03 | INFO    | sprint04_eval.stage01_environment | requirements_validated hard_ok=True soft={'pyarrow': 'present'}
2026-07-05 05:30:04 | INFO    | sprint04_eval.stage01_environment | dataset_found role=registry_manifests path=anupsharma1730/visionserveai-training-artifacts-v1
2026-07-05 05:30:04 | INFO    | sprint04_eval.stage01_environment | dataset_found role=artifacts path=anupsharma1730/visionserveai-sprint04-artifacts
2026-07-05 05:30:04 | INFO    | sprint04_eval.s

In [3]:
# =============================================================================
# Sprint04_Evaluation.ipynb
# STAGE 2 — Artifact Loader & Validator
# -----------------------------------------------------------------------------
# READ-ONLY. Discovers, loads, and validates all Sprint04 artifacts:
#   - best_model.pt (architecture + checkpoint compatibility)
#   - training_summary.json / training_report.json / training_history.json
#   - disease_registry / test_manifest (source of truth for labels)
# Does NOT recreate optimizer/scheduler/AMP/training loop. Rebuilds only the
# bare nn.Module architecture required to load and shape-check the checkpoint.
# Self-contained: re-derives paths from disk rather than trusting in-memory
# state left over from Stage 1, so it is independently rerunnable.
# =============================================================================

import os, sys, json, time, gc, re, fnmatch, logging
from pathlib import Path
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Optional, Dict, List, Tuple, Any

import torch
import torch.nn as nn
import pandas as pd

try:
    import timm
except ImportError as exc:
    raise ImportError("timm is required for Stage 2 (architecture reconstruction) "
                       "but is not installed.") from exc

# -----------------------------------------------------------------------------
# 2.0  Re-derive paths independently (do not trust in-memory Stage 1 state)
# -----------------------------------------------------------------------------
_KAGGLE_INPUT = Path("/kaggle/input")
_KAGGLE_WORKING = Path("/kaggle/working")
IS_KAGGLE = _KAGGLE_INPUT.is_dir()
INPUT_ROOT = _KAGGLE_INPUT if IS_KAGGLE else Path(os.environ.get("EVAL_INPUT_ROOT", "kaggle_input"))
OUTPUT_ROOT = (_KAGGLE_WORKING / "sprint04_evaluation") if IS_KAGGLE else Path(os.environ.get("EVAL_OUTPUT_ROOT", "sprint04_evaluation"))
STAGE1_SUMMARY_PATH = OUTPUT_ROOT / "stage01_environment" / "environment_summary.json"

if not STAGE1_SUMMARY_PATH.is_file():
    raise FileNotFoundError(
        f"Stage 1 output not found at {STAGE1_SUMMARY_PATH}. Run Stage 1 first — "
        "Stage 2 reads dataset roots from its artifact, not from memory."
    )
with open(STAGE1_SUMMARY_PATH) as f:
    STAGE1_SUMMARY = json.load(f)

ARTIFACTS_ROOT = Path(STAGE1_SUMMARY["dataset_discovery"]["artifacts"]["path"]) / "visionserveai" / "sprint04"
REGISTRY_ROOT  = Path(STAGE1_SUMMARY["dataset_discovery"]["registry_manifests"]["path"]) / "visionserveai" / "sprint03"

STAGE_DIR = OUTPUT_ROOT / "stage02_artifact_loader"
STAGE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = OUTPUT_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# 2.1  Logging (same pattern as Stage 1, reusable helper reproduced for
#      per-stage independence)
# -----------------------------------------------------------------------------
def setup_logging(log_dir: Path, stage_name: str) -> logging.Logger:
    logger = logging.getLogger(f"sprint04_eval.{stage_name}")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False
    fmt = logging.Formatter(fmt="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
                             datefmt="%Y-%m-%d %H:%M:%S")
    console = logging.StreamHandler(sys.stdout); console.setFormatter(fmt)
    logger.addHandler(console)
    fh = logging.FileHandler(log_dir / f"{stage_name}.log", mode="a"); fh.setFormatter(fmt)
    logger.addHandler(fh)
    return logger

logger = setup_logging(LOG_DIR, "stage02_artifact_loader")

class StageTimer:
    def __init__(self, stage_name: str, log: logging.Logger):
        self.stage_name, self.log, self.t0, self.warnings = stage_name, log, None, []
    def warn(self, msg: str) -> None:
        self.warnings.append(msg); self.log.warning(msg)
    def __enter__(self):
        self.t0 = time.perf_counter()
        self.log.info(f"START  stage={self.stage_name}")
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        dur = time.perf_counter() - self.t0
        status = "FAILED" if exc_type else "FINISH"
        self.log.info(f"{status} stage={self.stage_name} duration_sec={dur:.2f} warnings={len(self.warnings)}")
        return False

def log_resource_usage(log: logging.Logger, tag: str) -> Dict[str, float]:
    usage = {}
    if torch.cuda.is_available():
        usage["gpu_allocated_gb"] = round(torch.cuda.memory_allocated() / 1024**3, 3)
        usage["gpu_reserved_gb"] = round(torch.cuda.memory_reserved() / 1024**3, 3)
    try:
        import resource
        usage["cpu_maxrss_mb"] = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1)
    except Exception:
        pass
    log.info(f"resource_usage tag={tag} {usage}")
    return usage


# =============================================================================
# 2.2  Exclusion rules — filters out debug / smoke-test / backup artifacts
#      that sit alongside production files in the real Sprint04 tree.
# =============================================================================
_EXCLUDE_PATH_SEGMENT = re.compile(r"^_", re.IGNORECASE)                 # e.g. _s31_smoke, _s36_run_a
_EXCLUDE_FILENAME_PATTERNS = [
    "*smoke*",
    "*backup*",
    "*resume_test*",
    "*verification_checkpoint*",
    "*phase4_engine_verification*",
]

def is_excluded(path: Path, search_root: Path) -> Optional[str]:
    """Returns the exclusion reason, or None if the path should be kept."""
    rel_parts = path.relative_to(search_root).parts
    for part in rel_parts[:-1]:  # directory components only
        if _EXCLUDE_PATH_SEGMENT.match(part):
            return f"path segment {part!r} looks like a debug/experiment run"
    fname = rel_parts[-1]
    for pat in _EXCLUDE_FILENAME_PATTERNS:
        if fnmatch.fnmatch(fname.lower(), pat):
            return f"filename matches exclusion pattern {pat!r}"
    return None


def find_candidates(search_root: Path, filename_patterns: List[str], max_depth: int = 6) -> List[Path]:
    """Bounded rglob across filename_patterns, then filter through is_excluded."""
    kept: List[Path] = []
    base_depth = len(search_root.parts)
    for pat in filename_patterns:
        for p in search_root.rglob(pat):
            if not p.is_file():
                continue
            if len(p.parts) - base_depth > max_depth:
                continue
            reason = is_excluded(p, search_root)
            if reason is None:
                kept.append(p)
    return sorted(set(kept))


@dataclass
class ArtifactSpec:
    key: str
    filename_patterns: List[str]
    required: bool
    kind: str  # "json" | "csv" | "checkpoint"


ARTIFACT_REGISTRY: List[ArtifactSpec] = [
    ArtifactSpec("best_model",                  ["best_model.pt"],               True,  "checkpoint"),
    ArtifactSpec("last_model",                   ["last_model.pt"],               False, "checkpoint"),
    ArtifactSpec("checkpoint_summary",           ["checkpoint_summary.json"],     True,  "json"),
    ArtifactSpec("training_history",             ["training_history.json"],       True,  "json"),
    ArtifactSpec("best_model_metadata",          ["best_model_metadata.json"],    True,  "json"),
    ArtifactSpec("training_report",              ["training_report.json"],        True,  "json"),
    ArtifactSpec("training_summary",             ["training_summary.json"],       True,  "json"),
    ArtifactSpec("dataloader_summary",           ["dataloader_summary.json"],     False, "json"),
    ArtifactSpec("early_stopping_config",        ["early_stopping_config.json"],  False, "json"),
    ArtifactSpec("production_controller_summary",["production_controller_summary.json"], False, "json"),
    ArtifactSpec("engineering_verification_summary", ["engineering_verification_summary.json"], False, "json"),
]

REGISTRY_ARTIFACT_REGISTRY: List[ArtifactSpec] = [
    ArtifactSpec("disease_registry", [ "disease_registry.json"], True, "json"),
    ArtifactSpec("test_manifest",    [ "test_manifest.csv"],         True, "csv"),
    ArtifactSpec("train_manifest",   [ "train_manifest.csv"],       False, "csv"),
    ArtifactSpec("val_manifest",     [ "val_manifest.csv"],           False, "csv"),
]


def resolve_artifact(spec: ArtifactSpec, search_root: Path, log: logging.Logger,
                      report: Dict[str, Any]) -> Tuple[Optional[Path], List[str]]:
    """Resolve one ArtifactSpec to a single path using pattern precedence
    (first pattern in the list wins if multiple patterns match distinct files —
    this is how the versioned-vs-unversioned duplicate registry/manifest files
    are deterministically disambiguated)."""
    notes: List[str] = []
    for pattern_rank, pattern in enumerate(spec.filename_patterns):
        candidates = find_candidates(search_root, [pattern])
        if len(candidates) == 0:
            continue
        if len(candidates) > 1:
            notes.append(f"{spec.key}: multiple files matched pattern {pattern!r}: "
                          f"{[str(c.relative_to(search_root)) for c in candidates]} — "
                          f"using first by sorted order: {candidates[0]}")
            log.warning(notes[-1])
        chosen = candidates[0]
        if pattern_rank > 0:
            notes.append(f"{spec.key}: preferred pattern(s) "
                         f"{spec.filename_patterns[:pattern_rank]} not found; "
                         f"fell back to {pattern!r} -> {chosen}")
            log.warning(notes[-1])
        # Cross-check against lower-precedence duplicates, if present, for silent-drift detection
        for other_pattern in spec.filename_patterns[pattern_rank + 1:]:
            dupes = find_candidates(search_root, [other_pattern])
            if dupes:
                notes.append(f"{spec.key}: duplicate copy also exists at "
                              f"{dupes[0].relative_to(search_root)} (pattern {other_pattern!r}); "
                              f"using {chosen.relative_to(search_root)} per precedence, not cross-diffed byte-for-byte.")
                log.warning(notes[-1])
        return chosen, notes
    return None, notes


# =============================================================================
# 2.3  JSON / CSV loaders with explicit corruption handling
# =============================================================================
def load_json_strict(path: Path) -> Dict[str, Any]:
    try:
        with open(path) as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(f"Corrupted JSON at {path}: {e}") from e


def validate_csv_header(path: Path, required_columns: List[str]) -> Dict[str, Any]:
    try:
        head = pd.read_csv(path, nrows=5)
    except Exception as e:
        raise ValueError(f"Failed to parse CSV at {path}: {e}") from e
    missing_cols = [c for c in required_columns if c not in head.columns]
    if missing_cols:
        raise ValueError(f"CSV at {path} is missing required columns: {missing_cols}. "
                          f"Found columns: {list(head.columns)}")
    with open(path) as f:
        row_count = sum(1 for _ in f) - 1  # minus header
    return {"columns": list(head.columns), "row_count": row_count}


# =============================================================================
# 2.4  Model architecture reconstruction (factory logic reused from Sprint04
#      training code — NOT the training loop. Only used here to give the
#      checkpoint's state_dict something to load into for shape validation.)
# =============================================================================
@dataclass(frozen=True)
class BackboneSpec:
    timm_name: str
    family: str
    feature_dim: int
    classifier_attr: str
    status: str

BACKBONE_REGISTRY: Dict[str, BackboneSpec] = {
    "densenet121": BackboneSpec("densenet121", "DenseNet", 1024, "classifier", "ACTIVE"),
}

def _get_module_by_path(module: nn.Module, path: str) -> nn.Module:
    obj = module
    for part in path.split("."):
        obj = getattr(obj, part)
    return obj

def _set_module_by_path(module: nn.Module, path: str, new_submodule: nn.Module) -> None:
    parts = path.split(".")
    obj = module
    for part in parts[:-1]:
        obj = getattr(obj, part)
    setattr(obj, parts[-1], new_submodule)

def get_backbone(backbone_name: str, pretrained: bool = False) -> nn.Module:
    spec = BACKBONE_REGISTRY[backbone_name]
    return timm.create_model(spec.timm_name, pretrained=pretrained)

def replace_classifier(model: nn.Module, backbone_name: str, num_classes: int, dropout: float = 0.3):
    spec = BACKBONE_REGISTRY[backbone_name]
    old_head = _get_module_by_path(model, spec.classifier_attr)
    in_features = old_head.in_features
    new_head = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_features, num_classes))
    _set_module_by_path(model, spec.classifier_attr, new_head)
    return model, in_features

class ChestXrayClassifier(nn.Module):
    def __init__(self, backbone: nn.Module, backbone_name: str, num_classes: int) -> None:
        super().__init__()
        self.backbone = backbone
        self.backbone_name = backbone_name
        self.num_classes = num_classes
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)

def build_model_for_eval(backbone_name: str, num_classes: int, dropout: float) -> ChestXrayClassifier:
    """pretrained=False deliberately — checkpoint weights fully overwrite the
    backbone, so downloading ImageNet weights here would be wasted network I/O
    (and Kaggle's egress network may not permit it) and has zero effect on the
    loaded model."""
    raw_model = get_backbone(backbone_name, pretrained=False)
    raw_model, _ = replace_classifier(raw_model, backbone_name, num_classes, dropout)
    return ChestXrayClassifier(raw_model, backbone_name, num_classes)


def extract_state_dict(checkpoint_obj: Any) -> Dict[str, torch.Tensor]:
    if isinstance(checkpoint_obj, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            if key in checkpoint_obj and isinstance(checkpoint_obj[key], dict):
                return checkpoint_obj[key]
        # maybe it *is* already a raw state_dict
        if all(torch.is_tensor(v) for v in checkpoint_obj.values()):
            return checkpoint_obj
    raise ValueError(
        "Could not locate a model state_dict inside the checkpoint object. "
        f"Top-level type={type(checkpoint_obj)}, "
        f"keys={list(checkpoint_obj.keys()) if isinstance(checkpoint_obj, dict) else 'N/A'}"
    )


def load_checkpoint_into_model(ckpt_path: Path, model: nn.Module, log: logging.Logger) -> Dict[str, Any]:
    try:
        raw = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    except Exception as e:
        log.warning(f"weights_only=True load failed ({e}); retrying with weights_only=False "
                    f"(trusted first-party checkpoint from our own Sprint04 pipeline).")
        raw = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    state_dict = extract_state_dict(raw)

    # Try strict load; fall back to common prefix fixes; never silently accept partial loads.
    attempts = []
    for label, transform in [
        ("as-is", lambda sd: sd),
        ("strip 'module.' (DataParallel)", lambda sd: {k.replace("module.", "", 1): v for k, v in sd.items()}),
        ("strip 'backbone.' prefix", lambda sd: {k.replace("backbone.", "", 1): v for k, v in sd.items()}),
    ]:
        try:
            transformed = transform(state_dict)
            missing, unexpected = model.load_state_dict(transformed, strict=True)
            attempts.append({"transform": label, "result": "success"})
            return {
                "load_transform_used": label,
                "attempts": attempts,
                "missing_keys": [],
                "unexpected_keys": [],
                "checkpoint_top_level_keys": list(raw.keys()) if isinstance(raw, dict) else None,
                "param_count_loaded": sum(v.numel() for v in transformed.values()),
            }
        except RuntimeError as e:
            attempts.append({"transform": label, "result": "failed", "error": str(e)[:500]})
            continue

    raise RuntimeError(
        f"Checkpoint {ckpt_path} is architecture-incompatible with "
        f"ChestXrayClassifier(densenet121). All load strategies failed: {attempts}"
    )


# =============================================================================
# 2.5  Run Stage 2
# =============================================================================
def run_stage2() -> Dict[str, Any]:
    with StageTimer("stage02_artifact_loader", logger) as timer:
        report: Dict[str, Any] = {
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "artifacts_root": str(ARTIFACTS_ROOT),
            "registry_root": str(REGISTRY_ROOT),
            "resolved_paths": {},
            "missing_required": [],
            "missing_optional": [],
            "corrupted": [],
            "resolution_notes": [],
            "loaded": {},
        }
        log_resource_usage(logger, "stage02_start")

        # --- resolve every artifact -----------------------------------------
        for spec, root in [(s, ARTIFACTS_ROOT) for s in ARTIFACT_REGISTRY] + \
                           [(s, REGISTRY_ROOT) for s in REGISTRY_ARTIFACT_REGISTRY]:
            path, notes = resolve_artifact(spec, root, logger, report)
            report["resolution_notes"].extend(notes)
            if path is None:
                target = report["missing_required"] if spec.required else report["missing_optional"]
                target.append(spec.key)
                logger.info(f"artifact_missing key={spec.key} required={spec.required}")
                continue
            report["resolved_paths"][spec.key] = str(path)
            logger.info(f"artifact_resolved key={spec.key} path={path}")

        if report["missing_required"]:
            raise FileNotFoundError(
                f"Stage 2 aborted — required artifacts missing: {report['missing_required']}. "
                f"See resolution_notes for search details."
            )

        # --- load + validate JSON artifacts -----------------------------------
        json_keys = [s.key for s in ARTIFACT_REGISTRY + REGISTRY_ARTIFACT_REGISTRY if s.kind == "json"]
        for key in json_keys:
            if key not in report["resolved_paths"]:
                continue
            path = Path(report["resolved_paths"][key])
            try:
                report["loaded"][key] = load_json_strict(path)
            except ValueError as e:
                report["corrupted"].append({"key": key, "path": str(path), "error": str(e)})
                logger.error(str(e))

        required_json_keys = [s.key for s in ARTIFACT_REGISTRY + REGISTRY_ARTIFACT_REGISTRY
                               if s.kind == "json" and s.required]
        corrupted_required = [c for c in report["corrupted"] if c["key"] in required_json_keys]
        if corrupted_required:
            raise ValueError(f"Stage 2 aborted — required JSON artifacts are corrupted: {corrupted_required}")

        # --- surface the training pipeline's own self-reported flag ----------
        ts = report["loaded"]["training_summary"]
        if ts.get("artifact_validation_passed") is False:
            timer.warn(
                "training_summary.json self-reports artifact_validation_passed=False. "
                "Proceeding with Stage 2's OWN independent validation below — this is "
                "informational, not a pass-through of the training pipeline's verdict."
            )

        # --- validate manifest CSV structure ----------------------------------
        expected_manifest_cols = [
            "image_index", "finding_labels", "patient_id", "image_path",
            "parsed_labels", "encoded_labels", "num_labels",
        ]
        report["loaded"]["test_manifest_info"] = validate_csv_header(
            Path(report["resolved_paths"]["test_manifest"]), expected_manifest_cols
        )
        logger.info(f"test_manifest validated: {report['loaded']['test_manifest_info']}")

        # --- class count consistency -------------------------------------------
        registry_json = report["loaded"]["disease_registry"]
        n_classes_registry = len(registry_json["diseases"])
        n_classes_training_summary = ts["model"]["num_classes"]
        class_count_consistent = (n_classes_registry == n_classes_training_summary)
        report["class_count_consistency"] = {
            "disease_registry": n_classes_registry,
            "training_summary_model": n_classes_training_summary,
            "consistent": class_count_consistent,
        }
        if not class_count_consistent:
            raise ValueError(
                f"Class count mismatch: disease_registry has {n_classes_registry} classes, "
                f"training_summary.json model.num_classes={n_classes_training_summary}."
            )
        logger.info(f"class_count_consistency={report['class_count_consistency']}")

        # --- checkpoint / architecture compatibility ----------------------------
        backbone_name = ts["model"]["backbone"]
        dropout = ts["model"]["dropout"]
        if backbone_name not in BACKBONE_REGISTRY:
            raise ValueError(f"training_summary.json specifies backbone={backbone_name!r}, "
                              f"which is not in this evaluation notebook's BACKBONE_REGISTRY "
                              f"{list(BACKBONE_REGISTRY)}. Add it before proceeding.")

        eval_model = build_model_for_eval(backbone_name, n_classes_registry, dropout)
        param_counts_before = sum(p.numel() for p in eval_model.parameters())

        ckpt_path = Path(report["resolved_paths"]["best_model"])
        load_result = load_checkpoint_into_model(ckpt_path, eval_model, logger)
        report["checkpoint_compatibility"] = {
            "checkpoint_path": str(ckpt_path),
            "architecture": f"{backbone_name} + Linear({n_classes_registry})",
            "expected_param_count": param_counts_before,
            **load_result,
        }
        params_match = load_result["param_count_loaded"] == param_counts_before
        report["checkpoint_compatibility"]["param_counts_match"] = params_match
        if not params_match:
            timer.warn(f"Loaded param count {load_result['param_count_loaded']} != "
                       f"model param count {param_counts_before} (dropout layers have 0 "
                       f"params so this is usually fine, but flagging for audit).")
        logger.info(f"checkpoint_compatibility={report['checkpoint_compatibility']['load_transform_used']} "
                    f"params_match={params_match}")

        del eval_model
        gc.collect()

        # --- cross-check checkpoint metadata against training_summary (informational) ---
        best_meta = report["loaded"]["best_model_metadata"]
        cross_check = {}
        for field_name, ts_key in [("epoch", "best_epoch"), ("val_macro_auroc", "best_val_macro_auroc"),
                                     ("val_loss", "best_val_loss")]:
            meta_val = best_meta.get(field_name)
            ts_val = ts["training"].get(ts_key)
            match = (meta_val == ts_val) if meta_val is not None and ts_val is not None else None
            cross_check[field_name] = {"best_model_metadata": meta_val, "training_summary": ts_val, "match": match}
            if match is False:
                timer.warn(f"Metadata drift: best_model_metadata.{field_name}={meta_val} "
                           f"!= training_summary.{ts_key}={ts_val}")
        report["metadata_cross_check"] = cross_check

        log_resource_usage(logger, "stage02_end")

        out_path = STAGE_DIR / "artifact_validation.json"
        serializable_report = {k: v for k, v in report.items() if k != "loaded"}  # keep file lean; raw JSON stays on disk
        serializable_report["warnings"] = timer.warnings
        with open(out_path, "w") as f:
            json.dump(serializable_report, f, indent=2, default=str)
        with open(out_path) as f:
            json.load(f)  # round-trip corruption check on our own output

        logger.info(f"artifact_written path={out_path}")
        report["_artifact_path"] = str(out_path)
        return report


ARTIFACT_REPORT = run_stage2()

print("\n" + "=" * 70)
print("STAGE 2 — ARTIFACT VALIDATION SUMMARY")
print("=" * 70)
print(f"Required artifacts resolved : {len(ARTIFACT_REPORT['resolved_paths'])}")
print(f"Missing optional artifacts  : {ARTIFACT_REPORT['missing_optional']}")
print(f"Corrupted artifacts         : {len(ARTIFACT_REPORT['corrupted'])}")
print(f"Class count consistency     : {ARTIFACT_REPORT['class_count_consistency']}")
print(f"Checkpoint load strategy    : {ARTIFACT_REPORT['checkpoint_compatibility']['load_transform_used']}")
print(f"Checkpoint params match     : {ARTIFACT_REPORT['checkpoint_compatibility']['param_counts_match']}")
print(f"Metadata cross-check        : {ARTIFACT_REPORT['metadata_cross_check']}")
print(f"Resolution notes (dupes etc): {len(ARTIFACT_REPORT['resolution_notes'])}")
for n in ARTIFACT_REPORT["resolution_notes"]:
    print(f"  - {n}")
print(f"Artifact                    : {ARTIFACT_REPORT['_artifact_path']}")
print("Stage 2 — Artifact Loader & Validator : OK")

2026-07-05 05:30:04 | INFO    | sprint04_eval.stage02_artifact_loader | START  stage=stage02_artifact_loader
2026-07-05 05:30:04 | INFO    | sprint04_eval.stage02_artifact_loader | resource_usage tag=stage02_start {'gpu_allocated_gb': 0.0, 'gpu_reserved_gb': 0.0, 'cpu_maxrss_mb': 914.3}
2026-07-05 05:30:04 | INFO    | sprint04_eval.stage02_artifact_loader | artifact_resolved key=best_model path=/kaggle/input/datasets/anupsharma1730/visionserveai-sprint04-artifacts/visionserveai/sprint04/checkpoints/best_model.pt
2026-07-05 05:30:04 | INFO    | sprint04_eval.stage02_artifact_loader | artifact_resolved key=last_model path=/kaggle/input/datasets/anupsharma1730/visionserveai-sprint04-artifacts/visionserveai/sprint04/checkpoints/last_model.pt
2026-07-05 05:30:04 | INFO    | sprint04_eval.stage02_artifact_loader | artifact_resolved key=checkpoint_summary path=/kaggle/input/datasets/anupsharma1730/visionserveai-sprint04-artifacts/visionserveai/sprint04/checkpoints/checkpoint_summary.json
2026

In [4]:
# =============================================================================
# Sprint04_Evaluation.ipynb
# STAGE 3 — Production Evaluation Dataset Builder  (FIXED — rev. 2)
# -----------------------------------------------------------------------------
# READ-ONLY. Builds and validates the production evaluation Dataset/DataLoader
# for the NIH ChestXray14 test split. Does NOT load the model and does NOT
# run inference (that is Stage 4). Self-contained and independently rerunnable:
# re-derives every path from Stage 1's environment_summary.json and Stage 2's
# artifact_validation.json rather than trusting notebook memory.
#
# Per project rules: Stage 2's artifact_validation.json intentionally strips
# the "loaded" key before serialization. Stage 3 therefore does NOT rediscover
# files on disk — it reads resolved_paths from artifact_validation.json and
# reopens training_summary.json, disease_registry*.json and test_manifest*.csv
# using those exact, already-validated paths.
#
# FIX LOG:
#   rev.1 — dataloader_summary.json does NOT contain image_size/normalization.
#     Transform config is now resolved via priority-ordered search across
#     dataloader_summary -> training_summary -> checkpoint_summary ->
#     best_model_metadata (only accepting a *complete* {image_size, mean, std}
#     from a single source, never mixed). Confirmed against the real contents
#     of all three of those files: none carry it, so this correctly falls
#     through to timm's own pretrained_cfg for densenet121 — the actual
#     recipe those weights expect — logged loudly with full provenance.
#     Also fixed: DataLoader batch_size/shuffle now read from
#     dataloader_summary.json's splits.test block, not the top level (was
#     silently defaulting to 32 — right answer, wrong reason).
#   rev.2 — disease_registry.json's "diseases" key is a FLAT LIST OF STRINGS
#     in production, not a list of {"index"/"id","name"} objects. Indexing a
#     string with ["index"] crashed with TypeError. build_class_names now
#     detects the actual shape (flat string list / list-of-dicts with several
#     plausible key names / index<->name mapping dict) before extracting,
#     and only fails loudly if the shape is genuinely unrecognized.
# =============================================================================

import os, sys, json, time, ast, logging, gc
from pathlib import Path
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Optional, Dict, List, Tuple, Any

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image, UnidentifiedImageError
import torchvision.transforms as T

try:
    import timm
except ImportError as exc:
    raise ImportError(
        "timm is required for Stage 3 (fallback transform-config derivation "
        "from the backbone's pretrained recipe) but is not installed."
    ) from exc

# -----------------------------------------------------------------------------
# 3.0  Re-derive paths independently (do not trust in-memory Stage 1/2 state)
# -----------------------------------------------------------------------------
_KAGGLE_INPUT = Path("/kaggle/input")
_KAGGLE_WORKING = Path("/kaggle/working")
IS_KAGGLE = _KAGGLE_INPUT.is_dir()
INPUT_ROOT = _KAGGLE_INPUT if IS_KAGGLE else Path(os.environ.get("EVAL_INPUT_ROOT", "kaggle_input"))
OUTPUT_ROOT = (_KAGGLE_WORKING / "sprint04_evaluation") if IS_KAGGLE else Path(os.environ.get("EVAL_OUTPUT_ROOT", "sprint04_evaluation"))

STAGE1_SUMMARY_PATH = OUTPUT_ROOT / "stage01_environment" / "environment_summary.json"
STAGE2_SUMMARY_PATH = OUTPUT_ROOT / "stage02_artifact_loader" / "artifact_validation.json"

if not STAGE1_SUMMARY_PATH.is_file():
    raise FileNotFoundError(f"Stage 1 output not found at {STAGE1_SUMMARY_PATH}. Run Stage 1 first.")
if not STAGE2_SUMMARY_PATH.is_file():
    raise FileNotFoundError(f"Stage 2 output not found at {STAGE2_SUMMARY_PATH}. Run Stage 2 first.")

with open(STAGE1_SUMMARY_PATH) as f:
    STAGE1_SUMMARY = json.load(f)
with open(STAGE2_SUMMARY_PATH) as f:
    STAGE2_REPORT = json.load(f)

IMAGES_ROOT = Path(STAGE1_SUMMARY["dataset_discovery"]["images"]["path"])
RESOLVED_PATHS = {k: Path(v) for k, v in STAGE2_REPORT["resolved_paths"].items()}

REQUIRED_FOR_STAGE3 = [
    "training_summary", "disease_registry", "test_manifest", "dataloader_summary",
    "checkpoint_summary", "best_model_metadata",
]
missing = [k for k in REQUIRED_FOR_STAGE3 if k not in RESOLVED_PATHS]
if missing:
    raise FileNotFoundError(
        f"Stage 3 requires resolved_paths entries {missing} from Stage 2's "
        f"artifact_validation.json, but they are absent. Stage 2 must be rerun "
        f"or these artifacts must be attached. Stage 3 never rediscovers files "
        f"on its own — this is a hard stop."
    )

STAGE_DIR = OUTPUT_ROOT / "stage03_dataset_builder"
STAGE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = OUTPUT_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# 3.1  Logging / timing (identical pattern to Stage 1 and Stage 2, reproduced
#      for per-stage independence)
# -----------------------------------------------------------------------------
def setup_logging(log_dir: Path, stage_name: str) -> logging.Logger:
    logger = logging.getLogger(f"sprint04_eval.{stage_name}")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False
    fmt = logging.Formatter(fmt="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
                             datefmt="%Y-%m-%d %H:%M:%S")
    console = logging.StreamHandler(sys.stdout); console.setFormatter(fmt)
    logger.addHandler(console)
    fh = logging.FileHandler(log_dir / f"{stage_name}.log", mode="a"); fh.setFormatter(fmt)
    logger.addHandler(fh)
    return logger

logger = setup_logging(LOG_DIR, "stage03_dataset_builder")

class StageTimer:
    def __init__(self, stage_name: str, log: logging.Logger):
        self.stage_name, self.log, self.t0, self.warnings = stage_name, log, None, []
    def warn(self, msg: str) -> None:
        self.warnings.append(msg); self.log.warning(msg)
    def __enter__(self):
        self.t0 = time.perf_counter()
        self.log.info(f"START  stage={self.stage_name}")
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        dur = time.perf_counter() - self.t0
        status = "FAILED" if exc_type else "FINISH"
        self.log.info(f"{status} stage={self.stage_name} duration_sec={dur:.2f} warnings={len(self.warnings)}")
        return False

def log_resource_usage(log: logging.Logger, tag: str) -> Dict[str, float]:
    usage = {}
    if torch.cuda.is_available():
        usage["gpu_allocated_gb"] = round(torch.cuda.memory_allocated() / 1024**3, 3)
        usage["gpu_reserved_gb"] = round(torch.cuda.memory_reserved() / 1024**3, 3)
    try:
        import resource
        usage["cpu_maxrss_mb"] = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1)
    except Exception:
        pass
    log.info(f"resource_usage tag={tag} {usage}")
    return usage


# =============================================================================
# 3.2  Configuration
# =============================================================================
FULL_DECODE_SAMPLE_SIZE = 1500
FULL_DECODE_SEED = 42
MAX_CORRUPTION_RATIO = 0.001      # 0.1%
MAX_MISSING_IMAGE_RATIO = 0.001   # 0.1%


# =============================================================================
# 3.3  Reopen validated artifacts (never rediscover — paths came from Stage 2)
# =============================================================================
def load_json_strict(path: Path) -> Dict[str, Any]:
    try:
        with open(path) as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(f"Corrupted JSON at {path}: {e}") from e


def reopen_stage3_inputs(log: logging.Logger) -> Dict[str, Any]:
    training_summary = load_json_strict(RESOLVED_PATHS["training_summary"])
    disease_registry = load_json_strict(RESOLVED_PATHS["disease_registry"])
    dataloader_summary = load_json_strict(RESOLVED_PATHS["dataloader_summary"])
    checkpoint_summary = load_json_strict(RESOLVED_PATHS["checkpoint_summary"])
    best_model_metadata = load_json_strict(RESOLVED_PATHS["best_model_metadata"])
    test_manifest = pd.read_csv(RESOLVED_PATHS["test_manifest"])
    log.info(f"reopened training_summary path={RESOLVED_PATHS['training_summary']}")
    log.info(f"reopened disease_registry path={RESOLVED_PATHS['disease_registry']}")
    log.info(f"reopened dataloader_summary path={RESOLVED_PATHS['dataloader_summary']}")
    log.info(f"reopened checkpoint_summary path={RESOLVED_PATHS['checkpoint_summary']}")
    log.info(f"reopened best_model_metadata path={RESOLVED_PATHS['best_model_metadata']}")
    log.info(f"reopened test_manifest path={RESOLVED_PATHS['test_manifest']} rows={len(test_manifest)}")
    return {
        "training_summary": training_summary,
        "disease_registry": disease_registry,
        "dataloader_summary": dataloader_summary,
        "checkpoint_summary": checkpoint_summary,
        "best_model_metadata": best_model_metadata,
        "test_manifest": test_manifest,
    }


# =============================================================================
# 3.4  Class mapping — robust to multiple real-world registry shapes
# =============================================================================
_NAME_KEY_CANDIDATES = ["name", "label", "class_name", "disease_name"]
_INDEX_KEY_CANDIDATES = ["index", "id", "class_id", "label_id"]


def _extract_class_names_from_dict_entries(diseases: List[Dict[str, Any]]) -> List[str]:
    first = diseases[0]
    name_key = next((k for k in _NAME_KEY_CANDIDATES if k in first), None)
    index_key = next((k for k in _INDEX_KEY_CANDIDATES if k in first), None)
    if name_key is None or index_key is None:
        raise ValueError(
            f"disease_registry 'diseases' entries are dicts but expose none of the "
            f"expected name keys {_NAME_KEY_CANDIDATES} / index keys {_INDEX_KEY_CANDIDATES}. "
            f"First entry keys found: {sorted(first.keys())}."
        )
    ordered = sorted(diseases, key=lambda d: d[index_key])
    return [d[name_key] for d in ordered]


def build_class_names(disease_registry: Dict[str, Any], training_summary: Dict[str, Any],
                       timer: "StageTimer", log: logging.Logger) -> List[str]:
    """Robust to the realistic variety of registry shapes actually seen in the
    wild for this project:
      (a) diseases: ["Atelectasis", "Cardiomegaly", ...]      <- flat, list position IS the index
      (b) diseases: [{"index"/"id": 0, "name"/"label": ...}]  <- explicit per-entry ordering key
      (c) diseases: {"0": "Atelectasis", ...}                 <- index-string -> name mapping
      (d) diseases: {"Atelectasis": 0, ...}                   <- name -> index mapping
    Never assume one shape and index into it blindly — detect, then extract,
    then fail loudly only if the shape genuinely doesn't match any known case.
    """
    diseases = disease_registry.get("diseases")
    if diseases is None:
        raise ValueError("disease_registry.json has no top-level 'diseases' key.")

    if isinstance(diseases, dict):
        if not diseases:
            raise ValueError("disease_registry.json 'diseases' mapping is empty.")
        sample_val = next(iter(diseases.values()))
        if isinstance(sample_val, (int, float)):
            class_names = sorted(diseases.keys(), key=lambda k: diseases[k])
            log.info("disease_registry.diseases detected as a {name: index} mapping.")
        else:
            try:
                ordered_keys = sorted(diseases.keys(), key=lambda k: int(k))
            except (TypeError, ValueError):
                ordered_keys = sorted(diseases.keys())
                timer.warn("disease_registry.diseases mapping keys are not integer-parseable; "
                           "falling back to lexicographic key order. Verify this matches the "
                           "label encoding order used to build encoded_labels.")
            class_names = [diseases[k] if isinstance(diseases[k], str) else diseases[k].get("name")
                           for k in ordered_keys]
            log.info("disease_registry.diseases detected as an {index: name} mapping.")

    elif isinstance(diseases, list):
        if not diseases:
            raise ValueError("disease_registry.json 'diseases' list is empty.")
        first = diseases[0]
        if isinstance(first, str):
            if not all(isinstance(d, str) for d in diseases):
                raise ValueError("disease_registry.json 'diseases' list mixes strings with other types.")
            class_names = list(diseases)
            log.info("disease_registry.diseases detected as a flat list of class-name strings; "
                     "using list position as the class index order.")
        elif isinstance(first, dict):
            try:
                class_names = _extract_class_names_from_dict_entries(diseases)
            except (KeyError, ValueError) as e:
                raise ValueError(f"Could not derive class order from disease_registry: {e}") from e
            log.info("disease_registry.diseases detected as a list of per-class objects; "
                     "ordered via explicit index/id key.")
        else:
            raise ValueError(f"disease_registry.json 'diseases' list contains unsupported "
                              f"element type {type(first)}: {first!r}")
    else:
        raise ValueError(f"disease_registry.json 'diseases' has unsupported top-level type {type(diseases)}.")

    n_expected = training_summary["model"]["num_classes"]
    if len(class_names) != n_expected:
        raise ValueError(
            f"disease_registry produced {len(class_names)} classes but "
            f"training_summary.json model.num_classes={n_expected}. "
            f"This should have been caught by Stage 2; treating as a hard stop."
        )

    ts_class_list = training_summary.get("model", {}).get("class_names")
    if ts_class_list is not None:
        if list(ts_class_list) != class_names:
            timer.warn(
                f"Class order mismatch between disease_registry {class_names} "
                f"and training_summary.json model.class_names {ts_class_list}. "
                f"Proceeding with disease_registry order (source of truth for "
                f"labels), but this drift should be investigated before trusting "
                f"downstream metrics."
            )
    else:
        log.info("training_summary.json has no explicit class_names list to cross-check against; "
                 "trusting disease_registry order as-is.")
    return class_names


# =============================================================================
# 3.5  Label parsing
# =============================================================================
def _parse_numpy_array_string(raw: str) -> Optional[List[int]]:
    """Handles encoded_labels as literally written in production:
    str(np.array([...])) -> '[0. 0. 1. 0.]' — space-separated, no commas,
    trailing dots. Not valid Python list syntax, so ast.literal_eval always
    fails on it; this is the actual format, not a corruption."""
    s = raw.strip()
    if not (s.startswith("[") and s.endswith("]")):
        return None
    inner = s[1:-1].strip()
    if inner == "":
        return []
    tokens = inner.split()
    try:
        values = [float(tok) for tok in tokens]
    except ValueError:
        return None
    return [int(round(v)) for v in values]


def safe_parse_list(raw: Any, row_context: str) -> list:
    if isinstance(raw, (list, tuple)):
        return list(raw)
    if not isinstance(raw, str):
        raise ValueError(f"{row_context}: expected string/list, got {type(raw)}: {raw!r}")

    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
    except (ValueError, SyntaxError):
        pass  # fall through to numpy-string handling below

    numpy_parsed = _parse_numpy_array_string(raw)
    if numpy_parsed is not None:
        return numpy_parsed

    raise ValueError(f"{row_context}: could not parse {raw!r} as a Python literal "
                      f"or a numpy array string.")


def parse_and_validate_manifest(test_manifest: pd.DataFrame, class_names: List[str],
                                 images_root: Path, timer: "StageTimer",
                                 log: logging.Logger) -> pd.DataFrame:
    n_classes = len(class_names)
    df = test_manifest.copy()
    label_vectors: List[List[int]] = []
    parse_failures: List[str] = []

    for i, row in df.iterrows():
        ctx = f"row={i} image_index={row.get('image_index')}"
        try:
            vec = safe_parse_list(row["encoded_labels"], ctx)
            vec = [int(v) for v in vec]
        except (ValueError, TypeError) as e:
            parse_failures.append(str(e))
            vec = None
        if vec is not None and len(vec) != n_classes:
            parse_failures.append(f"{ctx}: encoded_labels length {len(vec)} != num_classes {n_classes}")
            vec = None
        if vec is not None:
            declared = row.get("num_labels")
            if pd.notna(declared) and int(declared) != sum(vec):
                parse_failures.append(
                    f"{ctx}: num_labels={declared} does not match sum(encoded_labels)={sum(vec)}"
                )
        label_vectors.append(vec)

    if parse_failures:
        preview = parse_failures[:20]
        raise ValueError(
            f"Stage 3 aborted — {len(parse_failures)} manifest row(s) failed label validation "
            f"(zero-tolerance for label integrity). First {len(preview)}:\n" + "\n".join(preview)
        )

    df["label_vector"] = label_vectors

    def _resolve(raw_path: str) -> str:
        p = Path(raw_path)
        if p.is_absolute() and p.is_file():
            return str(p)
        candidate = images_root / raw_path
        if candidate.is_file():
            return str(candidate)
        candidate2 = images_root / Path(raw_path).name
        if candidate2.is_file():
            return str(candidate2)
        return ""

    df["resolved_image_path"] = df["image_path"].map(_resolve)
    n_unresolved = int((df["resolved_image_path"] == "").sum())
    if n_unresolved:
        ratio = n_unresolved / len(df)
        msg = f"{n_unresolved}/{len(df)} ({ratio:.4%}) image_path values could not be resolved under {images_root}"
        if ratio > MAX_MISSING_IMAGE_RATIO:
            raise FileNotFoundError(msg + " — exceeds MAX_MISSING_IMAGE_RATIO, hard stop.")
        timer.warn(msg + " — within tolerance, rows will be flagged in dataset_validation.json.")

    log.info(f"manifest parsed rows={len(df)} unresolved_images={n_unresolved}")
    return df


# =============================================================================
# 3.6  Integrity validation: duplicates, corruption, tensor shapes
# =============================================================================
def validate_duplicates(df: pd.DataFrame, timer: "StageTimer") -> Dict[str, Any]:
    dup_image_index = df["image_index"].duplicated().sum()
    dup_rows = df.duplicated(subset=["image_index", "image_path"]).sum()
    if dup_image_index:
        timer.warn(f"{dup_image_index} duplicate image_index value(s) found in test_manifest.")
    return {
        "duplicate_image_index_count": int(dup_image_index),
        "duplicate_full_row_count": int(dup_rows),
    }


def validate_images(df: pd.DataFrame, timer: "StageTimer", log: logging.Logger) -> Dict[str, Any]:
    missing_mask = df["resolved_image_path"] == ""
    missing_rows = df.loc[missing_mask, "image_index"].tolist()

    header_failures: List[str] = []
    present = df.loc[~missing_mask]
    for idx, path in zip(present["image_index"], present["resolved_image_path"]):
        try:
            with Image.open(path) as im:
                im.verify()
        except (UnidentifiedImageError, OSError, ValueError) as e:
            header_failures.append(f"image_index={idx} path={path}: {e}")

    rng = np.random.default_rng(FULL_DECODE_SEED)
    sample_n = min(FULL_DECODE_SAMPLE_SIZE, len(present))
    sample_idx = rng.choice(len(present), size=sample_n, replace=False)
    decode_failures: List[str] = []
    decoded_shapes = set()
    for pos in sample_idx:
        row = present.iloc[int(pos)]
        try:
            with Image.open(row["resolved_image_path"]) as im:
                im2 = im.convert("RGB")
                decoded_shapes.add(im2.size)
        except (UnidentifiedImageError, OSError, ValueError) as e:
            decode_failures.append(f"image_index={row['image_index']} path={row['resolved_image_path']}: {e}")

    total_corrupted = len(header_failures) + len(decode_failures)
    ratio = total_corrupted / max(len(df), 1)
    result = {
        "missing_files": missing_rows,
        "missing_count": len(missing_rows),
        "header_check_failures": header_failures[:50],
        "header_check_failure_count": len(header_failures),
        "full_decode_sample_size": sample_n,
        "full_decode_failures": decode_failures[:50],
        "full_decode_failure_count": len(decode_failures),
        "distinct_decoded_pixel_sizes_sampled": len(decoded_shapes),
        "corruption_ratio": ratio,
    }
    log.info(f"image_validation missing={len(missing_rows)} header_failures={len(header_failures)} "
             f"decode_failures={len(decode_failures)} sampled={sample_n}")
    if ratio > MAX_CORRUPTION_RATIO:
        raise ValueError(
            f"Stage 3 aborted — corrupted/unreadable image ratio {ratio:.4%} exceeds "
            f"MAX_CORRUPTION_RATIO={MAX_CORRUPTION_RATIO:.4%}. See dataset_validation.json for detail."
        )
    if total_corrupted:
        timer.warn(f"{total_corrupted} corrupted/unreadable image(s) detected but within tolerance.")
    return result


# =============================================================================
# 3.7  Transform configuration — multi-source resolution
# =============================================================================
def _deep_get(d: Dict[str, Any], path: Tuple[str, ...]) -> Any:
    cur = d
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return None
        cur = cur[key]
    return cur

_IMAGE_SIZE_PATHS = [
    ("image_size",), ("transform", "image_size"), ("preprocessing", "image_size"),
    ("model", "image_size"), ("model", "input_size"), ("data", "image_size"),
]
_MEAN_PATHS = [
    ("normalization", "mean"), ("transform", "normalization", "mean"),
    ("preprocessing", "mean"), ("preprocessing", "normalization", "mean"),
    ("model", "normalization", "mean"), ("data", "normalization", "mean"), ("mean",),
]
_STD_PATHS = [
    ("normalization", "std"), ("transform", "normalization", "std"),
    ("preprocessing", "std"), ("preprocessing", "normalization", "std"),
    ("model", "normalization", "std"), ("data", "normalization", "std"), ("std",),
]
_INTERP_PATHS = [("interpolation",), ("transform", "interpolation"), ("preprocessing", "interpolation")]


def _find_first(source: Dict[str, Any], paths: List[Tuple[str, ...]]) -> Any:
    for p in paths:
        v = _deep_get(source, p)
        if v is not None:
            return v
    return None


def _find_complete_transform_config(source: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    size = _find_first(source, _IMAGE_SIZE_PATHS)
    mean = _find_first(source, _MEAN_PATHS)
    std = _find_first(source, _STD_PATHS)
    if size is None or mean is None or std is None:
        return None
    interp = _find_first(source, _INTERP_PATHS) or "bilinear"
    return {"image_size": size, "mean": mean, "std": std, "interpolation": interp}


def _timm_fallback_transform_config(backbone_name: str, log: logging.Logger) -> Dict[str, Any]:
    """Last-resort, non-guessing fallback: derive image_size/mean/std from the
    backbone's own pretrained recipe metadata via timm. This is the actual
    preprocessing that architecture's weights expect — not an assumption —
    but it is flagged loudly since it wasn't explicitly recorded by the
    training pipeline and could in principle diverge if training used a
    custom, non-default transform. Confirmed for this project: none of
    dataloader_summary / training_summary / checkpoint_summary /
    best_model_metadata carry an explicit resize/normalization block, so this
    path is the one actually exercised in production."""
    model = timm.create_model(backbone_name, pretrained=False)
    raw_cfg = getattr(model, "pretrained_cfg", None) or getattr(model, "default_cfg", None)
    del model
    gc.collect()
    if raw_cfg is None:
        raise RuntimeError(f"timm backbone {backbone_name!r} exposes no pretrained_cfg/default_cfg; "
                            f"cannot derive a fallback transform configuration.")
    if hasattr(raw_cfg, "to_dict"):
        raw_cfg = raw_cfg.to_dict()
    elif not isinstance(raw_cfg, dict):
        raw_cfg = dict(raw_cfg)

    input_size = raw_cfg.get("test_input_size") or raw_cfg.get("input_size")
    mean, std = raw_cfg.get("mean"), raw_cfg.get("std")
    if input_size is None or mean is None or std is None:
        raise RuntimeError(f"timm pretrained_cfg for {backbone_name!r} is missing input_size/mean/std: {raw_cfg}")
    _, h, w = input_size
    interp = raw_cfg.get("interpolation", "bicubic")
    log.info(f"transform_config_fallback source=timm.pretrained_cfg backbone={backbone_name} "
             f"input_size=({h},{w}) mean={mean} std={std} interpolation={interp}")
    return {"image_size": [h, w], "mean": list(mean), "std": list(std), "interpolation": interp}


def resolve_transform_config(inputs: Dict[str, Any], timer: "StageTimer",
                              log: logging.Logger) -> Tuple[Dict[str, Any], str]:
    candidates = [
        ("dataloader_summary", inputs["dataloader_summary"]),
        ("training_summary", inputs["training_summary"]),
        ("checkpoint_summary", inputs["checkpoint_summary"]),
        ("best_model_metadata", inputs["best_model_metadata"]),
    ]
    for source_name, source in candidates:
        cfg = _find_complete_transform_config(source)
        if cfg is not None:
            log.info(f"transform_config resolved from source={source_name}: {cfg}")
            return cfg, source_name

    backbone_name = inputs["training_summary"]["model"]["backbone"]
    timer.warn(
        f"No artifact (dataloader_summary/training_summary/checkpoint_summary/"
        f"best_model_metadata) contains an explicit image_size+normalization "
        f"block. Falling back to backbone {backbone_name!r}'s own timm "
        f"pretrained_cfg for the eval transform. This is the architecturally "
        f"correct recipe for that backbone, but was not explicitly recorded by "
        f"the training pipeline — flagging for audit."
    )
    cfg = _timm_fallback_transform_config(backbone_name, log)
    return cfg, "timm_pretrained_cfg_fallback"


def build_eval_transform(cfg: Dict[str, Any], log: logging.Logger) -> Tuple[T.Compose, Tuple[int, int]]:
    size = cfg["image_size"]
    hw = (int(size), int(size)) if isinstance(size, (int, float)) else (int(size[0]), int(size[1]))
    mean, std = cfg["mean"], cfg["std"]

    interp_map = {
        "bilinear": T.InterpolationMode.BILINEAR,
        "bicubic": T.InterpolationMode.BICUBIC,
        "nearest": T.InterpolationMode.NEAREST,
    }
    interp_name = cfg.get("interpolation", "bilinear")
    if interp_name not in interp_map:
        raise ValueError(f"Unsupported interpolation {interp_name!r}.")

    transform = T.Compose([
        T.Resize(hw, interpolation=interp_map[interp_name]),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])
    log.info(f"eval_transform built image_size={hw} mean={mean} std={std} interpolation={interp_name}")
    return transform, hw


def validate_tensor_shapes(df: pd.DataFrame, transform: T.Compose, expected_hw: Tuple[int, int],
                            n_samples: int, log: logging.Logger) -> Dict[str, Any]:
    ok_df = df.loc[df["resolved_image_path"] != ""]
    rng = np.random.default_rng(FULL_DECODE_SEED)
    n = min(n_samples, len(ok_df))
    idxs = rng.choice(len(ok_df), size=n, replace=False)
    shapes = set()
    failures = []
    for pos in idxs:
        row = ok_df.iloc[int(pos)]
        try:
            with Image.open(row["resolved_image_path"]) as im:
                tensor = transform(im.convert("RGB"))
            shapes.add(tuple(tensor.shape))
        except Exception as e:
            failures.append(f"image_index={row['image_index']}: {e}")
    expected_shape = (3, expected_hw[0], expected_hw[1])
    consistent = shapes == {expected_shape}
    log.info(f"tensor_shape_validation sampled={n} shapes_seen={shapes} expected={expected_shape} "
             f"consistent={consistent} failures={len(failures)}")
    if failures:
        raise RuntimeError(f"Stage 3 aborted — transform application failed on {len(failures)} sample(s): "
                            f"{failures[:10]}")
    if not consistent:
        raise RuntimeError(f"Stage 3 aborted — transform produced inconsistent tensor shapes {shapes}, "
                            f"expected only {expected_shape}.")
    return {"sampled": n, "expected_shape": list(expected_shape), "consistent": consistent}


# =============================================================================
# 3.8  DataLoader config — read from splits.test, never top level
# =============================================================================
def get_test_split_config(dataloader_summary: Dict[str, Any], manifest_rowcount: int,
                           resolved_test_manifest_path: Path, timer: "StageTimer",
                           log: logging.Logger) -> Dict[str, Any]:
    splits = dataloader_summary.get("splits")
    if not splits or "test" not in splits:
        raise KeyError(
            "dataloader_summary.json has no splits.test block; cannot determine "
            "the evaluation batch_size/shuffle setting used at training time."
        )
    test_cfg = splits["test"]
    required = ["batch_size", "shuffle"]
    missing_keys = [k for k in required if k not in test_cfg]
    if missing_keys:
        raise KeyError(f"dataloader_summary.json splits.test is missing required key(s) {missing_keys}.")
    if test_cfg["shuffle"] is not False:
        raise ValueError(
            f"dataloader_summary.json splits.test.shuffle={test_cfg['shuffle']!r}. "
            f"Evaluation must be deterministic and unshuffled; refusing to proceed."
        )

    recorded_n = test_cfg.get("num_samples")
    if recorded_n is not None and recorded_n != manifest_rowcount:
        timer.warn(
            f"dataloader_summary.json splits.test.num_samples={recorded_n} does not match "
            f"the {manifest_rowcount} rows in the Stage-2-resolved test manifest "
            f"({resolved_test_manifest_path}). Proceeding with Stage 2's manifest as the "
            f"single source of truth per project rules, but flagging the discrepancy for audit."
        )
    recorded_manifest_path = test_cfg.get("manifest_path")
    if recorded_manifest_path and Path(recorded_manifest_path) != resolved_test_manifest_path:
        log.info(
            f"dataloader_summary.json records a training-time test manifest at "
            f"{recorded_manifest_path} (different location/format than the evaluation "
            f"notebook's attached datasets). Using Stage 2's resolved path "
            f"{resolved_test_manifest_path} instead — informational only, not an error."
        )

    return {
        "batch_size": int(test_cfg["batch_size"]),
        "num_workers": int(dataloader_summary.get("num_workers", 2)),
        "pin_memory": bool(dataloader_summary.get("pin_memory", torch.cuda.is_available())),
        "persistent_workers": bool(dataloader_summary.get("persistent_workers", False)) and
                               int(dataloader_summary.get("num_workers", 0)) > 0,
    }


def cross_check_images_root(dataloader_summary: Dict[str, Any], images_root: Path,
                             log: logging.Logger) -> None:
    recorded_roots = dataloader_summary.get("image_roots", [])
    if not recorded_roots:
        return
    parents = {str(Path(p).parent.parent) for p in recorded_roots}
    if len(parents) == 1:
        common_parent = next(iter(parents))
        if common_parent == str(images_root):
            log.info(f"images_root cross-check OK: matches dataloader_summary.image_roots common parent")
        else:
            log.info(
                f"images_root cross-check: Stage 1 discovered {images_root}, dataloader_summary.json's "
                f"image_roots share common parent {common_parent}. Using Stage 1's discovery result "
                f"(environment_summary.json is the sole source of truth for dataset locations) — "
                f"informational only."
            )


# =============================================================================
# 3.9  Dataset / DataLoader
# =============================================================================
class ChestXrayEvalDataset(Dataset):
    """Map-style evaluation dataset. Returns (image_tensor, label_tensor, sample_id).
    Order is fixed and never shuffled — evaluation requires stable, reproducible
    row-to-prediction alignment for every downstream stage."""

    def __init__(self, manifest_df: pd.DataFrame, transform: T.Compose):
        self.df = manifest_df.reset_index(drop=True)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        try:
            with Image.open(row["resolved_image_path"]) as im:
                tensor = self.transform(im.convert("RGB"))
        except Exception as e:
            raise RuntimeError(f"Failed to load image at idx={idx} "
                               f"image_index={row['image_index']} path={row['resolved_image_path']}: {e}") from e
        label = torch.tensor(row["label_vector"], dtype=torch.float32)
        return tensor, label, str(row["image_index"])


def smoke_test_dataloader(dataset: ChestXrayEvalDataset, loader_cfg: Dict[str, Any],
                           log: logging.Logger) -> Dict[str, Any]:
    kwargs = dict(
        batch_size=loader_cfg["batch_size"], shuffle=False, drop_last=False,
        num_workers=loader_cfg["num_workers"], pin_memory=loader_cfg["pin_memory"],
    )
    if loader_cfg["num_workers"] > 0:
        kwargs["persistent_workers"] = loader_cfg["persistent_workers"]

    loader = DataLoader(dataset, **kwargs)
    t0 = time.perf_counter()
    images, labels, ids = next(iter(loader))
    elapsed = time.perf_counter() - t0
    del loader
    gc.collect()
    log.info(f"dataloader_smoke_test cfg={loader_cfg} first_batch_shape={tuple(images.shape)} "
             f"label_shape={tuple(labels.shape)} elapsed_sec={elapsed:.2f}")
    return {
        **loader_cfg,
        "first_batch_image_shape": list(images.shape),
        "first_batch_label_shape": list(labels.shape),
        "first_batch_load_seconds": round(elapsed, 3),
    }


# =============================================================================
# 3.10  Statistics
# =============================================================================
def compute_class_distribution(df: pd.DataFrame, class_names: List[str]) -> pd.DataFrame:
    matrix = np.array(df["label_vector"].tolist())
    pos = matrix.sum(axis=0)
    n = len(df)
    return pd.DataFrame({
        "class_name": class_names,
        "positive_count": pos.astype(int),
        "negative_count": (n - pos).astype(int),
        "prevalence": pos / n,
    })


def compute_dataset_statistics(df: pd.DataFrame, class_names: List[str]) -> Dict[str, Any]:
    class_dist = compute_class_distribution(df, class_names)
    cardinality = df["label_vector"].map(sum)
    stats = {
        "n_samples": len(df),
        "n_unique_patients": int(df["patient_id"].nunique()) if "patient_id" in df.columns else None,
        "label_cardinality_histogram": {int(k): int(v) for k, v in cardinality.value_counts().sort_index().items()},
        "mean_labels_per_sample": float(cardinality.mean()),
        "view_position_counts": df["view_position"].value_counts().to_dict() if "view_position" in df.columns else {},
        "gender_counts": df["patient_gender"].value_counts().to_dict() if "patient_gender" in df.columns else {},
        "age_stats": {
            "min": float(df["patient_age"].min()),
            "max": float(df["patient_age"].max()),
            "mean": float(df["patient_age"].mean()),
            "median": float(df["patient_age"].median()),
        } if "patient_age" in df.columns else {},
        "class_prevalence": {row.class_name: row.prevalence for row in class_dist.itertuples()},
        "zero_positive_classes": class_dist.loc[class_dist["positive_count"] == 0, "class_name"].tolist(),
    }
    return stats


# =============================================================================
# 3.11  Run Stage 3
# =============================================================================
def run_stage3() -> Dict[str, Any]:
    with StageTimer("stage03_dataset_builder", logger) as timer:
        log_resource_usage(logger, "stage03_start")

        inputs = reopen_stage3_inputs(logger)
        class_names = build_class_names(inputs["disease_registry"], inputs["training_summary"], timer, logger)

        df = parse_and_validate_manifest(inputs["test_manifest"], class_names, IMAGES_ROOT, timer, logger)
        cross_check_images_root(inputs["dataloader_summary"], IMAGES_ROOT, logger)

        dup_report = validate_duplicates(df, timer)
        image_report = validate_images(df, timer, logger)

        transform_cfg, transform_source = resolve_transform_config(inputs, timer, logger)
        transform, expected_hw = build_eval_transform(transform_cfg, logger)
        shape_report = validate_tensor_shapes(df, transform, expected_hw, n_samples=200, log=logger)

        loader_cfg = get_test_split_config(
            inputs["dataloader_summary"], manifest_rowcount=len(df),
            resolved_test_manifest_path=RESOLVED_PATHS["test_manifest"], timer=timer, log=logger,
        )

        clean_df = df.loc[df["resolved_image_path"] != ""].reset_index(drop=True)

        dataset = ChestXrayEvalDataset(clean_df, transform)
        loader_report = smoke_test_dataloader(dataset, loader_cfg, logger)

        stats = compute_dataset_statistics(clean_df, class_names)
        class_dist_df = compute_class_distribution(clean_df, class_names)

        log_resource_usage(logger, "stage03_end")

        validation_report = {
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "duplicates": dup_report,
            "images": image_report,
            "tensor_shapes": shape_report,
            "dataloader_smoke_test": loader_report,
            "rows_dropped_unresolved_images": int(len(df) - len(clean_df)),
            "warnings": timer.warnings,
        }
        validation_path = STAGE_DIR / "dataset_validation.json"
        with open(validation_path, "w") as f:
            json.dump(validation_report, f, indent=2, default=str)

        stats_path = STAGE_DIR / "dataset_statistics.json"
        with open(stats_path, "w") as f:
            json.dump(stats, f, indent=2, default=str)

        class_dist_path = STAGE_DIR / "class_distribution.csv"
        class_dist_df.to_csv(class_dist_path, index=False)

        export_cols = ["image_index", "patient_id", "resolved_image_path", "label_vector",
                        "num_labels", "finding_labels", "view_position", "patient_age", "patient_gender"]
        export_cols = [c for c in export_cols if c in clean_df.columns]
        flat_df = clean_df[export_cols].copy()
        flat_df["label_vector"] = flat_df["label_vector"].map(json.dumps)
        summary_csv_path = STAGE_DIR / "dataset_summary.csv"
        flat_df.to_csv(summary_csv_path, index=False)

        summary = {
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "class_names": class_names,
            "num_classes": len(class_names),
            "n_samples_total_manifest": len(df),
            "n_samples_usable": len(clean_df),
            "images_root": str(IMAGES_ROOT),
            "transform": {
                "image_size": list(expected_hw),
                "mean": transform_cfg["mean"],
                "std": transform_cfg["std"],
                "interpolation": transform_cfg.get("interpolation", "bilinear"),
                "config_source": transform_source,
            },
            "dataloader_config": {**loader_cfg, "shuffle": False, "drop_last": False},
            "source_artifacts": {k: str(v) for k, v in RESOLVED_PATHS.items() if k in REQUIRED_FOR_STAGE3},
            "outputs": {
                "dataset_validation": str(validation_path),
                "dataset_statistics": str(stats_path),
                "class_distribution": str(class_dist_path),
                "dataset_summary_csv": str(summary_csv_path),
            },
            "warnings": timer.warnings,
        }
        summary_path = STAGE_DIR / "dataset_summary.json"
        with open(summary_path, "w") as f:
            json.dump(summary, f, indent=2, default=str)
        with open(summary_path) as f:
            json.load(f)  # round-trip corruption check

        logger.info(f"artifact_written path={summary_path}")
        summary["_artifact_path"] = str(summary_path)
        return summary


DATASET_SUMMARY = run_stage3()

print("\n" + "=" * 70)
print("STAGE 3 — DATASET BUILDER SUMMARY")
print("=" * 70)
print(f"Classes              : {DATASET_SUMMARY['num_classes']}")
print(f"Samples (manifest)   : {DATASET_SUMMARY['n_samples_total_manifest']}")
print(f"Samples (usable)     : {DATASET_SUMMARY['n_samples_usable']}")
print(f"Image size           : {DATASET_SUMMARY['transform']['image_size']} "
      f"(source: {DATASET_SUMMARY['transform']['config_source']})")
print(f"Batch size           : {DATASET_SUMMARY['dataloader_config']['batch_size']}")
print(f"Warnings             : {len(DATASET_SUMMARY['warnings'])}")
for w in DATASET_SUMMARY["warnings"]:
    print(f"  - {w}")
print(f"Artifact             : {DATASET_SUMMARY['_artifact_path']}")
print("Stage 3 — Dataset Builder : OK")

2026-07-05 05:30:05 | INFO    | sprint04_eval.stage03_dataset_builder | START  stage=stage03_dataset_builder
2026-07-05 05:30:05 | INFO    | sprint04_eval.stage03_dataset_builder | resource_usage tag=stage03_start {'gpu_allocated_gb': 0.0, 'gpu_reserved_gb': 0.0, 'cpu_maxrss_mb': 976.5}
2026-07-05 05:30:05 | INFO    | sprint04_eval.stage03_dataset_builder | reopened training_summary path=/kaggle/input/datasets/anupsharma1730/visionserveai-sprint04-artifacts/visionserveai/sprint04/training_summary.json
2026-07-05 05:30:05 | INFO    | sprint04_eval.stage03_dataset_builder | reopened disease_registry path=/kaggle/input/datasets/anupsharma1730/visionserveai-training-artifacts-v1/visionserveai/sprint03/registry/disease_registry.json
2026-07-05 05:30:05 | INFO    | sprint04_eval.stage03_dataset_builder | reopened dataloader_summary path=/kaggle/input/datasets/anupsharma1730/visionserveai-sprint04-artifacts/visionserveai/sprint04/dataloader_summary.json
2026-07-05 05:30:05 | INFO    | sprint0

In [5]:
# =============================================================================
# Sprint04_Evaluation.ipynb
# STAGE 4 — Production Inference Engine
# -----------------------------------------------------------------------------
# READ-ONLY. Consumes Stage 1-3 outputs exactly as written to disk (never
# trusts notebook memory). Loads best_model.pt using Stage 2's validated
# path, reconstructs the exact training architecture, independently
# re-validates checkpoint compatibility, then runs deterministic batched
# inference over the FULL Stage-3-cleaned evaluation set.
#
# Scope discipline (per project rules): inference ONLY. No metrics, no
# thresholding beyond the fixed 0.5 operating point, no GradCAM, no
# confusion matrices, no ROC/PR curves. Those are Stage 5+.
#
# Modularity note: this stage is intentionally split into the same
# functional seams it will later become:
#   model_loader.py   -> Section 4.4 (architecture + checkpoint)
#   predictor.py       -> Section 4.6 (batched inference loop)
#   artifact_manager.py-> Section 4.7 (parquet/csv/pt/json writers)
#   infer.py           -> Section 4.9 (orchestration / run_stage4)
# =============================================================================

import os, sys, json, time, gc, logging, platform
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Dict, List, Tuple, Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import torchvision.transforms as T

try:
    import timm
except ImportError as exc:
    raise ImportError(
        "timm is required for Stage 4 (architecture reconstruction) but is not installed."
    ) from exc

# -----------------------------------------------------------------------------
# 4.0  Re-derive paths independently (do not trust in-memory Stage 1-3 state)
# -----------------------------------------------------------------------------
_KAGGLE_INPUT = Path("/kaggle/input")
_KAGGLE_WORKING = Path("/kaggle/working")
IS_KAGGLE = _KAGGLE_INPUT.is_dir()
INPUT_ROOT = _KAGGLE_INPUT if IS_KAGGLE else Path(os.environ.get("EVAL_INPUT_ROOT", "kaggle_input"))
OUTPUT_ROOT = (_KAGGLE_WORKING / "sprint04_evaluation") if IS_KAGGLE else Path(os.environ.get("EVAL_OUTPUT_ROOT", "sprint04_evaluation"))

STAGE1_SUMMARY_PATH = OUTPUT_ROOT / "stage01_environment" / "environment_summary.json"
STAGE2_SUMMARY_PATH = OUTPUT_ROOT / "stage02_artifact_loader" / "artifact_validation.json"
STAGE3_SUMMARY_PATH = OUTPUT_ROOT / "stage03_dataset_builder" / "dataset_summary.json"

for _label, _p in [("Stage 1", STAGE1_SUMMARY_PATH), ("Stage 2", STAGE2_SUMMARY_PATH), ("Stage 3", STAGE3_SUMMARY_PATH)]:
    if not _p.is_file():
        raise FileNotFoundError(
            f"{_label} output not found at {_p}. Stage 4 never regenerates prior "
            f"stages — run them first, in order."
        )

with open(STAGE1_SUMMARY_PATH) as f:
    STAGE1_SUMMARY = json.load(f)
with open(STAGE2_SUMMARY_PATH) as f:
    STAGE2_REPORT = json.load(f)
with open(STAGE3_SUMMARY_PATH) as f:
    STAGE3_SUMMARY = json.load(f)

RESOLVED_PATHS = {k: Path(v) for k, v in STAGE2_REPORT["resolved_paths"].items()}

REQUIRED_FOR_STAGE4 = [
    "best_model", "training_summary", "checkpoint_summary", "best_model_metadata", "disease_registry",
]
_missing = [k for k in REQUIRED_FOR_STAGE4 if k not in RESOLVED_PATHS]
if _missing:
    raise FileNotFoundError(
        f"Stage 4 requires resolved_paths entries {_missing} from Stage 2's "
        f"artifact_validation.json, but they are absent. Rerun Stage 2. Stage 4 "
        f"never rediscovers checkpoints on its own — this is a hard stop."
    )

DATASET_SUMMARY_CSV_PATH = Path(STAGE3_SUMMARY["outputs"]["dataset_summary_csv"])
if not DATASET_SUMMARY_CSV_PATH.is_file():
    raise FileNotFoundError(
        f"Stage 3's dataset_summary.csv not found at {DATASET_SUMMARY_CSV_PATH}. "
        f"Stage 4 consumes Stage 3's cleaned/filtered manifest verbatim and never "
        f"rebuilds it from the raw test_manifest.csv."
    )

STAGE_DIR = OUTPUT_ROOT / "stage04_inference_engine"
STAGE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = OUTPUT_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

_PROGRESS_PATH = STAGE_DIR / "_inference_progress.json"
_PARTIAL_PATH = STAGE_DIR / "_partial_predictions.pt"

# -----------------------------------------------------------------------------
# 4.1  Logging / timing (identical pattern to Stages 1-3, reproduced for
#      per-stage independence)
# -----------------------------------------------------------------------------
def setup_logging(log_dir: Path, stage_name: str) -> logging.Logger:
    logger = logging.getLogger(f"sprint04_eval.{stage_name}")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False
    fmt = logging.Formatter(fmt="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
                             datefmt="%Y-%m-%d %H:%M:%S")
    console = logging.StreamHandler(sys.stdout); console.setFormatter(fmt)
    logger.addHandler(console)
    fh = logging.FileHandler(log_dir / f"{stage_name}.log", mode="a"); fh.setFormatter(fmt)
    logger.addHandler(fh)
    return logger

logger = setup_logging(LOG_DIR, "stage04_inference_engine")

class StageTimer:
    def __init__(self, stage_name: str, log: logging.Logger):
        self.stage_name, self.log, self.t0, self.warnings = stage_name, log, None, []
    def warn(self, msg: str) -> None:
        self.warnings.append(msg); self.log.warning(msg)
    def __enter__(self):
        self.t0 = time.perf_counter()
        self.log.info(f"START  stage={self.stage_name}")
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        dur = time.perf_counter() - self.t0
        status = "FAILED" if exc_type else "FINISH"
        self.log.info(f"{status} stage={self.stage_name} duration_sec={dur:.2f} warnings={len(self.warnings)}")
        return False

def log_resource_usage(log: logging.Logger, tag: str) -> Dict[str, float]:
    usage = {}
    if torch.cuda.is_available():
        usage["gpu_allocated_mb"] = round(torch.cuda.memory_allocated() / 1024**2, 2)
        usage["gpu_reserved_mb"] = round(torch.cuda.memory_reserved() / 1024**2, 2)
        usage["gpu_peak_allocated_mb"] = round(torch.cuda.max_memory_allocated() / 1024**2, 2)
    try:
        import resource
        usage["cpu_maxrss_mb"] = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1)
    except Exception:
        pass
    log.info(f"resource_usage tag={tag} {usage}")
    return usage


class ValidationLedger:
    """Accumulates PASS/FAIL engineering checks (section 11). Every check is
    logged immediately; hard-required checks raise on failure (never silently
    continue), soft/informational checks are recorded but don't abort."""

    def __init__(self, log: logging.Logger):
        self.log = log
        self.results: List[Dict[str, Any]] = []

    def check(self, name: str, passed: bool, detail: str = "", hard: bool = True) -> None:
        status = "PASS" if passed else "FAIL"
        entry = {"check": name, "status": status, "detail": detail}
        self.results.append(entry)
        (self.log.info if passed else self.log.error)(f"validation check={name} status={status} detail={detail}")
        if hard and not passed:
            raise RuntimeError(f"Stage 4 aborted — validation check {name!r} FAILED: {detail}")

    def all_passed(self) -> bool:
        return all(r["status"] == "PASS" for r in self.results)


# =============================================================================
# 4.2  Reopen validated artifacts (never rediscover — paths came from Stage 2)
# =============================================================================
def load_json_strict(path: Path) -> Dict[str, Any]:
    try:
        with open(path) as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(f"Corrupted JSON at {path}: {e}") from e


def reopen_stage4_inputs(log: logging.Logger) -> Dict[str, Any]:
    training_summary = load_json_strict(RESOLVED_PATHS["training_summary"])
    checkpoint_summary = load_json_strict(RESOLVED_PATHS["checkpoint_summary"])
    best_model_metadata = load_json_strict(RESOLVED_PATHS["best_model_metadata"])
    disease_registry = load_json_strict(RESOLVED_PATHS["disease_registry"])
    log.info(f"reopened training_summary path={RESOLVED_PATHS['training_summary']}")
    log.info(f"reopened checkpoint_summary path={RESOLVED_PATHS['checkpoint_summary']}")
    log.info(f"reopened best_model_metadata path={RESOLVED_PATHS['best_model_metadata']}")
    log.info(f"reopened disease_registry path={RESOLVED_PATHS['disease_registry']}")
    return {
        "training_summary": training_summary,
        "checkpoint_summary": checkpoint_summary,
        "best_model_metadata": best_model_metadata,
        "disease_registry": disease_registry,
    }


# =============================================================================
# 4.3  Reload Stage 3's cleaned/filtered evaluation manifest (never rebuild it)
# =============================================================================
def load_stage3_manifest(csv_path: Path, expected_n: int, log: logging.Logger) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    if "label_vector" not in df.columns:
        raise ValueError(f"{csv_path} is missing the 'label_vector' column written by Stage 3.")
    df["label_vector"] = df["label_vector"].map(json.loads)
    if len(df) != expected_n:
        raise ValueError(
            f"dataset_summary.csv row count ({len(df)}) does not match "
            f"dataset_summary.json n_samples_usable ({expected_n}). Stage 3's own "
            f"artifacts are inconsistent with each other — rerun Stage 3."
        )
    missing_files = [p for p in df["resolved_image_path"] if not Path(p).is_file()]
    if missing_files:
        raise FileNotFoundError(
            f"{len(missing_files)} image path(s) recorded in dataset_summary.csv no "
            f"longer exist on disk (first few: {missing_files[:5]}). Dataset drifted "
            f"since Stage 3 ran — rerun Stage 3."
        )
    log.info(f"stage3_manifest_loaded rows={len(df)} path={csv_path}")
    return df


def build_eval_transform(cfg: Dict[str, Any], log: logging.Logger) -> Tuple[T.Compose, Tuple[int, int]]:
    """Reconstructs the exact eval transform Stage 3 resolved and recorded in
    dataset_summary.json['transform'] — never re-derives it independently, to
    guarantee pixel-for-pixel identical preprocessing between dataset
    validation and actual inference."""
    size = cfg["image_size"]
    hw = (int(size[0]), int(size[1])) if isinstance(size, (list, tuple)) else (int(size), int(size))
    mean, std = cfg["mean"], cfg["std"]
    interp_map = {
        "bilinear": T.InterpolationMode.BILINEAR,
        "bicubic": T.InterpolationMode.BICUBIC,
        "nearest": T.InterpolationMode.NEAREST,
    }
    interp_name = cfg.get("interpolation", "bilinear")
    if interp_name not in interp_map:
        raise ValueError(f"Unsupported interpolation {interp_name!r} recorded in dataset_summary.json.")
    transform = T.Compose([
        T.Resize(hw, interpolation=interp_map[interp_name]),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])
    log.info(f"eval_transform reconstructed image_size={hw} mean={mean} std={std} "
             f"interpolation={interp_name} source={cfg.get('config_source', 'stage3_dataset_summary')}")
    return transform, hw


class InferenceDataset(Dataset):
    """Map-style dataset over Stage 3's frozen, already-validated evaluation
    manifest. Order is fixed (matches dataset_summary.csv row order) and never
    shuffled — required for stable sample_index <-> prediction alignment."""

    def __init__(self, df: pd.DataFrame, transform: T.Compose):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.has_patient_id = "patient_id" in self.df.columns

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        try:
            with Image.open(row["resolved_image_path"]) as im:
                tensor = self.transform(im.convert("RGB"))
        except Exception as e:
            raise RuntimeError(
                f"Failed to load image at idx={idx} image_index={row['image_index']} "
                f"path={row['resolved_image_path']}: {e}"
            ) from e
        label = torch.tensor(row["label_vector"], dtype=torch.float32)
        patient_id = str(row["patient_id"]) if self.has_patient_id and pd.notna(row["patient_id"]) else ""
        return {
            "image": tensor,
            "label": label,
            "sample_index": idx,
            "image_index": str(row["image_index"]),
            "image_path": str(row["resolved_image_path"]),
            "patient_id": patient_id,
        }


# =============================================================================
# 4.4  Model architecture reconstruction + checkpoint loading
#      (mirrors Stage 2's factory logic; duplicated deliberately for
#      per-stage independence, and extended with the deeper validation
#      Stage 4 owns: shape mismatches, parameter counts, checkpoint version,
#      metadata compatibility)
# =============================================================================
@dataclass(frozen=True)
class BackboneSpec:
    timm_name: str
    family: str
    feature_dim: int
    classifier_attr: str
    status: str

BACKBONE_REGISTRY: Dict[str, BackboneSpec] = {
    "densenet121": BackboneSpec("densenet121", "DenseNet", 1024, "classifier", "ACTIVE"),
}

def _get_module_by_path(module: nn.Module, path: str) -> nn.Module:
    obj = module
    for part in path.split("."):
        obj = getattr(obj, part)
    return obj

def _set_module_by_path(module: nn.Module, path: str, new_submodule: nn.Module) -> None:
    parts = path.split(".")
    obj = module
    for part in parts[:-1]:
        obj = getattr(obj, part)
    setattr(obj, parts[-1], new_submodule)

def get_backbone(backbone_name: str, pretrained: bool = False) -> nn.Module:
    spec = BACKBONE_REGISTRY[backbone_name]
    return timm.create_model(spec.timm_name, pretrained=pretrained)

def replace_classifier(model: nn.Module, backbone_name: str, num_classes: int, dropout: float = 0.3):
    spec = BACKBONE_REGISTRY[backbone_name]
    old_head = _get_module_by_path(model, spec.classifier_attr)
    in_features = old_head.in_features
    new_head = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_features, num_classes))
    _set_module_by_path(model, spec.classifier_attr, new_head)
    return model, in_features

class ChestXrayClassifier(nn.Module):
    def __init__(self, backbone: nn.Module, backbone_name: str, num_classes: int) -> None:
        super().__init__()
        self.backbone = backbone
        self.backbone_name = backbone_name
        self.num_classes = num_classes
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)

def build_model_for_eval(backbone_name: str, num_classes: int, dropout: float) -> ChestXrayClassifier:
    """pretrained=False deliberately: checkpoint weights fully overwrite the
    backbone, so downloading ImageNet weights would be wasted I/O and has no
    effect on the loaded model."""
    raw_model = get_backbone(backbone_name, pretrained=False)
    raw_model, _ = replace_classifier(raw_model, backbone_name, num_classes, dropout)
    return ChestXrayClassifier(raw_model, backbone_name, num_classes)


def validate_architecture_config(inputs: Dict[str, Any], vl: "ValidationLedger") -> Tuple[str, int, float]:
    ts_model = inputs["training_summary"]["model"]
    backbone_name = ts_model["backbone"]
    num_classes = ts_model["num_classes"]
    dropout = ts_model["dropout"]

    vl.check("architecture_backbone_registered", backbone_name in BACKBONE_REGISTRY,
              detail=f"backbone={backbone_name}")
    vl.check("architecture_num_classes_positive", isinstance(num_classes, int) and num_classes > 0,
              detail=f"num_classes={num_classes}")
    vl.check("architecture_dropout_valid", isinstance(dropout, (int, float)) and 0.0 <= dropout < 1.0,
              detail=f"dropout={dropout}")
    return backbone_name, num_classes, float(dropout)


def extract_state_dict(checkpoint_obj: Any) -> Tuple[Dict[str, torch.Tensor], Dict[str, Any]]:
    """Returns (state_dict, sidecar_metadata) — sidecar_metadata captures any
    non-tensor top-level keys (epoch, val_loss, etc.) the training pipeline
    may have bundled into the checkpoint, used for the metadata cross-check."""
    if isinstance(checkpoint_obj, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            if key in checkpoint_obj and isinstance(checkpoint_obj[key], dict):
                sidecar = {k: v for k, v in checkpoint_obj.items() if k != key}
                return checkpoint_obj[key], sidecar
        if all(torch.is_tensor(v) for v in checkpoint_obj.values()):
            return checkpoint_obj, {}
    raise ValueError(
        "Could not locate a model state_dict inside the checkpoint object. "
        f"Top-level type={type(checkpoint_obj)}, "
        f"keys={list(checkpoint_obj.keys()) if isinstance(checkpoint_obj, dict) else 'N/A'}"
    )


_STATE_DICT_TRANSFORMS = [
    ("as-is", lambda sd: sd),
    ("strip 'module.' (DataParallel)", lambda sd: {k.replace("module.", "", 1): v for k, v in sd.items()}),
    ("strip 'backbone.' prefix", lambda sd: {k.replace("backbone.", "", 1): v for k, v in sd.items()}),
]


def load_and_validate_checkpoint(ckpt_path: Path, model: nn.Module, inputs: Dict[str, Any],
                                  vl: "ValidationLedger", timer: "StageTimer",
                                  log: logging.Logger) -> Dict[str, Any]:
    if not ckpt_path.is_file():
        raise FileNotFoundError(f"Checkpoint not found at {ckpt_path} (from Stage 2's resolved_paths). "
                                 f"Stage 4 never rediscovers checkpoints.")

    try:
        raw = torch.load(ckpt_path, map_location="cpu", weights_only=True)
        load_mode = "weights_only=True"
    except Exception as e:
        log.warning(f"weights_only=True load failed ({e}); retrying with weights_only=False "
                    f"(trusted first-party checkpoint from our own Sprint04 pipeline).")
        raw = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        load_mode = "weights_only=False (fallback)"

    state_dict, sidecar = extract_state_dict(raw)
    model_sd = model.state_dict()

    # --- reuse Stage 2's already-resolved load strategy where available, but
    #     independently re-derive the diagnostics (missing/unexpected/shape
    #     mismatches/param counts) rather than trusting its verdict verbatim.
    preferred_label = STAGE2_REPORT.get("checkpoint_compatibility", {}).get("load_transform_used")
    ordered_transforms = _STATE_DICT_TRANSFORMS
    if preferred_label:
        ordered_transforms = sorted(_STATE_DICT_TRANSFORMS, key=lambda t: t[0] != preferred_label)

    chosen = None
    diag = None
    attempts = []
    for label, transform_fn in ordered_transforms:
        transformed = transform_fn(state_dict)
        missing = [k for k in model_sd if k not in transformed]
        unexpected = [k for k in transformed if k not in model_sd]
        shape_mismatches = [
            {"key": k, "model_shape": list(model_sd[k].shape), "checkpoint_shape": list(transformed[k].shape)}
            for k in model_sd if k in transformed and model_sd[k].shape != transformed[k].shape
        ]
        attempts.append({"transform": label, "missing": len(missing), "unexpected": len(unexpected),
                          "shape_mismatches": len(shape_mismatches)})
        if not missing and not unexpected and not shape_mismatches:
            chosen = (label, transformed)
            diag = {"missing_keys": missing, "unexpected_keys": unexpected, "shape_mismatches": shape_mismatches}
            break

    vl.check("checkpoint_transform_resolved", chosen is not None,
              detail=f"attempts={attempts}")
    label, transformed_sd = chosen

    model.load_state_dict(transformed_sd, strict=True)
    param_count_loaded = sum(v.numel() for v in transformed_sd.values())
    param_count_model = sum(v.numel() for v in model_sd.values())

    vl.check("checkpoint_no_missing_tensors", len(diag["missing_keys"]) == 0, detail=str(diag["missing_keys"][:10]))
    vl.check("checkpoint_no_unexpected_tensors", len(diag["unexpected_keys"]) == 0, detail=str(diag["unexpected_keys"][:10]))
    vl.check("checkpoint_no_shape_mismatches", len(diag["shape_mismatches"]) == 0, detail=str(diag["shape_mismatches"][:5]))
    vl.check("checkpoint_param_count_matches", param_count_loaded == param_count_model,
              detail=f"loaded={param_count_loaded} model={param_count_model}")

    # --- checkpoint version cross-check (informational, non-fatal) ----------
    ts_pipeline_version = inputs["training_summary"].get("pipeline_version")
    ckpt_version = sidecar.get("pipeline_version") or sidecar.get("version")
    if ckpt_version is not None and ts_pipeline_version is not None and ckpt_version != ts_pipeline_version:
        timer.warn(f"Checkpoint sidecar pipeline_version={ckpt_version} != "
                   f"training_summary.json pipeline_version={ts_pipeline_version}.")
    version_note = (f"checkpoint={ckpt_version} vs training_summary={ts_pipeline_version}"
                    if ckpt_version is not None else
                    f"checkpoint carries no explicit version field; training_summary.json "
                    f"pipeline_version={ts_pipeline_version} treated as authoritative.")
    log.info(f"checkpoint_version_check {version_note}")

    # --- metadata compatibility cross-check (informational, non-fatal) ------
    best_meta = inputs["best_model_metadata"]
    ckpt_summary = inputs["checkpoint_summary"]
    metadata_cross_check = {}
    for field_name, source_dict, source_key in [
        ("epoch", sidecar, "epoch"),
        ("val_loss", sidecar, "val_loss"),
        ("val_macro_auroc", sidecar, "val_macro_auroc"),
    ]:
        ckpt_val = source_dict.get(source_key)
        meta_val = best_meta.get(field_name)
        match = (ckpt_val == meta_val) if ckpt_val is not None and meta_val is not None else None
        metadata_cross_check[field_name] = {"checkpoint_sidecar": ckpt_val, "best_model_metadata": meta_val, "match": match}
        if match is False:
            timer.warn(f"Checkpoint sidecar {field_name}={ckpt_val} != best_model_metadata.{field_name}={meta_val}.")

    log.info(f"checkpoint_loaded transform={label} load_mode={load_mode} "
             f"param_count={param_count_loaded} metadata_cross_check={metadata_cross_check}")

    return {
        "checkpoint_path": str(ckpt_path),
        "load_mode": load_mode,
        "transform_used": label,
        "attempts": attempts,
        "param_count_loaded": param_count_loaded,
        "param_count_model": param_count_model,
        "checkpoint_version_note": version_note,
        "metadata_cross_check": metadata_cross_check,
        "best_model_metadata_reference": {
            "epoch": best_meta.get("epoch"),
            "val_loss": best_meta.get("val_loss"),
            "val_macro_auroc": best_meta.get("val_macro_auroc"),
        },
        "checkpoint_summary_reference": {
            "best_epoch": ckpt_summary.get("best_epoch"),
            "should_stop": ckpt_summary.get("should_stop"),
        },
    }


# =============================================================================
# 4.5  Device / precision setup
# =============================================================================
def select_device(log: logging.Logger) -> Tuple[torch.device, Dict[str, Any]]:
    info: Dict[str, Any] = {}
    if torch.cuda.is_available():
        device = torch.device("cuda")
        idx = torch.cuda.current_device()
        props = torch.cuda.get_device_properties(idx)
        info.update({
            "backend": "cuda", "device_name": props.name,
            "total_memory_gb": round(props.total_memory / (1024 ** 3), 2),
            "cuda_capability": f"{props.major}.{props.minor}",
        })
    else:
        device = torch.device("cpu")
        info.update({"backend": "cpu", "device_name": platform.processor() or "unknown-cpu"})
    log.info(f"device_selected={device} info={info}")
    return device, info


def resolve_amp_policy(device: torch.device, training_summary: Dict[str, Any], log: logging.Logger) -> Dict[str, Any]:
    requested = bool(training_summary.get("hyperparameters", {}).get("use_amp", False))
    enabled = requested and device.type == "cuda"
    if requested and device.type != "cuda":
        log.warning("training used AMP but no CUDA device is available here; AMP gracefully disabled for CPU inference.")
    amp_dtype = torch.float16 if device.type == "cuda" else torch.bfloat16
    log.info(f"amp_policy requested={requested} enabled={enabled} dtype={amp_dtype}")
    return {"requested": requested, "enabled": enabled, "dtype": amp_dtype}


def reconfirm_determinism(training_summary: Dict[str, Any], log: logging.Logger) -> Dict[str, Any]:
    seed = training_summary.get("hyperparameters", {}).get("random_seed", 42)
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    info = {"seed": seed, "cudnn_deterministic": True, "cudnn_benchmark": False}
    log.info(f"determinism_reconfirmed {info}")
    return info


# =============================================================================
# 4.6  Batched inference (predictor.py-equivalent)
# =============================================================================
class PredictionAccumulator:
    """Buffers per-sample results as plain Python / numpy objects (never as
    live GPU tensors) so memory stays bounded regardless of dataset size.
    Periodically checkpoints itself to disk so a killed kernel can resume
    without repeating already-completed batches (resume-safe requirement)."""

    def __init__(self, checkpoint_every_n_batches: int = 25):
        self.sample_index: List[int] = []
        self.image_index: List[str] = []
        self.image_path: List[str] = []
        self.patient_id: List[str] = []
        self.ground_truth: List[List[int]] = []
        self.logits: List[List[float]] = []
        self.probabilities: List[List[float]] = []
        self.predictions: List[List[int]] = []
        self.batch_perf: List[Dict[str, Any]] = []
        self.completed_samples = 0
        self.checkpoint_every_n_batches = checkpoint_every_n_batches

    def add_batch(self, batch: Dict[str, Any], logits: torch.Tensor, probs: torch.Tensor,
                  preds: torch.Tensor, perf: Dict[str, Any]) -> None:
        n = logits.shape[0]
        self.sample_index.extend(batch["sample_index"].tolist() if torch.is_tensor(batch["sample_index"])
                                  else list(batch["sample_index"]))
        self.image_index.extend(list(batch["image_index"]))
        self.image_path.extend(list(batch["image_path"]))
        self.patient_id.extend(list(batch["patient_id"]))
        self.ground_truth.extend(batch["label"].cpu().numpy().astype(int).tolist())
        self.logits.extend(logits.cpu().numpy().astype(float).tolist())
        self.probabilities.extend(probs.cpu().numpy().astype(float).tolist())
        self.predictions.extend(preds.cpu().numpy().astype(int).tolist())
        self.batch_perf.append(perf)
        self.completed_samples += n

    def save_checkpoint(self, path: Path) -> None:
        torch.save({
            "sample_index": self.sample_index, "image_index": self.image_index,
            "image_path": self.image_path, "patient_id": self.patient_id,
            "ground_truth": self.ground_truth, "logits": self.logits,
            "probabilities": self.probabilities, "predictions": self.predictions,
            "batch_perf": self.batch_perf, "completed_samples": self.completed_samples,
        }, path)

    @classmethod
    def load_checkpoint(cls, path: Path) -> "PredictionAccumulator":
        state = torch.load(path, map_location="cpu", weights_only=False)
        acc = cls()
        for k in ["sample_index", "image_index", "image_path", "patient_id", "ground_truth",
                  "logits", "probabilities", "predictions", "batch_perf"]:
            setattr(acc, k, state[k])
        acc.completed_samples = state["completed_samples"]
        return acc


def check_reproducibility(model: nn.Module, dataset: InferenceDataset, device: torch.device,
                           amp_policy: Dict[str, Any], log: logging.Logger) -> bool:
    """Runs the same single sample through the model twice under
    inference_mode + eval() and requires bit-identical logits. With dropout
    disabled and a deterministic transform, any divergence indicates a real
    non-determinism bug (e.g. missing eval() call) rather than expected
    stochasticity."""
    sample = dataset[0]
    x = sample["image"].unsqueeze(0).to(device)
    outs = []
    for _ in range(2):
        with torch.inference_mode():
            with torch.autocast(device_type=device.type, enabled=amp_policy["enabled"], dtype=amp_policy["dtype"]):
                out = model(x)
        outs.append(out.float().cpu())
    identical = torch.equal(outs[0], outs[1])
    log.info(f"reproducibility_check identical_logits={identical}")
    return identical


def run_inference_loop(model: nn.Module, dataset: InferenceDataset, loader_cfg: Dict[str, Any],
                        device: torch.device, amp_policy: Dict[str, Any], acc: PredictionAccumulator,
                        start_index: int, log: logging.Logger) -> Dict[str, Any]:
    remaining = Subset(dataset, list(range(start_index, len(dataset)))) if start_index > 0 else dataset

    kwargs = dict(batch_size=loader_cfg["batch_size"], shuffle=False, drop_last=False,
                  num_workers=loader_cfg["num_workers"], pin_memory=loader_cfg["pin_memory"])
    if loader_cfg["num_workers"] > 0:
        kwargs["persistent_workers"] = loader_cfg["persistent_workers"]
    loader = DataLoader(remaining, **kwargs)

    total_batches = len(loader)
    log_every = max(1, total_batches // 10)
    t_loop_start = time.perf_counter()

    for batch_idx, batch in enumerate(loader):
        t0 = time.perf_counter()
        images = batch["image"].to(device, non_blocking=loader_cfg["pin_memory"])

        with torch.inference_mode():
            with torch.autocast(device_type=device.type, enabled=amp_policy["enabled"], dtype=amp_policy["dtype"]):
                logits = model(images)
            logits = logits.float()
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).int()

        batch_latency = time.perf_counter() - t0
        gpu_mem_mb = round(torch.cuda.memory_allocated() / 1024**2, 2) if device.type == "cuda" else 0.0
        perf = {
            "batch_index": batch_idx,
            "batch_size": images.shape[0],
            "latency_sec": round(batch_latency, 4),
            "throughput_samples_per_sec": round(images.shape[0] / batch_latency, 2) if batch_latency > 0 else None,
            "gpu_allocated_mb": gpu_mem_mb,
        }
        acc.add_batch(batch, logits, probs, preds, perf)

        if batch_idx % log_every == 0 or batch_idx == total_batches - 1:
            log.info(f"batch_progress {batch_idx + 1}/{total_batches} "
                     f"samples_done={acc.completed_samples}/{len(dataset)} "
                     f"latency_sec={perf['latency_sec']} throughput={perf['throughput_samples_per_sec']} "
                     f"gpu_mb={gpu_mem_mb}")

        if (batch_idx + 1) % acc.checkpoint_every_n_batches == 0:
            acc.save_checkpoint(_PARTIAL_PATH)
            with open(_PROGRESS_PATH, "w") as f:
                json.dump({"completed_samples": acc.completed_samples, "total_samples": len(dataset)}, f)
            log.info(f"resume_checkpoint_written completed_samples={acc.completed_samples}")

    total_time = time.perf_counter() - t_loop_start
    del loader
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return {"total_batches_this_run": total_batches, "total_time_sec": total_time}


# =============================================================================
# 4.7  Artifact generation (artifact_manager.py-equivalent)
# =============================================================================
def build_predictions_dataframe(acc: PredictionAccumulator) -> pd.DataFrame:
    return pd.DataFrame({
        "sample_index": acc.sample_index,
        "image_index": acc.image_index,
        "image_path": acc.image_path,
        "patient_id": acc.patient_id,
        "ground_truth": acc.ground_truth,
        "logits": acc.logits,
        "probabilities": acc.probabilities,
        "predictions": acc.predictions,
    })


def write_prediction_artifacts(df: pd.DataFrame, raw_logits: torch.Tensor, class_names: List[str],
                                threshold: float, log: logging.Logger) -> Dict[str, str]:
    parquet_path = STAGE_DIR / "predictions.parquet"
    try:
        df.to_parquet(parquet_path, index=False)
    except ImportError as e:
        raise ImportError(
            "Writing predictions.parquet requires pyarrow (or fastparquet), which is "
            "not installed. Install with `pip install pyarrow`."
        ) from e
    _round_trip = pd.read_parquet(parquet_path)
    if len(_round_trip) != len(df):
        raise ValueError("predictions.parquet failed round-trip row-count check.")

    csv_path = STAGE_DIR / "predictions.csv"
    csv_df = df.copy()
    for col in ["ground_truth", "logits", "probabilities", "predictions"]:
        csv_df[col] = csv_df[col].map(json.dumps)
    csv_df.to_csv(csv_path, index=False)

    raw_logits_path = STAGE_DIR / "raw_logits.pt"
    torch.save({"logits": raw_logits, "class_names": class_names,
                "sample_index": df["sample_index"].tolist()}, raw_logits_path)

    log.info(f"artifacts_written parquet={parquet_path} csv={csv_path} raw_logits={raw_logits_path}")
    return {"predictions_parquet": str(parquet_path), "predictions_csv": str(csv_path),
            "raw_logits_pt": str(raw_logits_path)}


# =============================================================================
# 4.8  Full-array validations (NaN / Inf / shape checks against the complete
#      in-memory result, run once after the loop finishes)
# =============================================================================
def validate_result_arrays(df: pd.DataFrame, class_names: List[str], dataset_len: int,
                            vl: "ValidationLedger") -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    n_classes = len(class_names)
    logits_arr = np.array(df["logits"].tolist(), dtype=np.float64)
    probs_arr = np.array(df["probabilities"].tolist(), dtype=np.float64)
    preds_arr = np.array(df["predictions"].tolist(), dtype=np.int64)
    gt_arr = np.array(df["ground_truth"].tolist(), dtype=np.int64)

    vl.check("prediction_count_equals_dataset_size", len(df) == dataset_len,
              detail=f"predictions={len(df)} dataset={dataset_len}")
    vl.check("logits_shape_correct", logits_arr.shape == (dataset_len, n_classes),
              detail=f"shape={logits_arr.shape} expected=({dataset_len}, {n_classes})")
    vl.check("probabilities_shape_correct", probs_arr.shape == (dataset_len, n_classes),
              detail=f"shape={probs_arr.shape} expected=({dataset_len}, {n_classes})")
    vl.check("predictions_shape_correct", preds_arr.shape == (dataset_len, n_classes),
              detail=f"shape={preds_arr.shape} expected=({dataset_len}, {n_classes})")
    vl.check("ground_truth_shape_correct", gt_arr.shape == (dataset_len, n_classes),
              detail=f"shape={gt_arr.shape} expected=({dataset_len}, {n_classes})")
    vl.check("no_nan_logits", not np.isnan(logits_arr).any(), detail=f"nan_count={int(np.isnan(logits_arr).sum())}")
    vl.check("no_nan_probabilities", not np.isnan(probs_arr).any(), detail=f"nan_count={int(np.isnan(probs_arr).sum())}")
    vl.check("no_infinite_values", not np.isinf(logits_arr).any() and not np.isinf(probs_arr).any(),
              detail=f"inf_logits={int(np.isinf(logits_arr).sum())} inf_probs={int(np.isinf(probs_arr).sum())}")
    vl.check("probabilities_in_unit_range", bool((probs_arr >= 0).all() and (probs_arr <= 1).all()),
              detail=f"min={probs_arr.min():.6f} max={probs_arr.max():.6f}")
    return logits_arr, probs_arr, preds_arr, gt_arr


# =============================================================================
# 4.9  Orchestration (infer.py-equivalent)
# =============================================================================
def run_stage4() -> Dict[str, Any]:
    with StageTimer("stage04_inference_engine", logger) as timer:
        vl = ValidationLedger(logger)
        log_resource_usage(logger, "stage04_start")

        inputs = reopen_stage4_inputs(logger)
        class_names = STAGE3_SUMMARY["class_names"]
        n_classes_expected = inputs["training_summary"]["model"]["num_classes"]
        vl.check("class_names_count_matches_training_summary", len(class_names) == n_classes_expected,
                  detail=f"class_names={len(class_names)} training_summary={n_classes_expected}")

        manifest_df = load_stage3_manifest(DATASET_SUMMARY_CSV_PATH, STAGE3_SUMMARY["n_samples_usable"], logger)
        transform, expected_hw = build_eval_transform(STAGE3_SUMMARY["transform"], logger)
        dataset = InferenceDataset(manifest_df, transform)
        logger.info(f"inference_dataset_built n_samples={len(dataset)} image_size={expected_hw}")
        vl.check("dataset_non_empty", len(dataset) > 0, detail=f"n_samples={len(dataset)}")

        device, device_info = select_device(logger)
        amp_policy = resolve_amp_policy(device, inputs["training_summary"], logger)
        determinism_info = reconfirm_determinism(inputs["training_summary"], logger)

        backbone_name, num_classes, dropout = validate_architecture_config(inputs, vl)
        model = build_model_for_eval(backbone_name, num_classes, dropout)
        vl.check("model_loaded", model is not None, detail=f"backbone={backbone_name} num_classes={num_classes}")

        checkpoint_info = load_and_validate_checkpoint(
            RESOLVED_PATHS["best_model"], model, inputs, vl, timer, logger)
        vl.check("checkpoint_restored", True, detail=checkpoint_info["transform_used"])

        model.to(device)
        model.eval()
        for p in model.parameters():
            p.requires_grad_(False)
        vl.check("model_in_eval_mode", not model.training, detail="model.training==False")
        vl.check("gradients_disabled", all(not p.requires_grad for p in model.parameters()),
                  detail="all parameters requires_grad=False")

        reproducible = check_reproducibility(model, dataset, device, amp_policy, logger)
        vl.check("inference_reproducible", reproducible, detail="two forward passes on sample[0] identical")
        vl.check("deterministic_execution", determinism_info["cudnn_deterministic"] and not determinism_info["cudnn_benchmark"],
                  detail=str(determinism_info))

        log_resource_usage(logger, "stage04_pre_inference")

        # --- resume support: pick up an in-progress run if one exists --------
        acc = PredictionAccumulator(checkpoint_every_n_batches=25)
        start_index = 0
        if _PROGRESS_PATH.is_file() and _PARTIAL_PATH.is_file():
            with open(_PROGRESS_PATH) as f:
                progress = json.load(f)
            if progress.get("total_samples") == len(dataset):
                acc = PredictionAccumulator.load_checkpoint(_PARTIAL_PATH)
                start_index = acc.completed_samples
                timer.warn(f"Resuming inference from a previous partial run: "
                           f"{start_index}/{len(dataset)} samples already completed.")
            else:
                logger.info("Stale resume checkpoint found (dataset size differs) — starting fresh.")

        loader_cfg = STAGE3_SUMMARY["dataloader_config"]
        loop_stats = run_inference_loop(model, dataset, loader_cfg, device, amp_policy, acc, start_index, logger)

        log_resource_usage(logger, "stage04_post_inference")

        df = build_predictions_dataframe(acc)
        logits_arr, probs_arr, preds_arr, gt_arr = validate_result_arrays(df, class_names, len(dataset), vl)
        raw_logits_tensor = torch.tensor(logits_arr, dtype=torch.float32)

        written = write_prediction_artifacts(df, raw_logits_tensor, class_names, threshold=0.5, log=logger)
        vl.check("parquet_written", Path(written["predictions_parquet"]).is_file())
        vl.check("csv_written", Path(written["predictions_csv"]).is_file())

        # clear resume checkpoint now that the run completed successfully
        for p in (_PROGRESS_PATH, _PARTIAL_PATH):
            if p.is_file():
                p.unlink()

        perf_list = acc.batch_perf
        latencies = [b["latency_sec"] for b in perf_list if b["latency_sec"] is not None]
        throughputs = [b["throughput_samples_per_sec"] for b in perf_list if b["throughput_samples_per_sec"]]
        if device.type == "cuda":
            peak_gpu_mb = round(torch.cuda.max_memory_allocated() / 1024**2, 2)
        else:
            peak_gpu_mb = max((b["gpu_allocated_mb"] for b in perf_list), default=0.0)
        gpu_utilization_pct = None
        if device.type == "cuda":
            try:
                gpu_utilization_pct = torch.cuda.utilization()
            except Exception as e:
                logger.info(f"gpu_utilization unavailable (requires pynvml): {e}")
        cpu_mem = log_resource_usage(logger, "stage04_end")

        prediction_metadata = {
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "class_names": class_names,
            "num_classes": len(class_names),
            "threshold": 0.5,
            "columns": {
                "sample_index": "int, 0-based position in Stage 3's frozen manifest order",
                "image_index": "str, original NIH ChestXray14 image identifier",
                "image_path": "str, resolved absolute path used for inference",
                "patient_id": "str, empty string if not available",
                "ground_truth": f"list[int] len={len(class_names)}, multi-hot label vector, class order matches class_names",
                "logits": f"list[float] len={len(class_names)}, raw model outputs pre-sigmoid",
                "probabilities": f"list[float] len={len(class_names)}, sigmoid(logits)",
                "predictions": f"list[int] len={len(class_names)}, (probabilities >= 0.5).astype(int)",
            },
            "checkpoint": checkpoint_info,
            "n_samples": len(df),
        }
        pm_path = STAGE_DIR / "prediction_metadata.json"
        with open(pm_path, "w") as f:
            json.dump(prediction_metadata, f, indent=2, default=str)

        inference_summary = {
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "device": device_info,
            "amp": {"requested": amp_policy["requested"], "enabled": amp_policy["enabled"]},
            "n_samples": len(df),
            "n_batches_this_run": loop_stats["total_batches_this_run"],
            "resumed_from_sample": start_index if start_index > 0 else None,
            "total_inference_time_sec": round(loop_stats["total_time_sec"], 3),
            "avg_batch_latency_sec": round(float(np.mean(latencies)), 4) if latencies else None,
            "p50_batch_latency_sec": round(float(np.percentile(latencies, 50)), 4) if latencies else None,
            "p95_batch_latency_sec": round(float(np.percentile(latencies, 95)), 4) if latencies else None,
            "samples_per_second_avg": round(float(np.mean(throughputs)), 2) if throughputs else None,
            "peak_gpu_allocated_mb": peak_gpu_mb,
            "gpu_utilization_pct": gpu_utilization_pct,
            "cpu_maxrss_mb": cpu_mem.get("cpu_maxrss_mb"),
            "batch_perf_trace": perf_list,
            "validation_checks": vl.results,
            "all_validation_checks_passed": vl.all_passed(),
            "warnings": timer.warnings,
            "artifacts": {**written, "prediction_metadata": str(pm_path)},
        }
        is_path = STAGE_DIR / "inference_summary.json"
        with open(is_path, "w") as f:
            json.dump(inference_summary, f, indent=2, default=str)

        # Round-trip corruption check on both JSON artifacts. This check is
        # deliberately NOT embedded inside inference_summary.json itself
        # (a file cannot validate its own write before the write completes);
        # it is logged here and re-affirmed in the printed run report instead.
        json_ok = True
        try:
            with open(is_path) as f:
                json.load(f)
            with open(pm_path) as f:
                json.load(f)
        except json.JSONDecodeError as e:
            json_ok = False
            logger.error(f"json_written round-trip check FAILED: {e}")
        if not json_ok:
            raise ValueError("inference_summary.json / prediction_metadata.json failed round-trip validation.")
        logger.info(f"validation check=json_written status=PASS detail=round-trip OK for both JSON artifacts")

        logger.info(f"artifact_written path={is_path}")
        inference_summary["_artifact_path"] = str(is_path)
        inference_summary["_prediction_metadata_path"] = str(pm_path)
        inference_summary["_json_written_check"] = "PASS"
        return inference_summary


INFERENCE_SUMMARY = run_stage4()

print("\n" + "=" * 70)
print("STAGE 4 — INFERENCE ENGINE SUMMARY")
print("=" * 70)
print(f"Device                    : {INFERENCE_SUMMARY['device']['backend']} "
      f"({INFERENCE_SUMMARY['device'].get('device_name', 'n/a')})")
print(f"AMP enabled               : {INFERENCE_SUMMARY['amp']['enabled']}")
print(f"Samples processed         : {INFERENCE_SUMMARY['n_samples']}")
print(f"Batches (this run)        : {INFERENCE_SUMMARY['n_batches_this_run']}")
print(f"Total inference time (s)  : {INFERENCE_SUMMARY['total_inference_time_sec']}")
print(f"Avg batch latency (s)     : {INFERENCE_SUMMARY['avg_batch_latency_sec']}")
print(f"Samples/sec (avg)         : {INFERENCE_SUMMARY['samples_per_second_avg']}")
print(f"Peak GPU memory (MB)      : {INFERENCE_SUMMARY['peak_gpu_allocated_mb']}")
print(f"All validation checks OK  : {INFERENCE_SUMMARY['all_validation_checks_passed']}")
print(f"Warnings                  : {len(INFERENCE_SUMMARY['warnings'])}")
for w in INFERENCE_SUMMARY["warnings"]:
    print(f"  - {w}")
print(f"predictions.parquet       : {INFERENCE_SUMMARY['artifacts']['predictions_parquet']}")
print(f"predictions.csv           : {INFERENCE_SUMMARY['artifacts']['predictions_csv']}")
print(f"raw_logits.pt             : {INFERENCE_SUMMARY['artifacts']['raw_logits_pt']}")
print(f"prediction_metadata.json  : {INFERENCE_SUMMARY['artifacts']['prediction_metadata']}")
print(f"inference_summary.json   : {INFERENCE_SUMMARY['_artifact_path']}")
print("Stage 4 — Production Inference Engine : OK")
print("Stage 4 complete. Waiting for validation before Stage 5.")

2026-07-05 05:34:40 | INFO    | sprint04_eval.stage04_inference_engine | START  stage=stage04_inference_engine
2026-07-05 05:34:40 | INFO    | sprint04_eval.stage04_inference_engine | resource_usage tag=stage04_start {'gpu_allocated_mb': 0.0, 'gpu_reserved_mb': 0.0, 'gpu_peak_allocated_mb': 0.0, 'cpu_maxrss_mb': 1257.6}
2026-07-05 05:34:40 | INFO    | sprint04_eval.stage04_inference_engine | reopened training_summary path=/kaggle/input/datasets/anupsharma1730/visionserveai-sprint04-artifacts/visionserveai/sprint04/training_summary.json
2026-07-05 05:34:40 | INFO    | sprint04_eval.stage04_inference_engine | reopened checkpoint_summary path=/kaggle/input/datasets/anupsharma1730/visionserveai-sprint04-artifacts/visionserveai/sprint04/checkpoints/checkpoint_summary.json
2026-07-05 05:34:40 | INFO    | sprint04_eval.stage04_inference_engine | reopened best_model_metadata path=/kaggle/input/datasets/anupsharma1730/visionserveai-sprint04-artifacts/visionserveai/sprint04/metrics/best_model_me

In [6]:
# =============================================================================
# Sprint04_Evaluation.ipynb
# STAGE 5 — Production Metrics Engine
# -----------------------------------------------------------------------------
# READ-ONLY. Consumes ONLY Stage 1-4 artifacts exactly as written to disk
# (never trusts notebook memory, never reruns inference, never reloads the
# model except to read its own recorded metadata). Computes every numerical
# evaluation metric for the frozen NIH ChestXray14 multi-label test set at
# the fixed operating threshold of 0.5.
#
# Scope discipline (per project rules): metrics ONLY. No visualization, no
# threshold optimization, no GradCAM, no calibration, no ROC/PR images, no
# confusion-matrix images. Those belong to Stage 6+.
#
# Modularity note: this stage is intentionally split into the same
# functional seams it will later become:
#   validators.py    -> Section 5.3 (input/schema/consistency validation)
#   statistics.py     -> Section 5.4 (support / prevalence / distribution)
#   metrics.py        -> Section 5.5-5.7 (AUROC, PR, threshold, multilabel)
#   report_writer.py  -> Section 5.8 (artifact writers)
#   evaluate.py       -> Section 5.9 (orchestration / run_stage5)
# =============================================================================

import os, sys, json, time, gc, logging, warnings
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_fscore_support,
    matthews_corrcoef, balanced_accuracy_score, hamming_loss, jaccard_score,
    label_ranking_average_precision_score, coverage_error, label_ranking_loss,
    accuracy_score, multilabel_confusion_matrix,
)
from sklearn.exceptions import UndefinedMetricWarning

# -----------------------------------------------------------------------------
# 5.0  Re-derive paths independently (do not trust in-memory Stage 1-4 state)
# -----------------------------------------------------------------------------
_KAGGLE_INPUT = Path("/kaggle/input")
_KAGGLE_WORKING = Path("/kaggle/working")
IS_KAGGLE = _KAGGLE_INPUT.is_dir()
INPUT_ROOT = _KAGGLE_INPUT if IS_KAGGLE else Path(os.environ.get("EVAL_INPUT_ROOT", "kaggle_input"))
OUTPUT_ROOT = (_KAGGLE_WORKING / "sprint04_evaluation") if IS_KAGGLE else Path(os.environ.get("EVAL_OUTPUT_ROOT", "sprint04_evaluation"))

STAGE1_SUMMARY_PATH = OUTPUT_ROOT / "stage01_environment" / "environment_summary.json"
STAGE2_SUMMARY_PATH = OUTPUT_ROOT / "stage02_artifact_loader" / "artifact_validation.json"
STAGE3_SUMMARY_PATH = OUTPUT_ROOT / "stage03_dataset_builder" / "dataset_summary.json"
STAGE4_SUMMARY_PATH = OUTPUT_ROOT / "stage04_inference_engine" / "inference_summary.json"

for _label, _p in [("Stage 1", STAGE1_SUMMARY_PATH), ("Stage 2", STAGE2_SUMMARY_PATH),
                    ("Stage 3", STAGE3_SUMMARY_PATH), ("Stage 4", STAGE4_SUMMARY_PATH)]:
    if not _p.is_file():
        raise FileNotFoundError(
            f"{_label} output not found at {_p}. Stage 5 never regenerates prior "
            f"stages, and never runs inference itself — run Stages 1-4 first, in order."
        )

with open(STAGE1_SUMMARY_PATH) as f:
    STAGE1_SUMMARY = json.load(f)
with open(STAGE2_SUMMARY_PATH) as f:
    STAGE2_REPORT = json.load(f)
with open(STAGE3_SUMMARY_PATH) as f:
    STAGE3_SUMMARY = json.load(f)
with open(STAGE4_SUMMARY_PATH) as f:
    STAGE4_SUMMARY = json.load(f)

RESOLVED_PATHS = {k: Path(v) for k, v in STAGE2_REPORT["resolved_paths"].items()}

REQUIRED_FOR_STAGE5 = ["training_summary", "best_model_metadata"]
_missing = [k for k in REQUIRED_FOR_STAGE5 if k not in RESOLVED_PATHS]
if _missing:
    raise FileNotFoundError(
        f"Stage 5 requires resolved_paths entries {_missing} from Stage 2's "
        f"artifact_validation.json, but they are absent. Rerun Stage 2. Stage 5 "
        f"never rediscovers artifacts on its own — this is a hard stop."
    )

# Stage 4's own artifact manifest is the ONLY source of truth for where the
# prediction outputs live — Stage 5 never re-derives these paths itself.
_STAGE4_ARTIFACTS = STAGE4_SUMMARY.get("artifacts", {})
for _key in ["predictions_parquet", "raw_logits_pt", "prediction_metadata"]:
    if _key not in _STAGE4_ARTIFACTS:
        raise FileNotFoundError(
            f"Stage 4's inference_summary.json is missing artifacts.{_key}. "
            f"Stage 5 cannot proceed without a complete Stage 4 artifact manifest."
        )

PREDICTIONS_PARQUET_PATH = Path(_STAGE4_ARTIFACTS["predictions_parquet"])
RAW_LOGITS_PT_PATH = Path(_STAGE4_ARTIFACTS["raw_logits_pt"])
PREDICTION_METADATA_PATH = Path(_STAGE4_ARTIFACTS["prediction_metadata"])

for _label, _p in [("predictions.parquet", PREDICTIONS_PARQUET_PATH),
                    ("raw_logits.pt", RAW_LOGITS_PT_PATH),
                    ("prediction_metadata.json", PREDICTION_METADATA_PATH)]:
    if not _p.is_file():
        raise FileNotFoundError(
            f"Stage 4 artifact {_label} not found at {_p} even though it is "
            f"listed in inference_summary.json. Stage 4 must be rerun."
        )

if not STAGE4_SUMMARY.get("all_validation_checks_passed", False):
    raise RuntimeError(
        "Stage 5 aborted — Stage 4's inference_summary.json reports "
        "all_validation_checks_passed=False. Stage 5 must never compute metrics "
        "on top of an inference run that failed its own engineering validation."
    )

STAGE_DIR = OUTPUT_ROOT / "stage05_metrics_engine"
STAGE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = OUTPUT_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# 5.1  Logging / timing (identical pattern to Stages 1-4, reproduced for
#      per-stage independence)
# -----------------------------------------------------------------------------
def setup_logging(log_dir: Path, stage_name: str) -> logging.Logger:
    logger = logging.getLogger(f"sprint04_eval.{stage_name}")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False
    fmt = logging.Formatter(fmt="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
                             datefmt="%Y-%m-%d %H:%M:%S")
    console = logging.StreamHandler(sys.stdout); console.setFormatter(fmt)
    logger.addHandler(console)
    fh = logging.FileHandler(log_dir / f"{stage_name}.log", mode="a"); fh.setFormatter(fmt)
    logger.addHandler(fh)
    return logger

logger = setup_logging(LOG_DIR, "stage05_metrics_engine")

class StageTimer:
    def __init__(self, stage_name: str, log: logging.Logger):
        self.stage_name, self.log, self.t0, self.warnings = stage_name, log, None, []
    def warn(self, msg: str) -> None:
        self.warnings.append(msg); self.log.warning(msg)
    def __enter__(self):
        self.t0 = time.perf_counter()
        self.log.info(f"START  stage={self.stage_name}")
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        dur = time.perf_counter() - self.t0
        status = "FAILED" if exc_type else "FINISH"
        self.log.info(f"{status} stage={self.stage_name} duration_sec={dur:.2f} warnings={len(self.warnings)}")
        return False

def log_resource_usage(log: logging.Logger, tag: str) -> Dict[str, float]:
    usage = {}
    if torch.cuda.is_available():
        usage["gpu_allocated_mb"] = round(torch.cuda.memory_allocated() / 1024**2, 2)
        usage["gpu_reserved_mb"] = round(torch.cuda.memory_reserved() / 1024**2, 2)
    try:
        import resource
        usage["cpu_maxrss_mb"] = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1)
    except Exception:
        pass
    log.info(f"resource_usage tag={tag} {usage}")
    return usage


class ValidationLedger:
    """Accumulates PASS/FAIL engineering checks. Every check is logged
    immediately; hard-required checks raise on failure (never silently
    continue), soft/informational checks are recorded but don't abort."""

    def __init__(self, log: logging.Logger):
        self.log = log
        self.results: List[Dict[str, Any]] = []

    def check(self, name: str, passed: bool, detail: str = "", hard: bool = True) -> None:
        status = "PASS" if passed else "FAIL"
        entry = {"check": name, "status": status, "detail": detail, "hard": hard}
        self.results.append(entry)
        (self.log.info if passed else self.log.error)(f"validation check={name} status={status} detail={detail}")
        if hard and not passed:
            raise RuntimeError(f"Stage 5 aborted — validation check {name!r} FAILED: {detail}")

    def all_passed(self) -> bool:
        return all(r["status"] == "PASS" for r in self.results)

    def n_passed(self) -> int:
        return sum(1 for r in self.results if r["status"] == "PASS")


# =============================================================================
# 5.2  Reopen validated artifacts (never rediscover — paths came from Stage 2/4)
# =============================================================================
def load_json_strict(path: Path) -> Dict[str, Any]:
    try:
        with open(path) as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(f"Corrupted JSON at {path}: {e}") from e


def reopen_stage5_inputs(log: logging.Logger) -> Dict[str, Any]:
    training_summary = load_json_strict(RESOLVED_PATHS["training_summary"])
    best_model_metadata = load_json_strict(RESOLVED_PATHS["best_model_metadata"])
    prediction_metadata = load_json_strict(PREDICTION_METADATA_PATH)
    log.info(f"reopened training_summary path={RESOLVED_PATHS['training_summary']}")
    log.info(f"reopened best_model_metadata path={RESOLVED_PATHS['best_model_metadata']}")
    log.info(f"reopened prediction_metadata path={PREDICTION_METADATA_PATH}")

    predictions_df = pd.read_parquet(PREDICTIONS_PARQUET_PATH)
    log.info(f"reopened predictions.parquet path={PREDICTIONS_PARQUET_PATH} rows={len(predictions_df)}")

    raw_logits_blob = torch.load(RAW_LOGITS_PT_PATH, weights_only=False)
    log.info(f"reopened raw_logits.pt path={RAW_LOGITS_PT_PATH} "
             f"tensor_shape={tuple(raw_logits_blob['logits'].shape)}")

    return {
        "training_summary": training_summary,
        "best_model_metadata": best_model_metadata,
        "prediction_metadata": prediction_metadata,
        "predictions_df": predictions_df,
        "raw_logits_blob": raw_logits_blob,
    }


# =============================================================================
# 5.3  Validation — structural / schema / cross-artifact consistency
#      Fails loudly (hard=True) on any inconsistency. This is the gate that
#      must pass before a single metric is computed.
# =============================================================================
def validate_inputs(inputs: Dict[str, Any], vl: "ValidationLedger", log: logging.Logger) -> Dict[str, Any]:
    df = inputs["predictions_df"]
    pm = inputs["prediction_metadata"]
    ts = inputs["training_summary"]
    raw = inputs["raw_logits_blob"]

    dataset_class_names = STAGE3_SUMMARY["class_names"]
    pm_class_names = pm["class_names"]
    raw_class_names = list(raw["class_names"])

    dataset_num_classes = STAGE3_SUMMARY["num_classes"]
    pm_num_classes = pm["num_classes"]
    ts_num_classes = ts["model"]["num_classes"]
    registry_num_classes = STAGE2_REPORT["class_count_consistency"]["disease_registry"]

    # --- class order: exact positional equality across every artifact ------
    vl.check(
        "class_order_consistent",
        dataset_class_names == pm_class_names == raw_class_names,
        detail=f"dataset_summary vs prediction_metadata vs raw_logits.pt class name lists "
               f"{'match exactly' if dataset_class_names == pm_class_names == raw_class_names else 'DIFFER'}",
    )

    # --- number of classes: consistent across every recorded source --------
    class_counts = {
        "dataset_summary": dataset_num_classes,
        "prediction_metadata": pm_num_classes,
        "training_summary_model": ts_num_classes,
        "disease_registry_via_stage2": registry_num_classes,
        "raw_logits_class_names": len(raw_class_names),
    }
    vl.check(
        "class_count_matches_registry",
        len(set(class_counts.values())) == 1,
        detail=f"class_counts={class_counts}",
    )
    n_classes = dataset_num_classes
    class_names = dataset_class_names

    # --- prediction count vs Stage 4's own recorded n_samples ---------------
    vl.check(
        "prediction_count_matches_stage4",
        len(df) == pm["n_samples"] == STAGE4_SUMMARY["n_samples"],
        detail=f"predictions.parquet_rows={len(df)} prediction_metadata.n_samples={pm['n_samples']} "
               f"inference_summary.n_samples={STAGE4_SUMMARY['n_samples']}",
    )

    # --- dataset size vs Stage 3's frozen usable-sample count ---------------
    vl.check(
        "dataset_size_matches_stage3",
        len(df) == STAGE3_SUMMARY["n_samples_usable"],
        detail=f"predictions.parquet_rows={len(df)} dataset_summary.n_samples_usable={STAGE3_SUMMARY['n_samples_usable']}",
    )

    # --- parse array columns and check dimensionality ------------------------
    gt_arr = np.array(df["ground_truth"].tolist(), dtype=np.float64)
    probs_arr = np.array(df["probabilities"].tolist(), dtype=np.float64)
    logits_arr = np.array(df["logits"].tolist(), dtype=np.float64)
    preds_arr = np.array(df["predictions"].tolist(), dtype=np.int64)

    vl.check("label_dimensions_correct", gt_arr.shape == (len(df), n_classes),
              detail=f"ground_truth shape={gt_arr.shape} expected=({len(df)}, {n_classes})")
    vl.check("probability_dimensions_correct", probs_arr.shape == (len(df), n_classes),
              detail=f"probabilities shape={probs_arr.shape} expected=({len(df)}, {n_classes})")
    vl.check("logit_dimensions_correct", logits_arr.shape == (len(df), n_classes),
              detail=f"logits shape={logits_arr.shape} expected=({len(df)}, {n_classes})")
    vl.check("prediction_dimensions_correct", preds_arr.shape == (len(df), n_classes),
              detail=f"predictions shape={preds_arr.shape} expected=({len(df)}, {n_classes})")

    # --- ground truth must be strictly binary --------------------------------
    vl.check("ground_truth_is_binary", bool(np.isin(gt_arr, [0, 1]).all()),
              detail=f"unique_values={np.unique(gt_arr).tolist()[:10]}")

    # --- probabilities must lie in the closed unit interval ------------------
    vl.check("probabilities_in_unit_range", bool((probs_arr >= 0).all() and (probs_arr <= 1).all()),
              detail=f"min={probs_arr.min():.6f} max={probs_arr.max():.6f}")

    # --- no NaN / Inf anywhere in the raw prediction artifacts ---------------
    vl.check("no_nan_in_predictions",
              not (np.isnan(gt_arr).any() or np.isnan(probs_arr).any() or np.isnan(logits_arr).any()),
              detail="checked ground_truth, probabilities, logits")
    vl.check("no_inf_in_predictions",
              not (np.isinf(logits_arr).any() or np.isinf(probs_arr).any()),
              detail="checked probabilities, logits")

    # --- raw_logits.pt must agree numerically with predictions.parquet ------
    raw_logits_np = raw["logits"].numpy().astype(np.float64)
    vl.check("raw_logits_shape_matches_parquet", raw_logits_np.shape == logits_arr.shape,
              detail=f"raw_logits.pt shape={raw_logits_np.shape} predictions.parquet shape={logits_arr.shape}")
    logits_allclose = bool(np.allclose(raw_logits_np, logits_arr, atol=1e-5, rtol=1e-4))
    vl.check("raw_logits_values_match_parquet", logits_allclose,
              detail=f"max_abs_diff={np.max(np.abs(raw_logits_np - logits_arr)):.2e}"
                     if raw_logits_np.shape == logits_arr.shape else "shape mismatch, cannot diff")

    # --- stored 'predictions' column must equal thresholding probabilities
    #     at exactly 0.5 (catches any silent drift between Stage 4's stored
    #     threshold and what this stage assumes) -----------------------------
    threshold = pm.get("threshold", 0.5)
    vl.check("threshold_is_fixed_at_point_five", threshold == 0.5,
              detail=f"prediction_metadata.threshold={threshold}")
    recomputed_preds = (probs_arr >= 0.5).astype(np.int64)
    preds_match = bool(np.array_equal(recomputed_preds, preds_arr))
    vl.check("stored_predictions_match_threshold_0.5", preds_match,
              detail="recomputed (probabilities>=0.5) vs stored predictions column"
                     if preds_match else
                     f"{int((recomputed_preds != preds_arr).sum())} label cells disagree")

    log.info(f"input_validation_passed n_samples={len(df)} n_classes={n_classes} "
             f"checks_passed={vl.n_passed()}/{len(vl.results)}")

    return {
        "n_samples": len(df), "n_classes": n_classes, "class_names": class_names,
        "y_true": gt_arr.astype(int), "y_prob": probs_arr, "y_pred": preds_arr,
        "logits": logits_arr, "threshold": threshold,
    }


# =============================================================================
# 5.4  Statistics — support / prevalence per class
# =============================================================================
def compute_support_and_prevalence(y_true: np.ndarray, y_pred: np.ndarray,
                                    class_names: List[str]) -> pd.DataFrame:
    n = len(y_true)
    pos = y_true.sum(axis=0)
    neg = n - pos
    pred_pos = y_pred.sum(axis=0)
    return pd.DataFrame({
        "class_name": class_names,
        "support": pos.astype(int),
        "positive_count": pos.astype(int),
        "negative_count": neg.astype(int),
        "positive_prevalence": pos / n,
        "negative_prevalence": neg / n,
        "predicted_positive_count": pred_pos.astype(int),
        "prediction_prevalence": pred_pos / n,
    })


# =============================================================================
# 5.5  Ranking-quality metrics (AUROC, Average Precision) — per class, with
#      explicit degenerate-class handling. A class is "degenerate" here if
#      the test set contains only one of {positive, negative} for it (e.g. an
#      extremely rare finding such as Hernia). AUROC/AP are mathematically
#      undefined in that case; sklearn either raises or silently returns a
#      meaningless number depending on the exact function, so this stage
#      checks unique-value count itself BEFORE calling into sklearn, rather
#      than trusting whichever behavior a given metric happens to have.
# =============================================================================
def compute_auroc_and_ap(y_true: np.ndarray, y_prob: np.ndarray,
                          class_names: List[str], log: logging.Logger) -> pd.DataFrame:
    n_classes = len(class_names)
    aurocs, auroc_valid = [], []
    aps, ap_valid = [], []

    for i, cls in enumerate(class_names):
        col_true, col_prob = y_true[:, i], y_prob[:, i]
        both_classes_present = len(np.unique(col_true)) >= 2

        if both_classes_present:
            aurocs.append(roc_auc_score(col_true, col_prob))
            auroc_valid.append(True)
        else:
            aurocs.append(float("nan"))
            auroc_valid.append(False)
            log.warning(f"class={cls} AUROC undefined — only one label value present "
                        f"in ground truth (positives={int(col_true.sum())}, n={len(col_true)}); "
                        f"recorded as NaN and excluded from macro/weighted AUROC.")

        if col_true.sum() > 0:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=UndefinedMetricWarning)
                aps.append(average_precision_score(col_true, col_prob))
            ap_valid.append(True)
        else:
            aps.append(float("nan"))
            ap_valid.append(False)
            log.warning(f"class={cls} Average Precision undefined — zero positive examples "
                        f"in ground truth; recorded as NaN and excluded from macro AP.")

    return pd.DataFrame({
        "class_name": class_names,
        "auroc": aurocs, "auroc_valid": auroc_valid,
        "average_precision": aps, "average_precision_valid": ap_valid,
    })


def aggregate_auroc(ranking_df: pd.DataFrame, support: np.ndarray, log: logging.Logger) -> Dict[str, Any]:
    valid = ranking_df["auroc_valid"].values
    n_excluded = int((~valid).sum())
    if n_excluded:
        log.warning(f"macro/weighted AUROC computed over {valid.sum()}/{len(valid)} classes "
                    f"({n_excluded} excluded as undefined).")
    macro = float(np.nanmean(ranking_df["auroc"].values)) if valid.any() else None
    if valid.any():
        weighted = float(np.sum(ranking_df["auroc"].values[valid] * support[valid]) / support[valid].sum())
    else:
        weighted = None
    return {
        "auroc_macro": macro,
        "auroc_weighted": weighted,
        "n_classes_excluded_undefined": n_excluded,
        "excluded_classes": ranking_df.loc[~valid, "class_name"].tolist(),
    }


def compute_micro_auroc(y_true: np.ndarray, y_prob: np.ndarray, log: logging.Logger) -> Optional[float]:
    try:
        return float(roc_auc_score(y_true.ravel(), y_prob.ravel()))
    except ValueError as e:
        log.warning(f"micro AUROC could not be computed on flattened array: {e}")
        return None


def aggregate_average_precision(ranking_df: pd.DataFrame, support: np.ndarray) -> Dict[str, Any]:
    valid = ranking_df["average_precision_valid"].values
    macro = float(np.nanmean(ranking_df["average_precision"].values)) if valid.any() else None
    weighted = (float(np.sum(ranking_df["average_precision"].values[valid] * support[valid]) / support[valid].sum())
                if valid.any() else None)
    return {"average_precision_macro": macro, "average_precision_weighted": weighted}


# =============================================================================
# 5.6  Threshold-based metrics at the fixed 0.5 operating point: precision,
#      recall/sensitivity, specificity, F1, per-class accuracy, MCC,
#      balanced accuracy — per class, plus macro/micro/weighted aggregates.
# =============================================================================
def compute_threshold_metrics(y_true: np.ndarray, y_pred: np.ndarray,
                               class_names: List[str], log: logging.Logger) -> Dict[str, Any]:
    n_classes = len(class_names)

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="micro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    # sanity cross-check: macro precision must equal the mean of the
    # per-class precision vector (catches sklearn API misuse immediately)
    if not np.isclose(precision_macro, np.mean(precision), atol=1e-9):
        log.warning(f"macro precision sanity check mismatch: aggregate={precision_macro} "
                    f"mean_of_per_class={np.mean(precision)}")

    mcm = multilabel_confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = mcm[:, 0, 0].astype(float), mcm[:, 0, 1].astype(float), \
                     mcm[:, 1, 0].astype(float), mcm[:, 1, 1].astype(float)

    with np.errstate(divide="ignore", invalid="ignore"):
        specificity = np.where((tn + fp) > 0, tn / (tn + fp), np.nan)
    specificity_valid = (tn + fp) > 0
    if not specificity_valid.all():
        bad = [class_names[i] for i in np.where(~specificity_valid)[0]]
        log.warning(f"specificity undefined (no true negatives) for classes: {bad}")

    accuracy_per_class = (tp + tn) / (tp + tn + fp + fn)

    mcc_per_class, mcc_valid = [], []
    bal_acc_per_class, bal_acc_valid = [], []
    for i, cls in enumerate(class_names):
        col_true, col_pred = y_true[:, i], y_pred[:, i]
        if len(np.unique(col_true)) < 2:
            mcc_per_class.append(float("nan")); mcc_valid.append(False)
            bal_acc_per_class.append(float("nan")); bal_acc_valid.append(False)
            log.warning(f"class={cls} MCC/balanced-accuracy undefined — only one label "
                        f"value present in ground truth; recorded as NaN.")
            continue
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=UndefinedMetricWarning)
            mcc_per_class.append(float(matthews_corrcoef(col_true, col_pred)))
            bal_acc_per_class.append(float(balanced_accuracy_score(col_true, col_pred)))
        mcc_valid.append(True)
        bal_acc_valid.append(True)

    mcc_per_class = np.array(mcc_per_class)
    bal_acc_per_class = np.array(bal_acc_per_class)
    mcc_valid = np.array(mcc_valid)
    bal_acc_valid = np.array(bal_acc_valid)

    # global ("micro-style") flattened variants — MCC and balanced accuracy
    # have no native sklearn multilabel aggregation, so the flattened
    # treatment (every (sample, class) cell as one independent binary
    # decision) is used as the micro-equivalent, documented explicitly.
    mcc_flattened = float(matthews_corrcoef(y_true.ravel(), y_pred.ravel()))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UndefinedMetricWarning)
        bal_acc_flattened = float(balanced_accuracy_score(y_true.ravel(), y_pred.ravel()))

    per_class_df = pd.DataFrame({
        "class_name": class_names,
        "precision": precision, "recall_sensitivity": recall, "specificity": specificity,
        "f1": f1, "accuracy": accuracy_per_class,
        "mcc": mcc_per_class, "mcc_valid": mcc_valid,
        "balanced_accuracy": bal_acc_per_class, "balanced_accuracy_valid": bal_acc_valid,
        "true_positive": tp.astype(int), "true_negative": tn.astype(int),
        "false_positive": fp.astype(int), "false_negative": fn.astype(int),
    })

    specificity_macro = float(np.nanmean(specificity)) if specificity_valid.any() else None
    specificity_micro = float(tn.sum() / (tn.sum() + fp.sum())) if (tn.sum() + fp.sum()) > 0 else None
    support_f = support.astype(float)
    specificity_weighted = (
        float(np.sum(specificity[specificity_valid] * support_f[specificity_valid]) / support_f[specificity_valid].sum())
        if specificity_valid.any() and support_f[specificity_valid].sum() > 0 else None
    )

    aggregates = {
        "precision_macro": float(precision_macro), "precision_micro": float(precision_micro),
        "precision_weighted": float(precision_weighted),
        "recall_macro": float(recall_macro), "recall_micro": float(recall_micro),
        "recall_weighted": float(recall_weighted),
        "sensitivity_macro": float(recall_macro),  # sensitivity == recall (positive class)
        "sensitivity_micro": float(recall_micro),
        "sensitivity_weighted": float(recall_weighted),
        "specificity_macro": specificity_macro, "specificity_micro": specificity_micro,
        "specificity_weighted": specificity_weighted,
        "f1_macro": float(f1_macro), "f1_micro": float(f1_micro), "f1_weighted": float(f1_weighted),
        "accuracy_macro": float(np.mean(accuracy_per_class)),
        "accuracy_micro": float((tp.sum() + tn.sum()) / (tp.sum() + tn.sum() + fp.sum() + fn.sum())),
        "mcc_macro": float(np.nanmean(mcc_per_class)) if mcc_valid.any() else None,
        "mcc_global_flattened": mcc_flattened,
        "balanced_accuracy_macro": float(np.nanmean(bal_acc_per_class)) if bal_acc_valid.any() else None,
        "balanced_accuracy_global_flattened": bal_acc_flattened,
        "n_classes_excluded_mcc_balacc_undefined": int((~mcc_valid).sum()),
    }
    return {"per_class_df": per_class_df, "aggregates": aggregates}


# =============================================================================
# 5.7  Whole-sample-set multilabel metrics: subset accuracy / exact match
#      ratio, Hamming loss, Jaccard (macro/micro/weighted/samples), and the
#      three ranking-order metrics — computed ONLY on rows where sklearn's
#      ranking metrics are mathematically defined (at least one positive AND
#      at least one negative label). The NIH ChestXray14 label set used here
#      has no explicit "No Finding" class, so a large fraction of rows are
#      legitimately all-zero; these are excluded from ranking metrics only,
#      never silently included with a fabricated value.
# =============================================================================
def compute_multilabel_summary(y_true: np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray,
                                log: logging.Logger) -> Dict[str, Any]:
    n_samples, n_classes = y_true.shape

    subset_accuracy = float(accuracy_score(y_true, y_pred))
    exact_match_ratio = subset_accuracy  # identical definition; reported under both names per spec
    hamming = float(hamming_loss(y_true, y_pred))

    jaccard_macro = float(jaccard_score(y_true, y_pred, average="macro", zero_division=0))
    jaccard_micro = float(jaccard_score(y_true, y_pred, average="micro", zero_division=0))
    jaccard_weighted = float(jaccard_score(y_true, y_pred, average="weighted", zero_division=0))
    jaccard_samples = float(jaccard_score(y_true, y_pred, average="samples", zero_division=0))

    row_sums = y_true.sum(axis=1)
    eligible_mask = (row_sums > 0) & (row_sums < n_classes)
    n_eligible = int(eligible_mask.sum())
    n_excluded = n_samples - n_eligible

    ranking_metrics: Dict[str, Any] = {
        "n_samples_total": n_samples,
        "n_samples_eligible": n_eligible,
        "n_samples_excluded": n_excluded,
        "exclusion_reason": "row has all-zero or all-one label vector; LRAP / coverage error / "
                             "label ranking loss are mathematically undefined for such rows",
    }

    if n_eligible == 0:
        log.warning("No rows are eligible for ranking metrics (every row is all-zero or all-one). "
                    "label_ranking_average_precision / coverage_error / label_ranking_loss "
                    "marked not applicable.")
        ranking_metrics.update({
            "applicable": False,
            "label_ranking_average_precision": None,
            "coverage_error": None,
            "label_ranking_loss": None,
        })
    else:
        if n_excluded > 0:
            log.warning(f"Ranking metrics computed on {n_eligible}/{n_samples} rows "
                        f"({n_excluded} rows excluded as all-zero or all-one label vectors).")
        yt_e, yp_e = y_true[eligible_mask], y_prob[eligible_mask]
        ranking_metrics.update({
            "applicable": True,
            "label_ranking_average_precision": float(label_ranking_average_precision_score(yt_e, yp_e)),
            "coverage_error": float(coverage_error(yt_e, yp_e)),
            "label_ranking_loss": float(label_ranking_loss(yt_e, yp_e)),
        })

    return {
        "subset_accuracy": subset_accuracy,
        "exact_match_ratio": exact_match_ratio,
        "hamming_loss": hamming,
        "jaccard_macro": jaccard_macro,
        "jaccard_micro": jaccard_micro,
        "jaccard_weighted": jaccard_weighted,
        "jaccard_samples": jaccard_samples,
        "ranking_metrics": ranking_metrics,
    }


# =============================================================================
# 5.8  Report writers (report_writer.py-equivalent)
# =============================================================================
def _round_trip_json(path: Path) -> None:
    with open(path) as f:
        json.load(f)  # raises json.JSONDecodeError on corruption


def write_metrics_artifacts(
    validated: Dict[str, Any],
    support_df: pd.DataFrame,
    ranking_df: pd.DataFrame,
    threshold_result: Dict[str, Any],
    multilabel_summary: Dict[str, Any],
    auroc_agg: Dict[str, Any],
    micro_auroc: Optional[float],
    ap_agg: Dict[str, Any],
    inputs: Dict[str, Any],
    vl: "ValidationLedger",
    timer: "StageTimer",
    log: logging.Logger,
) -> Dict[str, str]:
    class_names = validated["class_names"]
    per_class_threshold_df = threshold_result["per_class_df"]

    # --- assemble the single per-class table ---------------------------------
    per_class_df = (
        support_df.merge(ranking_df, on="class_name")
                  .merge(per_class_threshold_df, on="class_name")
    )
    per_class_csv_path = STAGE_DIR / "per_class_metrics.csv"
    per_class_df.to_csv(per_class_csv_path, index=False)

    generated_at = datetime.now(timezone.utc).isoformat()

    # --- macro_metrics.json ----------------------------------------------------
    macro_metrics = {
        "generated_at_utc": generated_at,
        "auroc_macro": auroc_agg["auroc_macro"],
        "average_precision_macro": ap_agg["average_precision_macro"],
        "precision_macro": threshold_result["aggregates"]["precision_macro"],
        "recall_macro": threshold_result["aggregates"]["recall_macro"],
        "sensitivity_macro": threshold_result["aggregates"]["sensitivity_macro"],
        "specificity_macro": threshold_result["aggregates"]["specificity_macro"],
        "f1_macro": threshold_result["aggregates"]["f1_macro"],
        "accuracy_macro": threshold_result["aggregates"]["accuracy_macro"],
        "mcc_macro": threshold_result["aggregates"]["mcc_macro"],
        "balanced_accuracy_macro": threshold_result["aggregates"]["balanced_accuracy_macro"],
        "jaccard_macro": multilabel_summary["jaccard_macro"],
        "n_classes_excluded_from_auroc": auroc_agg["n_classes_excluded_undefined"],
        "excluded_classes_auroc": auroc_agg["excluded_classes"],
    }
    macro_path = STAGE_DIR / "macro_metrics.json"
    with open(macro_path, "w") as f:
        json.dump(macro_metrics, f, indent=2, default=str)

    # --- micro_metrics.json ------------------------------------------------
    micro_metrics = {
        "generated_at_utc": generated_at,
        "auroc_micro": micro_auroc,
        "precision_micro": threshold_result["aggregates"]["precision_micro"],
        "recall_micro": threshold_result["aggregates"]["recall_micro"],
        "sensitivity_micro": threshold_result["aggregates"]["sensitivity_micro"],
        "specificity_micro": threshold_result["aggregates"]["specificity_micro"],
        "f1_micro": threshold_result["aggregates"]["f1_micro"],
        "accuracy_micro": threshold_result["aggregates"]["accuracy_micro"],
        "mcc_global_flattened": threshold_result["aggregates"]["mcc_global_flattened"],
        "balanced_accuracy_global_flattened": threshold_result["aggregates"]["balanced_accuracy_global_flattened"],
        "jaccard_micro": multilabel_summary["jaccard_micro"],
        "note": "mcc_global_flattened and balanced_accuracy_global_flattened treat every "
                "(sample, class) cell as one independent binary decision — sklearn has no "
                "native multilabel 'micro' aggregation for MCC or balanced accuracy.",
    }
    micro_path = STAGE_DIR / "micro_metrics.json"
    with open(micro_path, "w") as f:
        json.dump(micro_metrics, f, indent=2, default=str)

    # --- weighted_metrics.json ------------------------------------------------
    weighted_metrics = {
        "generated_at_utc": generated_at,
        "auroc_weighted": auroc_agg["auroc_weighted"],
        "average_precision_weighted": ap_agg["average_precision_weighted"],
        "precision_weighted": threshold_result["aggregates"]["precision_weighted"],
        "recall_weighted": threshold_result["aggregates"]["recall_weighted"],
        "sensitivity_weighted": threshold_result["aggregates"]["sensitivity_weighted"],
        "specificity_weighted": threshold_result["aggregates"]["specificity_weighted"],
        "f1_weighted": threshold_result["aggregates"]["f1_weighted"],
        "jaccard_weighted": multilabel_summary["jaccard_weighted"],
        "weighting_basis": "per-class support (positive count) in the ground-truth test set",
    }
    weighted_path = STAGE_DIR / "weighted_metrics.json"
    with open(weighted_path, "w") as f:
        json.dump(weighted_metrics, f, indent=2, default=str)

    # --- evaluation_metrics.json (the comprehensive artifact) -----------------
    evaluation_metrics = {
        "generated_at_utc": generated_at,
        "threshold": validated["threshold"],
        "n_samples": validated["n_samples"],
        "n_classes": validated["n_classes"],
        "class_names": class_names,
        "auroc": {
            "per_class": {c: (None if pd.isna(v) else float(v))
                          for c, v in zip(class_names, ranking_df["auroc"])},
            "macro": auroc_agg["auroc_macro"],
            "micro": micro_auroc,
            "weighted": auroc_agg["auroc_weighted"],
            "n_classes_excluded_undefined": auroc_agg["n_classes_excluded_undefined"],
            "excluded_classes": auroc_agg["excluded_classes"],
        },
        "average_precision": {
            "per_class": {c: (None if pd.isna(v) else float(v))
                          for c, v in zip(class_names, ranking_df["average_precision"])},
            "macro": ap_agg["average_precision_macro"],
            "weighted": ap_agg["average_precision_weighted"],
        },
        "precision_recall_f1": {
            "per_class": {
                "precision": per_class_threshold_df["precision"].tolist(),
                "recall_sensitivity": per_class_threshold_df["recall_sensitivity"].tolist(),
                "specificity": [None if pd.isna(v) else float(v) for v in per_class_threshold_df["specificity"]],
                "f1": per_class_threshold_df["f1"].tolist(),
            },
            "macro": {k: v for k, v in threshold_result["aggregates"].items() if k.endswith("_macro")},
            "micro": {k: v for k, v in threshold_result["aggregates"].items() if k.endswith("_micro")},
            "weighted": {k: v for k, v in threshold_result["aggregates"].items() if k.endswith("_weighted")},
        },
        "accuracy": {
            "subset_accuracy": multilabel_summary["subset_accuracy"],
            "exact_match_ratio": multilabel_summary["exact_match_ratio"],
            "note": "subset_accuracy and exact_match_ratio are the same metric "
                    "(fraction of samples with a perfectly correct 14-label vector); "
                    "both keys are reported to satisfy naming used in different literature.",
            "mean_per_class_accuracy": threshold_result["aggregates"]["accuracy_macro"],
            "micro_accuracy": threshold_result["aggregates"]["accuracy_micro"],
        },
        "mcc": {
            "per_class": {c: (None if pd.isna(v) else float(v))
                          for c, v in zip(class_names, per_class_threshold_df["mcc"])},
            "macro": threshold_result["aggregates"]["mcc_macro"],
            "global_flattened": threshold_result["aggregates"]["mcc_global_flattened"],
        },
        "balanced_accuracy": {
            "per_class": {c: (None if pd.isna(v) else float(v))
                          for c, v in zip(class_names, per_class_threshold_df["balanced_accuracy"])},
            "macro": threshold_result["aggregates"]["balanced_accuracy_macro"],
            "global_flattened": threshold_result["aggregates"]["balanced_accuracy_global_flattened"],
        },
        "hamming_loss": multilabel_summary["hamming_loss"],
        "jaccard": {
            "macro": multilabel_summary["jaccard_macro"],
            "micro": multilabel_summary["jaccard_micro"],
            "weighted": multilabel_summary["jaccard_weighted"],
            "samples": multilabel_summary["jaccard_samples"],
        },
        "ranking_metrics": multilabel_summary["ranking_metrics"],
        "support_and_prevalence": support_df.to_dict(orient="records"),
        "warnings": timer.warnings,
    }
    evaluation_metrics_path = STAGE_DIR / "evaluation_metrics.json"
    with open(evaluation_metrics_path, "w") as f:
        json.dump(evaluation_metrics, f, indent=2, default=str)

    # --- evaluation_summary.json (narrative / provenance) ----------------------
    best_meta = inputs["best_model_metadata"]
    val_vs_test = {
        "note": "Informational only — validation-split performance from training is NOT "
                "expected to equal held-out test-split performance computed here, and a "
                "mismatch is not a validation failure.",
        "val_macro_auroc": best_meta.get("val_macro_auroc"),
        "test_macro_auroc": auroc_agg["auroc_macro"],
        "val_micro_auroc": best_meta.get("val_micro_auroc"),
        "test_micro_auroc": micro_auroc,
        "val_f1_macro": best_meta.get("val_f1_macro"),
        "test_f1_macro": threshold_result["aggregates"]["f1_macro"],
    }
    evaluation_summary = {
        "generated_at_utc": generated_at,
        "notebook": "Sprint04_Evaluation.ipynb",
        "stage": "Stage 5 — Metrics Engine",
        "threshold": validated["threshold"],
        "n_samples": validated["n_samples"],
        "n_classes": validated["n_classes"],
        "headline_metrics": {
            "auroc_macro": auroc_agg["auroc_macro"],
            "auroc_micro": micro_auroc,
            "auroc_weighted": auroc_agg["auroc_weighted"],
            "f1_macro": threshold_result["aggregates"]["f1_macro"],
            "f1_micro": threshold_result["aggregates"]["f1_micro"],
            "subset_accuracy": multilabel_summary["subset_accuracy"],
            "hamming_loss": multilabel_summary["hamming_loss"],
            "mcc_macro": threshold_result["aggregates"]["mcc_macro"],
            "balanced_accuracy_macro": threshold_result["aggregates"]["balanced_accuracy_macro"],
        },
        "ranking_metrics_applicability": multilabel_summary["ranking_metrics"],
        "training_vs_test_comparison": val_vs_test,
        "source_artifacts_consumed": {
            "predictions_parquet": str(PREDICTIONS_PARQUET_PATH),
            "raw_logits_pt": str(RAW_LOGITS_PT_PATH),
            "prediction_metadata": str(PREDICTION_METADATA_PATH),
            "dataset_summary": str(STAGE3_SUMMARY_PATH),
            "artifact_validation": str(STAGE2_SUMMARY_PATH),
            "training_summary": str(RESOLVED_PATHS["training_summary"]),
            "best_model_metadata": str(RESOLVED_PATHS["best_model_metadata"]),
        },
        "output_artifacts": {
            "evaluation_metrics": str(evaluation_metrics_path),
            "macro_metrics": str(macro_path),
            "micro_metrics": str(micro_path),
            "weighted_metrics": str(weighted_path),
            "per_class_metrics_csv": str(per_class_csv_path),
        },
        "warnings": timer.warnings,
    }
    evaluation_summary_path = STAGE_DIR / "evaluation_summary.json"
    with open(evaluation_summary_path, "w") as f:
        json.dump(evaluation_summary, f, indent=2, default=str)

    written = {
        "evaluation_metrics": str(evaluation_metrics_path),
        "macro_metrics": str(macro_path),
        "micro_metrics": str(micro_path),
        "weighted_metrics": str(weighted_path),
        "per_class_metrics_csv": str(per_class_csv_path),
        "evaluation_summary": str(evaluation_summary_path),
    }

    # --- round-trip corruption check on every JSON artifact just written ------
    for key, path_str in written.items():
        if path_str.endswith(".json"):
            _round_trip_json(Path(path_str))
    log.info(f"artifacts_written {list(written.keys())}")

    return written


# =============================================================================
# 5.9  Engineering validation artifact
# =============================================================================
def compute_engineering_validation(
    validated: Dict[str, Any],
    auroc_agg: Dict[str, Any],
    threshold_result: Dict[str, Any],
    written_artifacts: Dict[str, str],
    vl: "ValidationLedger",
    log: logging.Logger,
) -> Dict[str, Any]:
    agg = threshold_result["aggregates"]

    def _finite_or_none(x):
        return x is not None and np.isfinite(x)

    metric_finiteness_checks = {
        "auroc_macro": auroc_agg["auroc_macro"],
        "f1_macro": agg["f1_macro"],
        "precision_macro": agg["precision_macro"],
        "recall_macro": agg["recall_macro"],
        "specificity_macro": agg["specificity_macro"],
    }
    all_finite = all(_finite_or_none(v) for v in metric_finiteness_checks.values())
    vl.check("all_headline_metrics_finite", all_finite, detail=str(metric_finiteness_checks))

    def _in_range(name, value, lo=0.0, hi=1.0):
        ok = value is not None and lo - 1e-9 <= value <= hi + 1e-9
        vl.check(f"{name}_range_valid", ok, detail=f"value={value} expected_range=[{lo},{hi}]")

    _in_range("auroc", auroc_agg["auroc_macro"])
    _in_range("f1", agg["f1_macro"])
    _in_range("precision", agg["precision_macro"])
    _in_range("recall", agg["recall_macro"])

    for key, path_str in written_artifacts.items():
        vl.check(f"artifact_written_{key}", Path(path_str).is_file(), detail=path_str)

    engineering_validation = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "all_checks_passed": vl.all_passed(),
        "n_checks": len(vl.results),
        "n_passed": vl.n_passed(),
        "n_failed": len(vl.results) - vl.n_passed(),
        "checks": vl.results,
    }
    return engineering_validation


# =============================================================================
# 5.10  Run Stage 5 (evaluate.py-equivalent orchestration)
# =============================================================================
def run_stage5() -> Dict[str, Any]:
    with StageTimer("stage05_metrics_engine", logger) as timer:
        vl = ValidationLedger(logger)
        log_resource_usage(logger, "stage05_start")

        inputs = reopen_stage5_inputs(logger)
        validated = validate_inputs(inputs, vl, logger)

        y_true, y_prob, y_pred = validated["y_true"], validated["y_prob"], validated["y_pred"]
        class_names = validated["class_names"]

        support_df = compute_support_and_prevalence(y_true, y_pred, class_names)
        ranking_df = compute_auroc_and_ap(y_true, y_prob, class_names, logger)
        auroc_agg = aggregate_auroc(ranking_df, support_df["support"].values, logger)
        micro_auroc = compute_micro_auroc(y_true, y_prob, logger)
        ap_agg = aggregate_average_precision(ranking_df, support_df["support"].values)

        threshold_result = compute_threshold_metrics(y_true, y_pred, class_names, logger)
        multilabel_summary = compute_multilabel_summary(y_true, y_pred, y_prob, logger)

        log_resource_usage(logger, "stage05_pre_write")

        written_artifacts = write_metrics_artifacts(
            validated, support_df, ranking_df, threshold_result, multilabel_summary,
            auroc_agg, micro_auroc, ap_agg, inputs, vl, timer, logger,
        )

        engineering_validation = compute_engineering_validation(
            validated, auroc_agg, threshold_result, written_artifacts, vl, logger,
        )
        engineering_validation_path = STAGE_DIR / "engineering_validation.json"
        with open(engineering_validation_path, "w") as f:
            json.dump(engineering_validation, f, indent=2, default=str)
        _round_trip_json(engineering_validation_path)
        logger.info(f"artifact_written path={engineering_validation_path}")

        if not engineering_validation["all_checks_passed"]:
            raise RuntimeError(
                f"Stage 5 engineering validation FAILED: "
                f"{engineering_validation['n_failed']}/{engineering_validation['n_checks']} checks failed."
            )

        log_resource_usage(logger, "stage05_end")

        result = {
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "n_samples": validated["n_samples"],
            "n_classes": validated["n_classes"],
            "auroc_macro": auroc_agg["auroc_macro"],
            "auroc_micro": micro_auroc,
            "auroc_weighted": auroc_agg["auroc_weighted"],
            "f1_macro": threshold_result["aggregates"]["f1_macro"],
            "subset_accuracy": multilabel_summary["subset_accuracy"],
            "hamming_loss": multilabel_summary["hamming_loss"],
            "all_engineering_checks_passed": engineering_validation["all_checks_passed"],
            "n_checks": engineering_validation["n_checks"],
            "warnings": timer.warnings,
            "artifacts": {**written_artifacts, "engineering_validation": str(engineering_validation_path)},
        }
        result["_artifact_path"] = str(written_artifacts["evaluation_summary"])
        return result


METRICS_SUMMARY = run_stage5()

print("\n" + "=" * 70)
print("STAGE 5 — METRICS ENGINE SUMMARY")
print("=" * 70)
print(f"Samples evaluated         : {METRICS_SUMMARY['n_samples']}")
print(f"Classes                   : {METRICS_SUMMARY['n_classes']}")
print(f"AUROC  (macro/micro/wtd)  : {METRICS_SUMMARY['auroc_macro']:.4f} / "
      f"{METRICS_SUMMARY['auroc_micro']:.4f} / {METRICS_SUMMARY['auroc_weighted']:.4f}")
print(f"F1 macro                  : {METRICS_SUMMARY['f1_macro']:.4f}")
print(f"Subset accuracy (EMR)     : {METRICS_SUMMARY['subset_accuracy']:.4f}")
print(f"Hamming loss              : {METRICS_SUMMARY['hamming_loss']:.4f}")
print(f"Engineering checks        : {METRICS_SUMMARY['n_checks']} run, "
      f"all_passed={METRICS_SUMMARY['all_engineering_checks_passed']}")
print(f"Warnings                  : {len(METRICS_SUMMARY['warnings'])}")
for w in METRICS_SUMMARY["warnings"]:
    print(f"  - {w}")
print(f"evaluation_metrics.json   : {METRICS_SUMMARY['artifacts']['evaluation_metrics']}")
print(f"macro_metrics.json        : {METRICS_SUMMARY['artifacts']['macro_metrics']}")
print(f"micro_metrics.json        : {METRICS_SUMMARY['artifacts']['micro_metrics']}")
print(f"weighted_metrics.json     : {METRICS_SUMMARY['artifacts']['weighted_metrics']}")
print(f"per_class_metrics.csv     : {METRICS_SUMMARY['artifacts']['per_class_metrics_csv']}")
print(f"engineering_validation.json: {METRICS_SUMMARY['artifacts']['engineering_validation']}")
print(f"evaluation_summary.json   : {METRICS_SUMMARY['_artifact_path']}")
print("Stage 5 — Metrics Engine : OK")
print("Stage 5 complete. Stage 6 (threshold optimization, calibration, "
      "GradCAM, visualization) not implemented per scope.")

2026-07-05 05:39:43 | INFO    | sprint04_eval.stage05_metrics_engine | START  stage=stage05_metrics_engine
2026-07-05 05:39:43 | INFO    | sprint04_eval.stage05_metrics_engine | resource_usage tag=stage05_start {'gpu_allocated_mb': 9.12, 'gpu_reserved_mb': 90.0, 'cpu_maxrss_mb': 1784.5}
2026-07-05 05:39:43 | INFO    | sprint04_eval.stage05_metrics_engine | reopened training_summary path=/kaggle/input/datasets/anupsharma1730/visionserveai-sprint04-artifacts/visionserveai/sprint04/training_summary.json
2026-07-05 05:39:43 | INFO    | sprint04_eval.stage05_metrics_engine | reopened best_model_metadata path=/kaggle/input/datasets/anupsharma1730/visionserveai-sprint04-artifacts/visionserveai/sprint04/metrics/best_model_metadata.json
2026-07-05 05:39:43 | INFO    | sprint04_eval.stage05_metrics_engine | reopened prediction_metadata path=/kaggle/working/sprint04_evaluation/stage04_inference_engine/prediction_metadata.json
2026-07-05 05:39:43 | INFO    | sprint04_eval.stage05_metrics_engine | 

In [7]:
# =============================================================================
# Sprint04_Evaluation.ipynb
# STAGE 6 — Production Threshold Optimization & Calibration Engine
# -----------------------------------------------------------------------------
# READ-ONLY. Consumes ONLY Stage 4 (inference) and Stage 5 (metrics) artifacts
# exactly as written to disk (never trusts notebook memory, never reruns
# inference, never reloads the dataset, never touches/modifies Stage 5's
# outputs). All Stage 4/5 file locations are re-derived from each stage's own
# manifest (inference_summary.json["artifacts"], evaluation_summary.json
# ["output_artifacts"]) — never rediscovered or guessed.
#
# Scope discipline (per project rules): threshold sweeps, calibration,
# ROC/PR curve data, probability distributions, pathology detection, and
# deployment recommendations ONLY. No Grad-CAM, no explainability, no ONNX/
# TensorRT export, no MLflow, no deployment infrastructure. Those belong to
# later stages.
#
# Modularity note: intentionally split into the same functional seams it
# will later become:
#   validators.py    -> Section 6.2-6.3 (reopen + cross-artifact validation)
#   thresholds.py     -> Section 6.4 (per-class threshold sweep + optima)
#   calibration.py     -> Section 6.5 (reliability / ECE / MCE / Brier)
#   diagnostics.py     -> Section 6.6 (probability distributions + pathology)
#   writers.py         -> Section 6.7 (deployment recs + artifact writers)
#   evaluate.py         -> Section 6.8 (orchestration / run_stage6)
# =============================================================================

import os, sys, json, time, logging
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
import torch

# -----------------------------------------------------------------------------
# 6.0  Re-derive paths independently (do not trust in-memory Stage 1-5 state)
# -----------------------------------------------------------------------------
_KAGGLE_INPUT = Path("/kaggle/input")
_KAGGLE_WORKING = Path("/kaggle/working")
IS_KAGGLE = _KAGGLE_INPUT.is_dir()
OUTPUT_ROOT = (_KAGGLE_WORKING / "sprint04_evaluation") if IS_KAGGLE else Path(os.environ.get("EVAL_OUTPUT_ROOT", "sprint04_evaluation"))

STAGE4_SUMMARY_PATH = OUTPUT_ROOT / "stage04_inference_engine" / "inference_summary.json"
STAGE5_SUMMARY_PATH = OUTPUT_ROOT / "stage05_metrics_engine" / "evaluation_summary.json"
STAGE5_ENGVAL_PATH  = OUTPUT_ROOT / "stage05_metrics_engine" / "engineering_validation.json"

for _label, _p in [("Stage 4", STAGE4_SUMMARY_PATH), ("Stage 5", STAGE5_SUMMARY_PATH),
                    ("Stage 5 engineering_validation", STAGE5_ENGVAL_PATH)]:
    if not _p.is_file():
        raise FileNotFoundError(
            f"{_label} output not found at {_p}. Stage 6 never regenerates prior "
            f"stages, and never runs inference or metrics itself — run Stages 1-5 first, in order."
        )

with open(STAGE4_SUMMARY_PATH) as f:
    STAGE4_SUMMARY = json.load(f)
with open(STAGE5_SUMMARY_PATH) as f:
    STAGE5_SUMMARY = json.load(f)
with open(STAGE5_ENGVAL_PATH) as f:
    STAGE5_ENGVAL = json.load(f)

if not STAGE4_SUMMARY.get("all_validation_checks_passed", False):
    raise RuntimeError(
        "Stage 6 aborted — Stage 4's inference_summary.json reports "
        "all_validation_checks_passed=False. Stage 6 must never compute threshold/"
        "calibration diagnostics on top of an inference run that failed its own "
        "engineering validation."
    )
if not STAGE5_ENGVAL.get("all_checks_passed", False):
    raise RuntimeError(
        "Stage 6 aborted — Stage 5's engineering_validation.json reports "
        "all_checks_passed=False. Stage 5 must be fully green (frozen, trusted) "
        "before Stage 6 runs."
    )

# Stage 4's own artifact manifest is the ONLY source of truth for where the
# prediction outputs live — Stage 6 never re-derives these paths itself.
_STAGE4_ARTIFACTS = STAGE4_SUMMARY.get("artifacts", {})
for _key in ["predictions_parquet", "raw_logits_pt", "prediction_metadata"]:
    if _key not in _STAGE4_ARTIFACTS:
        raise FileNotFoundError(
            f"Stage 4's inference_summary.json is missing artifacts.{_key}. "
            f"Stage 6 cannot proceed without a complete Stage 4 artifact manifest."
        )

# Stage 5's own artifact manifest is the ONLY source of truth for its outputs.
_STAGE5_ARTIFACTS = STAGE5_SUMMARY.get("output_artifacts", {})
for _key in ["per_class_metrics_csv", "evaluation_metrics", "macro_metrics", "micro_metrics", "weighted_metrics"]:
    if _key not in _STAGE5_ARTIFACTS:
        raise FileNotFoundError(
            f"Stage 5's evaluation_summary.json is missing output_artifacts.{_key}. "
            f"Stage 6 cannot proceed without a complete Stage 5 artifact manifest."
        )

PREDICTIONS_PARQUET_PATH   = Path(_STAGE4_ARTIFACTS["predictions_parquet"])
RAW_LOGITS_PT_PATH         = Path(_STAGE4_ARTIFACTS["raw_logits_pt"])
PREDICTION_METADATA_PATH   = Path(_STAGE4_ARTIFACTS["prediction_metadata"])
PER_CLASS_METRICS_CSV_PATH = Path(_STAGE5_ARTIFACTS["per_class_metrics_csv"])
EVALUATION_METRICS_PATH    = Path(_STAGE5_ARTIFACTS["evaluation_metrics"])
MACRO_METRICS_PATH         = Path(_STAGE5_ARTIFACTS["macro_metrics"])
MICRO_METRICS_PATH         = Path(_STAGE5_ARTIFACTS["micro_metrics"])
WEIGHTED_METRICS_PATH      = Path(_STAGE5_ARTIFACTS["weighted_metrics"])

for _label, _p in [("predictions.parquet", PREDICTIONS_PARQUET_PATH),
                    ("raw_logits.pt", RAW_LOGITS_PT_PATH),
                    ("prediction_metadata.json", PREDICTION_METADATA_PATH),
                    ("per_class_metrics.csv", PER_CLASS_METRICS_CSV_PATH),
                    ("evaluation_metrics.json", EVALUATION_METRICS_PATH),
                    ("macro_metrics.json", MACRO_METRICS_PATH),
                    ("micro_metrics.json", MICRO_METRICS_PATH),
                    ("weighted_metrics.json", WEIGHTED_METRICS_PATH)]:
    if not _p.is_file():
        raise FileNotFoundError(f"Artifact {_label} not found at {_p} even though it is "
                                 f"listed in its stage's own summary. That stage must be rerun.")

STAGE_DIR = OUTPUT_ROOT / "stage06_threshold_calibration_engine"
STAGE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = OUTPUT_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

CALIBRATION_N_BINS   = 15
MIN_AUROC_FOR_DEPLOY = 0.65
MIN_F1_FOR_DEPLOY    = 0.15
COLLAPSE_STD_EPS     = 1e-4

# -----------------------------------------------------------------------------
# 6.1  Logging / timing / resource tracking (identical pattern to Stages 1-5)
# -----------------------------------------------------------------------------
def setup_logging(log_dir: Path, stage_name: str) -> logging.Logger:
    logger = logging.getLogger(f"sprint04_eval.{stage_name}")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False
    fmt = logging.Formatter(fmt="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
                             datefmt="%Y-%m-%d %H:%M:%S")
    console = logging.StreamHandler(sys.stdout); console.setFormatter(fmt)
    logger.addHandler(console)
    fh = logging.FileHandler(log_dir / f"{stage_name}.log", mode="a"); fh.setFormatter(fmt)
    logger.addHandler(fh)
    return logger

logger = setup_logging(LOG_DIR, "stage06_threshold_calibration_engine")

class StageTimer:
    def __init__(self, stage_name: str, log: logging.Logger):
        self.stage_name, self.log, self.t0, self.warnings = stage_name, log, None, []
    def warn(self, msg: str) -> None:
        self.warnings.append(msg); self.log.warning(msg)
    def __enter__(self):
        self.t0 = time.perf_counter()
        self.log.info(f"START  stage={self.stage_name}")
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        dur = time.perf_counter() - self.t0
        status = "FAILED" if exc_type else "FINISH"
        self.log.info(f"{status} stage={self.stage_name} duration_sec={dur:.2f} warnings={len(self.warnings)}")
        return False

def log_resource_usage(log: logging.Logger, tag: str) -> Dict[str, float]:
    usage = {}
    if torch.cuda.is_available():
        usage["gpu_allocated_mb"] = round(torch.cuda.memory_allocated() / 1024**2, 2)
        usage["gpu_reserved_mb"] = round(torch.cuda.memory_reserved() / 1024**2, 2)
    try:
        import resource
        usage["cpu_maxrss_mb"] = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1)
    except Exception:
        pass
    log.info(f"resource_usage tag={tag} {usage}")
    return usage

class ValidationLedger:
    """Accumulates PASS/FAIL engineering checks. Every check is logged
    immediately; hard-required checks raise on failure (never silently
    continue), soft/informational checks are recorded but don't abort."""
    def __init__(self, log: logging.Logger):
        self.log = log
        self.results: List[Dict[str, Any]] = []
    def check(self, name: str, passed: bool, detail: str = "", hard: bool = True) -> None:
        status = "PASS" if passed else "FAIL"
        entry = {"check": name, "status": status, "detail": detail, "hard": hard}
        self.results.append(entry)
        (self.log.info if passed else self.log.error)(f"validation check={name} status={status} detail={detail}")
        if hard and not passed:
            raise RuntimeError(f"Stage 6 aborted — validation check {name!r} FAILED: {detail}")
    def all_passed(self) -> bool:
        return all(r["status"] == "PASS" for r in self.results)
    def n_passed(self) -> int:
        return sum(1 for r in self.results if r["status"] == "PASS")


# =============================================================================
# 6.2  Reopen validated artifacts (never rediscover — paths came from Stage 4/5)
# =============================================================================
def load_json_strict(path: Path) -> Dict[str, Any]:
    try:
        with open(path) as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(f"Corrupted JSON at {path}: {e}") from e

def reopen_stage6_inputs(log: logging.Logger) -> Dict[str, Any]:
    prediction_metadata = load_json_strict(PREDICTION_METADATA_PATH)
    per_class_df = pd.read_csv(PER_CLASS_METRICS_CSV_PATH)
    evaluation_metrics = load_json_strict(EVALUATION_METRICS_PATH)
    macro_metrics = load_json_strict(MACRO_METRICS_PATH)
    micro_metrics = load_json_strict(MICRO_METRICS_PATH)
    weighted_metrics = load_json_strict(WEIGHTED_METRICS_PATH)
    log.info(f"reopened prediction_metadata path={PREDICTION_METADATA_PATH}")
    log.info(f"reopened per_class_metrics.csv path={PER_CLASS_METRICS_CSV_PATH} rows={len(per_class_df)}")

    predictions_df = pd.read_parquet(PREDICTIONS_PARQUET_PATH)
    log.info(f"reopened predictions.parquet path={PREDICTIONS_PARQUET_PATH} rows={len(predictions_df)}")

    raw_logits_blob = torch.load(RAW_LOGITS_PT_PATH, weights_only=False)
    log.info(f"reopened raw_logits.pt path={RAW_LOGITS_PT_PATH} "
             f"tensor_shape={tuple(raw_logits_blob['logits'].shape)}")

    return {
        "prediction_metadata": prediction_metadata,
        "per_class_df": per_class_df,
        "evaluation_metrics": evaluation_metrics,
        "macro_metrics": macro_metrics,
        "micro_metrics": micro_metrics,
        "weighted_metrics": weighted_metrics,
        "predictions_df": predictions_df,
        "raw_logits_blob": raw_logits_blob,
    }


# =============================================================================
# 6.3  Validation — structural / schema / cross-artifact consistency
#      Fails loudly (hard=True) on any inconsistency. This is the gate that
#      must pass before a single threshold/calibration diagnostic is computed.
# =============================================================================
def validate_inputs(inputs: Dict[str, Any], vl: "ValidationLedger", log: logging.Logger) -> Dict[str, Any]:
    df = inputs["predictions_df"]
    pm = inputs["prediction_metadata"]
    raw = inputs["raw_logits_blob"]
    per_class_df = inputs["per_class_df"]
    em = inputs["evaluation_metrics"]

    pm_class_names   = pm["class_names"]
    raw_class_names  = list(raw["class_names"])
    csv_class_names  = per_class_df["class_name"].tolist()
    em_class_names   = em.get("class_names", pm_class_names)

    vl.check(
        "class_order_consistent_stage4_stage5",
        pm_class_names == raw_class_names == csv_class_names == em_class_names,
        detail=f"prediction_metadata vs raw_logits.pt vs per_class_metrics.csv vs evaluation_metrics.json "
               f"class name lists {'match exactly' if pm_class_names == raw_class_names == csv_class_names == em_class_names else 'DIFFER'}",
    )
    class_names = pm_class_names
    n_classes = len(class_names)

    vl.check(
        "class_count_consistent",
        len(raw_class_names) == len(csv_class_names) == len(em_class_names) == pm["num_classes"],
        detail=f"pm={pm['num_classes']} raw={len(raw_class_names)} csv={len(csv_class_names)} em={len(em_class_names)}",
    )

    vl.check(
        "prediction_count_matches_metadata",
        len(df) == pm["n_samples"] == em["n_samples"],
        detail=f"predictions.parquet_rows={len(df)} prediction_metadata.n_samples={pm['n_samples']} "
               f"evaluation_metrics.n_samples={em['n_samples']}",
    )

    # --- parse array columns exactly as Stage 4/5 do ------------------------
    gt_arr    = np.array(df["ground_truth"].tolist(), dtype=np.float64)
    prob_arr  = np.array(df["probabilities"].tolist(), dtype=np.float64)
    logit_arr = np.array(df["logits"].tolist(), dtype=np.float64)
    pred_arr  = np.array(df["predictions"].tolist(), dtype=np.int64)

    vl.check("array_shapes_correct",
             gt_arr.shape == prob_arr.shape == logit_arr.shape == pred_arr.shape == (len(df), n_classes),
             detail=f"ground_truth={gt_arr.shape} probabilities={prob_arr.shape} "
                    f"logits={logit_arr.shape} predictions={pred_arr.shape} expected=({len(df)},{n_classes})")

    vl.check("ground_truth_is_binary", bool(np.isin(gt_arr, [0, 1]).all()),
             detail=f"unique_values={np.unique(gt_arr).tolist()[:10]}")
    vl.check("probabilities_in_unit_range", bool((prob_arr >= 0).all() and (prob_arr <= 1).all()),
             detail=f"min={prob_arr.min():.6f} max={prob_arr.max():.6f}")
    vl.check("no_nan_reloaded_arrays",
             not (np.isnan(gt_arr).any() or np.isnan(prob_arr).any() or np.isnan(logit_arr).any()),
             detail="checked ground_truth, probabilities, logits")
    vl.check("no_inf_reloaded_arrays",
             not (np.isinf(prob_arr).any() or np.isinf(logit_arr).any()),
             detail="checked probabilities, logits")

    raw_logits_np = raw["logits"].numpy().astype(np.float64)
    vl.check("raw_logits_values_match_parquet",
             raw_logits_np.shape == logit_arr.shape and bool(np.allclose(raw_logits_np, logit_arr, atol=1e-5, rtol=1e-4)),
             detail=f"max_abs_diff={np.max(np.abs(raw_logits_np - logit_arr)):.2e}" if raw_logits_np.shape == logit_arr.shape else "shape mismatch")

    threshold = pm.get("threshold", 0.5)
    recomputed_preds = (prob_arr >= threshold).astype(np.int64)
    vl.check("stored_predictions_match_threshold",
             bool(np.array_equal(recomputed_preds, pred_arr)),
             detail=f"recomputed (probabilities>={threshold}) vs stored predictions column")

    # --- cross-check against Stage 5's FROZEN per-class numbers (the key ---
    #     "reconstruct nothing from notebook state" guarantee: if these don't
    #     match, the reloaded arrays are NOT what Stage 5 actually scored) ---
    recomputed_pos = gt_arr.sum(axis=0).astype(int)
    stored_pos = per_class_df["positive_count"].to_numpy().astype(int)
    vl.check("positive_counts_match_stage5", bool(np.array_equal(recomputed_pos, stored_pos)),
             detail=f"recomputed={recomputed_pos.tolist()} stage5_csv={stored_pos.tolist()}")

    recomputed_predpos = pred_arr.sum(axis=0).astype(int)
    stored_predpos = per_class_df["predicted_positive_count"].to_numpy().astype(int)
    vl.check("predicted_positive_counts_match_stage5", bool(np.array_equal(recomputed_predpos, stored_predpos)),
             detail=f"recomputed={recomputed_predpos.tolist()} stage5_csv={stored_predpos.tolist()}")

    log.info(f"input_validation_passed n_samples={len(df)} n_classes={n_classes} "
             f"checks_passed={vl.n_passed()}/{len(vl.results)}")

    return {
        "n_samples": len(df), "n_classes": n_classes, "class_names": class_names,
        "y_true": gt_arr.astype(int), "y_prob": prob_arr, "y_pred": pred_arr,
        "threshold": threshold, "per_class_df": per_class_df.set_index("class_name").loc[class_names],
    }


# =============================================================================
# 6.4  Threshold sweeps — exact sweep over every unique probability value
# =============================================================================
def per_class_threshold_sweep(y_true: np.ndarray, y_prob: np.ndarray) -> pd.DataFrame:
    """Exact ROC/PR curve: every unique probability value is a candidate
    threshold (descending, stable order -> deterministic tie handling)."""
    y_true = y_true.astype(int)
    n = len(y_true)
    P = int(y_true.sum())
    N = n - P

    order = np.argsort(-y_prob, kind="mergesort")
    p_sorted = y_prob[order]
    y_sorted = y_true[order]

    cum_tp = np.cumsum(y_sorted)
    cum_fp = np.cumsum(1 - y_sorted)

    last_of_run = np.r_[p_sorted[1:] != p_sorted[:-1], True]
    thresholds = p_sorted[last_of_run]
    tp = cum_tp[last_of_run].astype(float)
    fp = cum_fp[last_of_run].astype(float)

    # prepend the "predict nothing positive" boundary operating point
    thresholds = np.r_[thresholds[0] + 1e-9, thresholds]
    tp = np.r_[0.0, tp]
    fp = np.r_[0.0, fp]

    fn = P - tp
    tn = N - fp

    with np.errstate(divide="ignore", invalid="ignore"):
        precision   = np.where((tp + fp) > 0, tp / (tp + fp), 0.0)
        recall      = np.where(P > 0, tp / P, 0.0)
        specificity = np.where(N > 0, tn / N, 0.0)
        f1          = np.where((precision + recall) > 0, 2 * precision * recall / (precision + recall), 0.0)
        bal_acc     = (recall + specificity) / 2.0
        youden_j    = recall + specificity - 1.0

    return pd.DataFrame({
        "threshold": np.clip(thresholds, 0.0, 1.0), "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        "precision": precision, "recall_sensitivity": recall, "specificity": specificity,
        "fpr": 1.0 - specificity, "f1": f1, "balanced_accuracy": bal_acc, "youden_j": youden_j,
    })

def optimal_thresholds_for_class(sweep_df: pd.DataFrame) -> Dict[str, float]:
    def pick(col):
        row = sweep_df.loc[sweep_df[col].idxmax()]
        return float(row["threshold"]), float(row[col])
    f1_thr, f1_val = pick("f1")
    ba_thr, ba_val = pick("balanced_accuracy")
    yj_thr, yj_val = pick("youden_j")
    return {
        "f1_optimal_threshold": f1_thr, "f1_optimal_value": f1_val,
        "balanced_accuracy_optimal_threshold": ba_thr, "balanced_accuracy_optimal_value": ba_val,
        "youden_j_optimal_threshold": yj_thr, "youden_j_optimal_value": yj_val,
    }


# =============================================================================
# 6.5  Calibration diagnostics — reliability bins, ECE, MCE, Brier score
# =============================================================================
def reliability_and_calibration(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 15):
    y_true = y_true.astype(float)
    n = len(y_true)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.clip(np.digitize(y_prob, bin_edges[1:-1], right=True), 0, n_bins - 1)

    rows, ece, mce = [], 0.0, 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        count = int(mask.sum())
        if count == 0:
            rows.append({"bin_index": b, "bin_lower": float(bin_edges[b]), "bin_upper": float(bin_edges[b+1]),
                         "count": 0, "mean_predicted_prob": None, "empirical_positive_rate": None, "abs_gap": None})
            continue
        mean_pred = float(y_prob[mask].mean())
        emp_rate  = float(y_true[mask].mean())
        gap = abs(mean_pred - emp_rate)
        ece += (count / n) * gap
        mce = max(mce, gap)
        rows.append({"bin_index": b, "bin_lower": float(bin_edges[b]), "bin_upper": float(bin_edges[b+1]),
                     "count": count, "mean_predicted_prob": mean_pred, "empirical_positive_rate": emp_rate, "abs_gap": gap})

    brier = float(np.mean((y_prob - y_true) ** 2))
    return pd.DataFrame(rows), float(ece), float(mce), brier


# =============================================================================
# 6.6  Diagnostics — probability distributions + pathological behavior
# =============================================================================
def probability_distribution_summary(class_names: List[str], probabilities: np.ndarray, ground_truth: np.ndarray) -> pd.DataFrame:
    qs = [0.05, 0.25, 0.5, 0.75, 0.95]
    def stats(arr):
        if arr.size == 0:
            return {"n": 0, "mean": None, "std": None, "min": None, "max": None,
                    **{f"q{int(q*100)}": None for q in qs}}
        d = {"n": int(arr.size), "mean": float(arr.mean()), "std": float(arr.std()),
             "min": float(arr.min()), "max": float(arr.max())}
        d.update({f"q{int(q*100)}": float(np.quantile(arr, q)) for q in qs})
        return d
    rows = []
    for i, c in enumerate(class_names):
        p, y = probabilities[:, i], ground_truth[:, i]
        pos_s, neg_s = stats(p[y == 1]), stats(p[y == 0])
        row = {"class_name": c}
        row.update({f"pos_{k}": v for k, v in pos_s.items()})
        row.update({f"neg_{k}": v for k, v in neg_s.items()})
        row["mean_separation"] = (pos_s["mean"] - neg_s["mean"]) if (pos_s["mean"] is not None and neg_s["mean"] is not None) else None
        rows.append(row)
    return pd.DataFrame(rows)

def diagnose_pathologies(class_names: List[str], per_class_df: pd.DataFrame,
                          probabilities: np.ndarray, ground_truth: np.ndarray, predictions_bin: np.ndarray) -> pd.DataFrame:
    n = ground_truth.shape[0]
    reports = []
    for i, c in enumerate(class_names):
        row = per_class_df.loc[c]
        prob_col, pred_col = probabilities[:, i], predictions_bin[:, i]
        std_prob = float(prob_col.std())
        flags = []
        if pred_col.sum() == n: flags.append("always_positive_at_threshold")
        if pred_col.sum() == 0: flags.append("always_negative_at_threshold")
        if std_prob < COLLAPSE_STD_EPS: flags.append("collapsed_probability_distribution")
        auroc_val = row.get("auroc", np.nan)
        if pd.notna(auroc_val) and auroc_val < 0.5: flags.append("auroc_below_chance")
        if int(row.get("positive_count", 0)) == 0: flags.append("no_positive_ground_truth_samples")
        if int(row.get("negative_count", 0)) == 0: flags.append("no_negative_ground_truth_samples")
        reports.append({
            "class_name": c, "prob_std": std_prob,
            "prob_min": float(prob_col.min()), "prob_max": float(prob_col.max()),
            "auroc": float(auroc_val) if pd.notna(auroc_val) else None,
            "pathological": len(flags) > 0, "flags": ";".join(flags) if flags else "",
        })
    return pd.DataFrame(reports)


# =============================================================================
# 6.7  Deployment recommendations + artifact writers
# =============================================================================
def deployment_recommendation(class_name: str, per_class_row: pd.Series,
                               pathology_row: pd.Series, optimal_thr_row: pd.Series) -> Dict[str, Any]:
    reasons, verdict = [], "DEPLOY_CANDIDATE"
    if pathology_row["pathological"]:
        verdict = "DO_NOT_DEPLOY"
        reasons.extend([f for f in pathology_row["flags"].split(";") if f])

    auroc = per_class_row.get("auroc", np.nan)
    if pd.notna(auroc) and auroc < MIN_AUROC_FOR_DEPLOY:
        verdict = "DO_NOT_DEPLOY"
        reasons.append(f"auroc_below_minimum({auroc:.3f}<{MIN_AUROC_FOR_DEPLOY})")

    f1_default = per_class_row.get("f1", np.nan)
    if pd.notna(f1_default) and f1_default < MIN_F1_FOR_DEPLOY and verdict != "DO_NOT_DEPLOY":
        verdict = "REVIEW_REQUIRED"
        reasons.append(f"f1_below_minimum({f1_default:.3f}<{MIN_F1_FOR_DEPLOY})")

    return {
        "class_name": class_name, "verdict": verdict,
        "reasons": ";".join(reasons) if reasons else "metrics_within_acceptable_range",
        "recommended_operating_threshold": round(float(optimal_thr_row["balanced_accuracy_optimal_threshold"]), 6),
        "current_fixed_threshold": 0.5,
    }

def _round_trip_json(path: Path) -> None:
    with open(path) as f:
        json.load(f)

def write_json(obj, path: Path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    _round_trip_json(path)
    logger.info(f"artifact_written path={path}")

def write_csv(df: pd.DataFrame, path: Path):
    df.to_csv(path, index=False)
    logger.info(f"artifact_written path={path}")


# =============================================================================
# 6.8  Run Stage 6 (evaluate.py-equivalent orchestration)
# =============================================================================
def run_stage6() -> Dict[str, Any]:
    with StageTimer("stage06_threshold_calibration_engine", logger) as timer:
        vl = ValidationLedger(logger)
        log_resource_usage(logger, "stage06_start")

        inputs = reopen_stage6_inputs(logger)
        validated = validate_inputs(inputs, vl, logger)

        class_names = validated["class_names"]
        y_true, y_prob, y_pred = validated["y_true"], validated["y_prob"], validated["y_pred"]
        per_class_df = validated["per_class_df"]

        optimal_rows, curve_rows = [], []
        for i, c in enumerate(class_names):
            sweep = per_class_threshold_sweep(y_true[:, i], y_prob[:, i])
            opt = optimal_thresholds_for_class(sweep)
            opt["class_name"] = c
            optimal_rows.append(opt)
            sweep_export = sweep.copy()
            sweep_export.insert(0, "class_name", c)
            curve_rows.append(sweep_export)

        optimal_thresholds_df = pd.DataFrame(optimal_rows)[
            ["class_name", "f1_optimal_threshold", "f1_optimal_value",
             "balanced_accuracy_optimal_threshold", "balanced_accuracy_optimal_value",
             "youden_j_optimal_threshold", "youden_j_optimal_value"]]
        roc_pr_curves_df = pd.concat(curve_rows, ignore_index=True)

        thr_cols = ["f1_optimal_threshold", "balanced_accuracy_optimal_threshold", "youden_j_optimal_threshold"]
        vl.check("optimal_thresholds_in_unit_range",
                 bool(optimal_thresholds_df[thr_cols].apply(lambda s: s.between(0.0, 1.0)).all().all()),
                 detail="all optimal thresholds must lie in [0,1]")
        logger.info(f"threshold_sweep_completed n_classes={len(class_names)}")

        calib_bin_rows, calib_summary_rows = [], []
        for i, c in enumerate(class_names):
            bins_df, ece, mce, brier = reliability_and_calibration(y_true[:, i], y_prob[:, i], n_bins=CALIBRATION_N_BINS)
            bins_df.insert(0, "class_name", c)
            calib_bin_rows.append(bins_df)
            calib_summary_rows.append({"class_name": c, "ece": ece, "mce": mce, "brier_score": brier})
        calibration_bins_df = pd.concat(calib_bin_rows, ignore_index=True)
        calibration_summary_df = pd.DataFrame(calib_summary_rows)
        vl.check("calibration_metrics_finite",
                 bool(np.isfinite(calibration_summary_df[["ece", "mce", "brier_score"]].to_numpy()).all()),
                 detail="ece/mce/brier must be finite for all classes")
        logger.info(f"calibration_diagnostics_completed n_classes={len(class_names)} n_bins={CALIBRATION_N_BINS}")

        prob_dist_df = probability_distribution_summary(class_names, y_prob, y_true)
        pathology_df = diagnose_pathologies(class_names, per_class_df, y_prob, y_true, y_pred)
        n_pathological = int(pathology_df["pathological"].sum())
        vl.check("pathology_scan_completed", True,
                 detail=f"{n_pathological}/{len(class_names)} classes flagged pathological", hard=False)
        logger.info(f"pathology_scan n_pathological={n_pathological}/{len(class_names)}")

        opt_by_class = optimal_thresholds_df.set_index("class_name")
        path_by_class = pathology_df.set_index("class_name")
        deploy_rows = [deployment_recommendation(c, per_class_df.loc[c], path_by_class.loc[c], opt_by_class.loc[c])
                       for c in class_names]
        deployment_df = pd.DataFrame(deploy_rows)
        vl.check("deployment_recommendations_generated", len(deployment_df) == len(class_names),
                 detail=f"{len(deployment_df)} recommendations for {len(class_names)} classes")

        log_resource_usage(logger, "stage06_pre_write")

        write_csv(optimal_thresholds_df, STAGE_DIR / "optimal_thresholds.csv")
        write_json(optimal_thresholds_df.to_dict(orient="records"), STAGE_DIR / "optimal_thresholds.json")
        write_csv(roc_pr_curves_df, STAGE_DIR / "roc_pr_threshold_curves.csv")
        write_csv(calibration_bins_df, STAGE_DIR / "reliability_bins.csv")
        write_csv(calibration_summary_df, STAGE_DIR / "calibration_summary.csv")
        write_json(calibration_summary_df.to_dict(orient="records"), STAGE_DIR / "calibration_summary.json")
        write_csv(prob_dist_df, STAGE_DIR / "probability_distribution_summary.csv")
        write_json(prob_dist_df.to_dict(orient="records"), STAGE_DIR / "probability_distribution_summary.json")
        write_csv(pathology_df, STAGE_DIR / "pathology_report.csv")
        write_json(pathology_df.to_dict(orient="records"), STAGE_DIR / "pathology_report.json")
        write_csv(deployment_df, STAGE_DIR / "deployment_recommendations.csv")
        write_json(deployment_df.to_dict(orient="records"), STAGE_DIR / "deployment_recommendations.json")

        for _key, _path in [("optimal_thresholds", STAGE_DIR / "optimal_thresholds.csv"),
                             ("roc_pr_threshold_curves", STAGE_DIR / "roc_pr_threshold_curves.csv"),
                             ("calibration_summary", STAGE_DIR / "calibration_summary.csv"),
                             ("probability_distribution_summary", STAGE_DIR / "probability_distribution_summary.csv"),
                             ("pathology_report", STAGE_DIR / "pathology_report.csv"),
                             ("deployment_recommendations", STAGE_DIR / "deployment_recommendations.csv")]:
            vl.check(f"artifact_written_{_key}", _path.is_file(), detail=str(_path))

        stage6_summary = {
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "n_samples": validated["n_samples"], "n_classes": validated["n_classes"],
            "macro_ece": float(calibration_summary_df["ece"].mean()),
            "macro_mce": float(calibration_summary_df["mce"].mean()),
            "macro_brier_score": float(calibration_summary_df["brier_score"].mean()),
            "n_pathological_classes": n_pathological,
            "n_deploy_candidate_classes": int((deployment_df["verdict"] == "DEPLOY_CANDIDATE").sum()),
            "n_do_not_deploy_classes": int((deployment_df["verdict"] == "DO_NOT_DEPLOY").sum()),
            "n_review_required_classes": int((deployment_df["verdict"] == "REVIEW_REQUIRED").sum()),
            "source_artifacts_consumed": {
                "predictions_parquet": str(PREDICTIONS_PARQUET_PATH),
                "raw_logits_pt": str(RAW_LOGITS_PT_PATH),
                "prediction_metadata": str(PREDICTION_METADATA_PATH),
                "per_class_metrics_csv": str(PER_CLASS_METRICS_CSV_PATH),
                "evaluation_metrics": str(EVALUATION_METRICS_PATH),
            },
            "output_artifacts": {
                "optimal_thresholds_csv": str(STAGE_DIR / "optimal_thresholds.csv"),
                "roc_pr_threshold_curves_csv": str(STAGE_DIR / "roc_pr_threshold_curves.csv"),
                "calibration_summary_csv": str(STAGE_DIR / "calibration_summary.csv"),
                "probability_distribution_summary_csv": str(STAGE_DIR / "probability_distribution_summary.csv"),
                "pathology_report_csv": str(STAGE_DIR / "pathology_report.csv"),
                "deployment_recommendations_csv": str(STAGE_DIR / "deployment_recommendations.csv"),
            },
            "warnings": timer.warnings,
        }
        stage6_summary_path = STAGE_DIR / "stage06_summary.json"
        write_json(stage6_summary, stage6_summary_path)

        engineering_validation = {
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "all_checks_passed": vl.all_passed(),
            "n_checks": len(vl.results),
            "n_passed": vl.n_passed(),
            "n_failed": len(vl.results) - vl.n_passed(),
            "checks": vl.results,
        }
        engineering_validation_path = STAGE_DIR / "engineering_validation.json"
        write_json(engineering_validation, engineering_validation_path)

        if not engineering_validation["all_checks_passed"]:
            raise RuntimeError(
                f"Stage 6 engineering validation FAILED: "
                f"{engineering_validation['n_failed']}/{engineering_validation['n_checks']} checks failed."
            )

        log_resource_usage(logger, "stage06_end")

        stage6_summary["all_engineering_checks_passed"] = engineering_validation["all_checks_passed"]
        stage6_summary["n_checks"] = engineering_validation["n_checks"]
        stage6_summary["_artifact_path"] = str(stage6_summary_path)
        stage6_summary["artifacts"] = {**stage6_summary["output_artifacts"],
                                        "engineering_validation": str(engineering_validation_path)}
        return stage6_summary


STAGE6_SUMMARY = run_stage6()

print("\n" + "=" * 70)
print("STAGE 6 — THRESHOLD OPTIMIZATION & CALIBRATION ENGINE SUMMARY")
print("=" * 70)
print(f"Samples evaluated         : {STAGE6_SUMMARY['n_samples']}")
print(f"Classes                   : {STAGE6_SUMMARY['n_classes']}")
print(f"Macro ECE / MCE / Brier   : {STAGE6_SUMMARY['macro_ece']:.4f} / "
      f"{STAGE6_SUMMARY['macro_mce']:.4f} / {STAGE6_SUMMARY['macro_brier_score']:.4f}")
print(f"Pathological classes      : {STAGE6_SUMMARY['n_pathological_classes']}/{STAGE6_SUMMARY['n_classes']}")
print(f"Deploy candidates         : {STAGE6_SUMMARY['n_deploy_candidate_classes']}")
print(f"Do-not-deploy             : {STAGE6_SUMMARY['n_do_not_deploy_classes']}")
print(f"Review required           : {STAGE6_SUMMARY['n_review_required_classes']}")
print(f"Engineering checks        : {STAGE6_SUMMARY['n_checks']} run, "
      f"all_passed={STAGE6_SUMMARY['all_engineering_checks_passed']}")
print(f"Warnings                  : {len(STAGE6_SUMMARY['warnings'])}")
for w in STAGE6_SUMMARY["warnings"]:
    print(f"  - {w}")
print(f"optimal_thresholds.csv    : {STAGE6_SUMMARY['artifacts']['optimal_thresholds_csv']}")
print(f"roc_pr_threshold_curves.csv: {STAGE6_SUMMARY['artifacts']['roc_pr_threshold_curves_csv']}")
print(f"calibration_summary.csv   : {STAGE6_SUMMARY['artifacts']['calibration_summary_csv']}")
print(f"deployment_recommendations.csv: {STAGE6_SUMMARY['artifacts']['deployment_recommendations_csv']}")
print(f"engineering_validation.json: {STAGE6_SUMMARY['artifacts']['engineering_validation']}")
print("Stage 6 — Threshold Optimization & Calibration Engine : OK")
print("Stage 6 complete. Stage 7+ (Grad-CAM, explainability, ONNX/TensorRT, "
      "MLflow, deployment) not implemented per scope.")

2026-07-05 05:39:47 | INFO    | sprint04_eval.stage06_threshold_calibration_engine | START  stage=stage06_threshold_calibration_engine
2026-07-05 05:39:47 | INFO    | sprint04_eval.stage06_threshold_calibration_engine | resource_usage tag=stage06_start {'gpu_allocated_mb': 9.12, 'gpu_reserved_mb': 90.0, 'cpu_maxrss_mb': 1799.0}
2026-07-05 05:39:47 | INFO    | sprint04_eval.stage06_threshold_calibration_engine | reopened prediction_metadata path=/kaggle/working/sprint04_evaluation/stage04_inference_engine/prediction_metadata.json
2026-07-05 05:39:47 | INFO    | sprint04_eval.stage06_threshold_calibration_engine | reopened per_class_metrics.csv path=/kaggle/working/sprint04_evaluation/stage05_metrics_engine/per_class_metrics.csv rows=14
2026-07-05 05:39:47 | INFO    | sprint04_eval.stage06_threshold_calibration_engine | reopened predictions.parquet path=/kaggle/working/sprint04_evaluation/stage04_inference_engine/predictions.parquet rows=25596
2026-07-05 05:39:47 | INFO    | sprint04_eva

In [8]:
# =============================================================================
# Sprint04_Evaluation.ipynb
# STAGE 7 — Production Explainability & Error Analysis Engine
# -----------------------------------------------------------------------------
# READ-ONLY. Consumes ONLY Stage 1-6 artifacts exactly as written to disk
# (never trusts notebook memory, never reruns inference, never regenerates
# metrics/thresholds/calibration). Stages 1-6 are FROZEN production code.
#
# Scope discipline (per project rules): model interpretability and visual
# error analysis ONLY. No ONNX/TensorRT/TorchScript export, no MLflow, no
# FastAPI/serving/Docker/deployment, no training/checkpointing, no threshold
# optimization, no calibration, no metrics recomputation. Those belong to
# other stages / other pipelines.
#
# ENGINEERING NOTE (2026-07-05 hotfix): the gradient-based explainability
# methods (GradCAM, GradCAM++, Guided Backprop, Integrated Gradients) were
# crashing with:
#   "Output 0 of BackwardHookFunctionBackward is a view and is being
#    modified inplace."
# Root cause: `register_full_backward_hook` was used both on the GradCAM
# target layer (norm5) and on every nn.ReLU inside GuidedBackprop. DenseNet121
# (torchvision/timm) runs an UN-HOOKABLE `F.relu(features, inplace=True)`
# functional call directly inside its own forward() right after `norm5`, and
# separately declares every internal nn.ReLU as `inplace=True`. A full
# backward hook wraps its target's *output* tensor in an internal autograd
# Function; if that wrapped tensor is later modified in place (exactly what
# both of the above do), autograd correctly refuses to proceed. Because
# GradCAM/GradCAM++/GuidedBackprop's hook *objects* were instantiated once
# before the sample loop and only removed after all samples finished, their
# broken hooks stayed attached to the shared model for the entire run and
# also detonated during Integrated Gradients' own (otherwise unrelated and
# correct) backward() calls -- explaining why all four methods failed
# identically, starting at the very first sample and repeating thereafter.
#
# Fix: replaced every module-level `register_full_backward_hook` with a
# forward hook (activations, cloned to break in-place aliasing with
# DenseNet's own in-place ReLU) + a `Tensor.register_hook()` attached
# directly to the live output tensor for gradient capture. Tensor-level
# hooks observe the gradient at the autograd edge that produced the tensor
# and have no restriction on what happens to that tensor object afterward,
# so they are immune to this entire failure class. This changes no CAM
# math and no numeric output versus the intended design (see inline
# docstrings) -- only the crash-prone plumbing was replaced. A circuit
# breaker (MethodHealthTracker) was also added so that if any single method
# ever fails again for an unrelated reason, it is logged once, disabled, and
# every other method + sample continues uninterrupted.
#
# Modularity note: intentionally split into the same functional seams it
# will later become:
#   validators.py           -> Section 7.2-7.4  (reopen + deep validation)
#   model_loader.py          -> Section 7.5       (architecture + checkpoint + transform)
#   cam_engine.py             -> Section 7.6       (target-layer detection, GradCAM,
#                                                    GradCAM++, ScoreCAM, EigenCAM)
#   guided_backprop.py         -> Section 7.7       (guided backpropagation)
#   integrated_gradients.py      -> Section 7.8      (integrated gradients)
#   occlusion.py                  -> Section 7.9     (occlusion sensitivity)
#   sample_selection (part of     -> Section 7.10    (bounded, stratified sample
#     error_analysis.py)                              selection, max 100-200 images)
#   visualization.py                -> Section 7.11   (overlay rendering + PNG writers)
#   error_analysis.py                -> Section 7.12  (error galleries, confusion,
#                                                        collapse, rare-disease failures)
#   artifact_writer.py                 -> Section 7.13 (CSV/JSON artifact writers)
#   run_stage7()                         -> Section 7.14 (orchestration)
#
# Performance requirements honored: inference is NEVER rerun over the full
# 25,596-sample evaluation set. Stage 4's frozen predictions/logits/ground
# truth are reused verbatim for all selection, ranking, and display logic.
# Only a bounded subset (<= STAGE7_CFG.max_total_samples images) receives
# additional forward/backward passes, and only for the purpose of computing
# explainability attributions.
# =============================================================================

import os, sys, json, time, gc, logging, platform, random
from pathlib import Path
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Optional, Dict, List, Tuple, Any, Callable

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as T

import matplotlib
matplotlib.use("Agg")  # headless, deterministic PNG rendering — no display backend required
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm

try:
    import timm
except ImportError as exc:
    raise ImportError(
        "timm is required for Stage 7 (architecture reconstruction, mirrors "
        "Stages 2/4) but is not installed."
    ) from exc

# -----------------------------------------------------------------------------
# 7.0  Re-derive paths independently (do not trust in-memory Stage 1-6 state)
# -----------------------------------------------------------------------------
_KAGGLE_INPUT = Path("/kaggle/input")
_KAGGLE_WORKING = Path("/kaggle/working")
IS_KAGGLE = _KAGGLE_INPUT.is_dir()
OUTPUT_ROOT = (_KAGGLE_WORKING / "sprint04_evaluation") if IS_KAGGLE else Path(os.environ.get("EVAL_OUTPUT_ROOT", "sprint04_evaluation"))

STAGE1_SUMMARY_PATH = OUTPUT_ROOT / "stage01_environment" / "environment_summary.json"
STAGE2_SUMMARY_PATH = OUTPUT_ROOT / "stage02_artifact_loader" / "artifact_validation.json"
STAGE3_SUMMARY_PATH = OUTPUT_ROOT / "stage03_dataset_builder" / "dataset_summary.json"
STAGE4_SUMMARY_PATH = OUTPUT_ROOT / "stage04_inference_engine" / "inference_summary.json"
STAGE5_SUMMARY_PATH = OUTPUT_ROOT / "stage05_metrics_engine" / "evaluation_summary.json"
STAGE5_ENGVAL_PATH  = OUTPUT_ROOT / "stage05_metrics_engine" / "engineering_validation.json"
STAGE6_SUMMARY_PATH = OUTPUT_ROOT / "stage06_threshold_calibration_engine" / "stage06_summary.json"
STAGE6_ENGVAL_PATH  = OUTPUT_ROOT / "stage06_threshold_calibration_engine" / "engineering_validation.json"

for _label, _p in [
    ("Stage 1", STAGE1_SUMMARY_PATH), ("Stage 2", STAGE2_SUMMARY_PATH),
    ("Stage 3", STAGE3_SUMMARY_PATH), ("Stage 4", STAGE4_SUMMARY_PATH),
    ("Stage 5", STAGE5_SUMMARY_PATH), ("Stage 5 engineering_validation", STAGE5_ENGVAL_PATH),
    ("Stage 6", STAGE6_SUMMARY_PATH), ("Stage 6 engineering_validation", STAGE6_ENGVAL_PATH),
]:
    if not _p.is_file():
        raise FileNotFoundError(
            f"{_label} output not found at {_p}. Stage 7 never regenerates prior "
            f"stages — run Stages 1-6 first, in order. Stages 1-6 are frozen."
        )

def _load_json(path: Path) -> Dict[str, Any]:
    try:
        with open(path) as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(f"Corrupted JSON at {path}: {e}") from e

STAGE1_SUMMARY = _load_json(STAGE1_SUMMARY_PATH)
STAGE2_REPORT  = _load_json(STAGE2_SUMMARY_PATH)
STAGE3_SUMMARY = _load_json(STAGE3_SUMMARY_PATH)
STAGE4_SUMMARY = _load_json(STAGE4_SUMMARY_PATH)
STAGE5_SUMMARY = _load_json(STAGE5_SUMMARY_PATH)
STAGE5_ENGVAL  = _load_json(STAGE5_ENGVAL_PATH)
STAGE6_SUMMARY = _load_json(STAGE6_SUMMARY_PATH)
STAGE6_ENGVAL  = _load_json(STAGE6_ENGVAL_PATH)

# --- hard stop unless every prior stage is fully green -----------------------
if not STAGE4_SUMMARY.get("all_validation_checks_passed", False):
    raise RuntimeError(
        "Stage 7 aborted — Stage 4's inference_summary.json reports "
        "all_validation_checks_passed=False. Stage 7 must never explain an "
        "inference run that failed its own engineering validation."
    )
if not STAGE5_ENGVAL.get("all_checks_passed", False):
    raise RuntimeError(
        "Stage 7 aborted — Stage 5's engineering_validation.json reports "
        "all_checks_passed=False. Stage 5 must be fully green (frozen, trusted) "
        "before Stage 7 runs."
    )
if not STAGE6_ENGVAL.get("all_checks_passed", False):
    raise RuntimeError(
        "Stage 7 aborted — Stage 6's engineering_validation.json reports "
        "all_checks_passed=False. Stage 6 must be fully green (frozen, trusted) "
        "before Stage 7 runs."
    )

RESOLVED_PATHS = {k: Path(v) for k, v in STAGE2_REPORT["resolved_paths"].items()}
REQUIRED_FOR_STAGE7 = ["best_model", "training_summary", "disease_registry"]
_missing = [k for k in REQUIRED_FOR_STAGE7 if k not in RESOLVED_PATHS]
if _missing:
    raise FileNotFoundError(
        f"Stage 7 requires resolved_paths entries {_missing} from Stage 2's "
        f"artifact_validation.json, but they are absent. Rerun Stage 2. Stage 7 "
        f"never rediscovers checkpoints on its own — this is a hard stop."
    )

_STAGE4_ARTIFACTS = STAGE4_SUMMARY.get("artifacts", {})
for _key in ["predictions_parquet", "raw_logits_pt", "prediction_metadata"]:
    if _key not in _STAGE4_ARTIFACTS:
        raise FileNotFoundError(
            f"Stage 4's inference_summary.json is missing artifacts.{_key}. "
            f"Stage 7 cannot proceed without a complete Stage 4 artifact manifest."
        )

_STAGE5_ARTIFACTS = STAGE5_SUMMARY.get("output_artifacts", {})
for _key in ["evaluation_metrics", "per_class_metrics_csv"]:
    if _key not in _STAGE5_ARTIFACTS:
        raise FileNotFoundError(
            f"Stage 5's evaluation_summary.json is missing output_artifacts.{_key}. "
            f"Stage 7 cannot proceed without a complete Stage 5 artifact manifest."
        )

_STAGE6_ARTIFACTS = STAGE6_SUMMARY.get("output_artifacts", {})
for _key in ["optimal_thresholds_csv", "calibration_summary_csv", "deployment_recommendations_csv"]:
    if _key not in _STAGE6_ARTIFACTS:
        raise FileNotFoundError(
            f"Stage 6's stage06_summary.json is missing output_artifacts.{_key}. "
            f"Stage 7 cannot proceed without a complete Stage 6 artifact manifest."
        )

PREDICTIONS_PARQUET_PATH        = Path(_STAGE4_ARTIFACTS["predictions_parquet"])
RAW_LOGITS_PT_PATH              = Path(_STAGE4_ARTIFACTS["raw_logits_pt"])
PREDICTION_METADATA_PATH        = Path(_STAGE4_ARTIFACTS["prediction_metadata"])
EVALUATION_METRICS_PATH         = Path(_STAGE5_ARTIFACTS["evaluation_metrics"])
PER_CLASS_METRICS_CSV_PATH      = Path(_STAGE5_ARTIFACTS["per_class_metrics_csv"])
OPTIMAL_THRESHOLDS_CSV_PATH     = Path(_STAGE6_ARTIFACTS["optimal_thresholds_csv"])
CALIBRATION_SUMMARY_CSV_PATH    = Path(_STAGE6_ARTIFACTS["calibration_summary_csv"])
DEPLOYMENT_RECOMMENDATIONS_PATH = Path(_STAGE6_ARTIFACTS["deployment_recommendations_csv"])
DATASET_SUMMARY_CSV_PATH        = Path(STAGE3_SUMMARY["outputs"]["dataset_summary_csv"])
BEST_MODEL_PATH                 = RESOLVED_PATHS["best_model"]
TRAINING_SUMMARY_PATH           = RESOLVED_PATHS["training_summary"]

for _label, _p in [
    ("predictions.parquet", PREDICTIONS_PARQUET_PATH), ("raw_logits.pt", RAW_LOGITS_PT_PATH),
    ("prediction_metadata.json", PREDICTION_METADATA_PATH),
    ("evaluation_metrics.json", EVALUATION_METRICS_PATH),
    ("per_class_metrics.csv", PER_CLASS_METRICS_CSV_PATH),
    ("optimal_thresholds.csv", OPTIMAL_THRESHOLDS_CSV_PATH),
    ("calibration_summary.csv", CALIBRATION_SUMMARY_CSV_PATH),
    ("deployment_recommendations.csv", DEPLOYMENT_RECOMMENDATIONS_PATH),
    ("dataset_summary.csv", DATASET_SUMMARY_CSV_PATH),
    ("best_model.pt", BEST_MODEL_PATH), ("training_summary.json", TRAINING_SUMMARY_PATH),
]:
    if not _p.is_file():
        raise FileNotFoundError(f"Artifact {_label} not found at {_p} even though it is "
                                 f"listed in its stage's own summary. That stage must be rerun.")

STAGE_DIR = OUTPUT_ROOT / "stage07_interpretability_engine"
LOG_DIR = OUTPUT_ROOT / "logs"
SUBDIRS = {
    "gradcam":              STAGE_DIR / "gradcam",
    "gradcam_plus":         STAGE_DIR / "gradcam_plus",
    "scorecam":             STAGE_DIR / "scorecam",
    "eigencam":             STAGE_DIR / "eigencam",
    "guided_backprop":      STAGE_DIR / "guided_backprop",
    "integrated_gradients": STAGE_DIR / "integrated_gradients",
    "occlusion":            STAGE_DIR / "occlusion",
    "error_gallery":        STAGE_DIR / "error_gallery",
    "correct_predictions":  STAGE_DIR / "error_gallery" / "correct_predictions",
    "incorrect_predictions": STAGE_DIR / "error_gallery" / "incorrect_predictions",
    "summary":              STAGE_DIR / "summary",
    "logs":                 STAGE_DIR / "logs",
}
for _d in [STAGE_DIR, LOG_DIR, *SUBDIRS.values()]:
    _d.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# 7.1  Logging / timing / resource usage (identical pattern to Stages 1-6,
#      reproduced for per-stage independence)
# -----------------------------------------------------------------------------
def setup_logging(log_dir: Path, stage_name: str) -> logging.Logger:
    logger = logging.getLogger(f"sprint04_eval.{stage_name}")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False
    fmt = logging.Formatter(fmt="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
                             datefmt="%Y-%m-%d %H:%M:%S")
    console = logging.StreamHandler(sys.stdout); console.setFormatter(fmt)
    logger.addHandler(console)
    fh = logging.FileHandler(log_dir / f"{stage_name}.log", mode="a"); fh.setFormatter(fmt)
    logger.addHandler(fh)
    return logger

logger = setup_logging(LOG_DIR, "stage07_interpretability_engine")

class StageTimer:
    def __init__(self, stage_name: str, log: logging.Logger):
        self.stage_name, self.log, self.t0, self.warnings = stage_name, log, None, []
    def warn(self, msg: str) -> None:
        self.warnings.append(msg); self.log.warning(msg)
    def __enter__(self):
        self.t0 = time.perf_counter()
        self.log.info(f"START  stage={self.stage_name}")
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        dur = time.perf_counter() - self.t0
        status = "FAILED" if exc_type else "FINISH"
        self.log.info(f"{status} stage={self.stage_name} duration_sec={dur:.2f} warnings={len(self.warnings)}")
        return False

def log_resource_usage(log: logging.Logger, tag: str) -> Dict[str, float]:
    usage = {}
    if torch.cuda.is_available():
        usage["gpu_allocated_mb"] = round(torch.cuda.memory_allocated() / 1024**2, 2)
        usage["gpu_reserved_mb"] = round(torch.cuda.memory_reserved() / 1024**2, 2)
        usage["gpu_peak_allocated_mb"] = round(torch.cuda.max_memory_allocated() / 1024**2, 2)
    try:
        import resource
        usage["cpu_maxrss_mb"] = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1)
    except Exception:
        pass
    log.info(f"resource_usage tag={tag} {usage}")
    return usage


class ValidationLedger:
    """Accumulates PASS/FAIL engineering checks. Hard checks raise on failure
    (fail loudly, no silent fallbacks); soft/informational checks are
    recorded but do not abort the stage."""

    def __init__(self, log: logging.Logger):
        self.log = log
        self.results: List[Dict[str, Any]] = []

    def check(self, name: str, passed: bool, detail: str = "", hard: bool = True) -> None:
        status = "PASS" if passed else "FAIL"
        entry = {"check": name, "status": status, "detail": detail}
        self.results.append(entry)
        (self.log.info if passed else self.log.error)(f"validation check={name} status={status} detail={detail}")
        if hard and not passed:
            raise RuntimeError(f"Stage 7 aborted — validation check {name!r} FAILED: {detail}")

    def n_passed(self) -> int:
        return sum(1 for r in self.results if r["status"] == "PASS")

    def all_passed(self) -> bool:
        return all(r["status"] == "PASS" for r in self.results)


class MethodHealthTracker:
    """Circuit breaker for explainability methods (production robustness
    requirement): if a method fails, the root cause is logged ONCE, that
    method is permanently disabled for the remainder of the run, and every
    other method + every other sample continues untouched. Threshold is
    configurable via Stage7Config.max_failures_before_disabling_method so a
    single transient failure doesn't necessarily kill a method if the team
    later wants more tolerance; default is 1 (fail-fast, no silent retries)."""

    def __init__(self, method_names: List[str], max_failures: int, log: logging.Logger):
        self.max_failures = max(1, max_failures)
        self.log = log
        self.failure_counts = {m: 0 for m in method_names}
        self.disabled = {m: False for m in method_names}
        self.disabled_reason: Dict[str, Optional[str]] = {m: None for m in method_names}

    def should_attempt(self, method: str) -> bool:
        return not self.disabled[method]

    def record_failure(self, method: str, error: str) -> None:
        self.failure_counts[method] += 1
        if not self.disabled[method] and self.failure_counts[method] >= self.max_failures:
            self.disabled[method] = True
            self.disabled_reason[method] = error
            self.log.error(
                f"method_disabled method={method} "
                f"failures={self.failure_counts[method]}/{self.max_failures} "
                f"reason={error!r} -- this method is now skipped for all "
                f"remaining samples; no further duplicate errors will be logged "
                f"for it."
            )

    def summary(self) -> Dict[str, Any]:
        return {
            m: {"failures": self.failure_counts[m], "disabled": self.disabled[m],
                "disabled_reason": self.disabled_reason[m]}
            for m in self.failure_counts
        }


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =============================================================================
# 7.2  Configuration
# =============================================================================
@dataclass(frozen=True)
class Stage7Config:
    random_seed: int = 42
    max_total_samples: int = 150          # hard cap per project spec (100-200)
    n_per_class_representative: int = 1
    ig_steps: int = 32                    # batched integrated-gradients path steps
    scorecam_max_channels: int = 64       # channel subsample for ScoreCAM (perf tradeoff)
    scorecam_batch_size: int = 32
    occlusion_patch_size: int = 32
    occlusion_stride: int = 16
    occlusion_batch_size: int = 64
    occlusion_baseline_value: float = 0.0  # baseline pixel value in *normalized* tensor space
    overlay_alpha: float = 0.45
    n_worst_calibrated_classes: int = 3
    n_rare_disease_classes: int = 3
    max_failures_before_disabling_method: int = 1  # circuit breaker threshold (production robustness)

STAGE7_CFG = Stage7Config()
set_global_seed(STAGE7_CFG.random_seed)
RNG = np.random.RandomState(STAGE7_CFG.random_seed)


# =============================================================================
# 7.3  Reopen Stage 1-6 inputs (validators.py — never rediscover, never rebuild)
# =============================================================================
def reopen_stage7_inputs(log: logging.Logger) -> Dict[str, Any]:
    training_summary = _load_json(TRAINING_SUMMARY_PATH)
    disease_registry = _load_json(RESOLVED_PATHS["disease_registry"])
    prediction_metadata = _load_json(PREDICTION_METADATA_PATH)
    raw_logits_bundle = torch.load(RAW_LOGITS_PT_PATH, map_location="cpu", weights_only=False)
    predictions_df = pd.read_parquet(PREDICTIONS_PARQUET_PATH)
    dataset_manifest_df = pd.read_csv(DATASET_SUMMARY_CSV_PATH)
    dataset_manifest_df["label_vector"] = dataset_manifest_df["label_vector"].map(json.loads)
    per_class_metrics_df = pd.read_csv(PER_CLASS_METRICS_CSV_PATH)
    optimal_thresholds_df = pd.read_csv(OPTIMAL_THRESHOLDS_CSV_PATH)
    calibration_summary_df = pd.read_csv(CALIBRATION_SUMMARY_CSV_PATH)
    deployment_recommendations_df = pd.read_csv(DEPLOYMENT_RECOMMENDATIONS_PATH)
    log.info(f"reopened training_summary path={TRAINING_SUMMARY_PATH}")
    log.info(f"reopened disease_registry path={RESOLVED_PATHS['disease_registry']}")
    log.info(f"reopened predictions.parquet rows={len(predictions_df)} path={PREDICTIONS_PARQUET_PATH}")
    log.info(f"reopened dataset_summary.csv rows={len(dataset_manifest_df)} path={DATASET_SUMMARY_CSV_PATH}")
    return {
        "training_summary": training_summary,
        "disease_registry": disease_registry,
        "prediction_metadata": prediction_metadata,
        "raw_logits_bundle": raw_logits_bundle,
        "predictions_df": predictions_df,
        "dataset_manifest_df": dataset_manifest_df,
        "per_class_metrics_df": per_class_metrics_df,
        "optimal_thresholds_df": optimal_thresholds_df,
        "calibration_summary_df": calibration_summary_df,
        "deployment_recommendations_df": deployment_recommendations_df,
    }


def validate_cross_artifact_consistency(inputs: Dict[str, Any], vl: ValidationLedger,
                                         log: logging.Logger) -> Dict[str, Any]:
    """validators.py — engineering validation performed BEFORE any explainability
    computation. Aborts immediately (raises) if any hard check fails."""
    pm = inputs["prediction_metadata"]
    raw = inputs["raw_logits_bundle"]
    df = inputs["predictions_df"]
    per_class_df = inputs["per_class_metrics_df"]
    dataset_class_names = STAGE3_SUMMARY["class_names"]

    pm_class_names = pm["class_names"]
    raw_class_names = list(raw["class_names"])
    csv_class_names = per_class_df["class_name"].tolist()
    thr_class_names = inputs["optimal_thresholds_df"]["class_name"].tolist()
    calib_class_names = inputs["calibration_summary_df"]["class_name"].tolist()

    all_match = (dataset_class_names == pm_class_names == raw_class_names ==
                 csv_class_names == thr_class_names == calib_class_names)
    vl.check("class_order_consistent_all_stages", all_match,
             detail=f"stage3/stage4(pm)/stage4(raw)/stage5(csv)/stage6(thr)/stage6(calib) "
                    f"class name lists {'match exactly' if all_match else 'DIFFER'}")

    class_names = dataset_class_names
    n_classes = len(class_names)
    vl.check("class_count_consistent",
             len(pm_class_names) == len(raw_class_names) == len(csv_class_names) == n_classes,
             detail=f"stage3={n_classes} pm={len(pm_class_names)} raw={len(raw_class_names)} csv={len(csv_class_names)}")

    n_samples_expected = pm["n_samples"]
    vl.check("prediction_count_matches_metadata", len(df) == n_samples_expected,
             detail=f"predictions.parquet_rows={len(df)} prediction_metadata.n_samples={n_samples_expected}")
    vl.check("dataset_manifest_row_count_matches", len(inputs["dataset_manifest_df"]) == n_samples_expected,
             detail=f"dataset_summary.csv_rows={len(inputs['dataset_manifest_df'])} expected={n_samples_expected}")

    gt_arr = np.array(df["ground_truth"].tolist(), dtype=np.float64)
    prob_arr = np.array(df["probabilities"].tolist(), dtype=np.float64)
    vl.check("array_shapes_correct",
             gt_arr.shape == prob_arr.shape == (len(df), n_classes),
             detail=f"ground_truth={gt_arr.shape} probabilities={prob_arr.shape} expected=({len(df)},{n_classes})")
    vl.check("no_nan_reloaded_arrays", not (np.isnan(gt_arr).any() or np.isnan(prob_arr).any()),
             detail="checked ground_truth, probabilities")
    vl.check("probabilities_in_unit_range", bool((prob_arr >= 0).all() and (prob_arr <= 1).all()),
             detail=f"min={prob_arr.min():.6f} max={prob_arr.max():.6f}")

    ts_model = inputs["training_summary"]["model"]
    backbone_name = ts_model["backbone"]
    vl.check("architecture_backbone_is_densenet121", backbone_name == "densenet121",
             detail=f"backbone={backbone_name}")
    vl.check("architecture_num_classes_matches", ts_model["num_classes"] == n_classes,
             detail=f"training_summary.num_classes={ts_model['num_classes']} class_names={n_classes}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    vl.check("device_resolved", True, detail=f"device={device}", hard=False)

    log.info(f"cross_artifact_validation_passed n_samples={len(df)} n_classes={n_classes} "
             f"checks_passed={vl.n_passed()}/{len(vl.results)}")

    return {
        "class_names": class_names, "n_classes": n_classes, "n_samples": len(df),
        "y_true": gt_arr.astype(int), "y_prob": prob_arr,
        "fixed_threshold": pm.get("threshold", 0.5), "device": device,
    }


# =============================================================================
# 7.4  Model architecture reconstruction (mirrors Stages 2/4 exactly —
#      duplicated deliberately for per-stage independence)
# =============================================================================
@dataclass(frozen=True)
class BackboneSpec:
    timm_name: str
    family: str
    feature_dim: int
    classifier_attr: str
    status: str

BACKBONE_REGISTRY: Dict[str, BackboneSpec] = {
    "densenet121": BackboneSpec("densenet121", "DenseNet", 1024, "classifier", "ACTIVE"),
}

def _get_module_by_path(module: nn.Module, path: str) -> nn.Module:
    obj = module
    for part in path.split("."):
        obj = getattr(obj, part)
    return obj

def _set_module_by_path(module: nn.Module, path: str, new_submodule: nn.Module) -> None:
    parts = path.split(".")
    obj = module
    for part in parts[:-1]:
        obj = getattr(obj, part)
    setattr(obj, parts[-1], new_submodule)

def get_backbone(backbone_name: str, pretrained: bool = False) -> nn.Module:
    spec = BACKBONE_REGISTRY[backbone_name]
    return timm.create_model(spec.timm_name, pretrained=pretrained)

def replace_classifier(model: nn.Module, backbone_name: str, num_classes: int, dropout: float = 0.3):
    spec = BACKBONE_REGISTRY[backbone_name]
    old_head = _get_module_by_path(model, spec.classifier_attr)
    in_features = old_head.in_features
    new_head = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_features, num_classes))
    _set_module_by_path(model, spec.classifier_attr, new_head)
    return model, in_features

class ChestXrayClassifier(nn.Module):
    def __init__(self, backbone: nn.Module, backbone_name: str, num_classes: int) -> None:
        super().__init__()
        self.backbone = backbone
        self.backbone_name = backbone_name
        self.num_classes = num_classes
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)

def build_model_for_eval(backbone_name: str, num_classes: int, dropout: float) -> ChestXrayClassifier:
    raw_model = get_backbone(backbone_name, pretrained=False)
    raw_model, _ = replace_classifier(raw_model, backbone_name, num_classes, dropout)
    return ChestXrayClassifier(raw_model, backbone_name, num_classes)

def extract_state_dict(checkpoint_obj: Any) -> Dict[str, torch.Tensor]:
    if isinstance(checkpoint_obj, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            if key in checkpoint_obj and isinstance(checkpoint_obj[key], dict):
                return checkpoint_obj[key]
        if all(torch.is_tensor(v) for v in checkpoint_obj.values()):
            return checkpoint_obj
    raise ValueError(
        "Could not locate a model state_dict inside the checkpoint object. "
        f"Top-level type={type(checkpoint_obj)}"
    )

_STATE_DICT_TRANSFORMS = [
    ("as-is", lambda sd: sd),
    ("strip 'module.' (DataParallel)", lambda sd: {k.replace("module.", "", 1): v for k, v in sd.items()}),
    ("strip 'backbone.' prefix", lambda sd: {k.replace("backbone.", "", 1): v for k, v in sd.items()}),
]

def load_checkpoint_into_model(ckpt_path: Path, model: nn.Module, preferred_label: Optional[str],
                                log: logging.Logger) -> Dict[str, Any]:
    try:
        raw = torch.load(ckpt_path, map_location="cpu", weights_only=True)
        load_mode = "weights_only=True"
    except Exception as e:
        log.warning(f"weights_only=True load failed ({e}); retrying with weights_only=False "
                    f"(trusted first-party checkpoint from our own Sprint04 pipeline).")
        raw = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        load_mode = "weights_only=False (fallback)"

    state_dict = extract_state_dict(raw)
    ordered = _STATE_DICT_TRANSFORMS
    if preferred_label:
        ordered = sorted(_STATE_DICT_TRANSFORMS, key=lambda t: t[0] != preferred_label)

    attempts = []
    for label, transform_fn in ordered:
        transformed = transform_fn(state_dict)
        try:
            model.load_state_dict(transformed, strict=True)
            attempts.append({"transform": label, "result": "success"})
            return {"load_transform_used": label, "load_mode": load_mode, "attempts": attempts,
                    "param_count_loaded": sum(v.numel() for v in transformed.values())}
        except RuntimeError as e:
            attempts.append({"transform": label, "result": "failed", "error": str(e)[:300]})
            continue

    raise RuntimeError(
        f"Checkpoint {ckpt_path} is architecture-incompatible with "
        f"ChestXrayClassifier(densenet121). All load strategies failed: {attempts}"
    )


def build_eval_transform(cfg: Dict[str, Any], log: logging.Logger) -> Tuple[T.Compose, Tuple[int, int]]:
    """Reconstructs the EXACT eval transform Stage 3 resolved and recorded in
    dataset_summary.json['transform'] — never re-derives it independently."""
    size = cfg["image_size"]
    hw = (int(size[0]), int(size[1])) if isinstance(size, (list, tuple)) else (int(size), int(size))
    mean, std = cfg["mean"], cfg["std"]
    interp_map = {
        "bilinear": T.InterpolationMode.BILINEAR,
        "bicubic": T.InterpolationMode.BICUBIC,
        "nearest": T.InterpolationMode.NEAREST,
    }
    interp_name = cfg.get("interpolation", "bilinear")
    if interp_name not in interp_map:
        raise ValueError(f"Unsupported interpolation {interp_name!r} recorded in dataset_summary.json.")
    transform = T.Compose([
        T.Resize(hw, interpolation=interp_map[interp_name]),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])
    log.info(f"eval_transform reconstructed image_size={hw} mean={mean} std={std} interpolation={interp_name}")
    return transform, hw


# =============================================================================
# 7.5  model_loader.py — orchestrates architecture build + checkpoint load +
#      deep engineering validation (checkpoint compatible, eval mode, target
#      layer exists, tensor dims, etc.) BEFORE any explainability computation.
# =============================================================================
def load_and_validate_model(inputs: Dict[str, Any], validated: Dict[str, Any],
                             vl: ValidationLedger, log: logging.Logger) -> Dict[str, Any]:
    ts_model = inputs["training_summary"]["model"]
    backbone_name = ts_model["backbone"]
    num_classes = ts_model["num_classes"]
    dropout = ts_model["dropout"]
    device = validated["device"]

    model = build_model_for_eval(backbone_name, num_classes, dropout)
    vl.check("model_constructed", model is not None, detail=f"backbone={backbone_name} num_classes={num_classes}")

    preferred_label = STAGE2_REPORT.get("checkpoint_compatibility", {}).get("load_transform_used")
    load_result = load_checkpoint_into_model(BEST_MODEL_PATH, model, preferred_label, log)
    vl.check("checkpoint_loads_successfully", True, detail=load_result["load_transform_used"])
    vl.check("checkpoint_compatible_densenet121_backbone", backbone_name == "densenet121",
             detail=f"backbone={backbone_name}")
    vl.check("checkpoint_num_classes_matches", num_classes == validated["n_classes"],
             detail=f"checkpoint_num_classes={num_classes} class_names={validated['n_classes']}")

    model.to(device)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    vl.check("model_in_eval_mode", not model.training, detail="model.training==False")

    transform, expected_hw = build_eval_transform(STAGE3_SUMMARY["transform"], log)
    vl.check("image_transforms_reconstructed", transform is not None,
             detail=f"image_size={expected_hw}")

    # --- tensor dimension smoke test: single dummy forward pass -------------
    dummy = torch.zeros(1, 3, expected_hw[0], expected_hw[1], device=device)
    with torch.no_grad():
        dummy_out = model(dummy)
    vl.check("model_output_shape_correct", tuple(dummy_out.shape) == (1, validated["n_classes"]),
             detail=f"output_shape={tuple(dummy_out.shape)} expected=(1,{validated['n_classes']})")
    vl.check("no_nan_in_dummy_forward", not torch.isnan(dummy_out).any().item(),
             detail="dummy forward pass produced no NaNs")

    vl.check("gpu_availability_checked", True,
             detail=f"cuda_available={torch.cuda.is_available()} device={device}", hard=False)

    # --- GradCAM target layer existence check (abort on architecture mismatch) --
    target_layer, target_name, target_shape = locate_target_layer(model, expected_hw, device, log)
    vl.check("gradcam_target_layer_exists", target_layer is not None,
             detail=f"layer=backbone.features.{target_name} shape={target_shape}")

    log.info(f"model_loaded_and_validated backbone={backbone_name} num_classes={num_classes} "
             f"checkpoint_strategy={load_result['load_transform_used']} target_layer={target_name}")

    return {
        "model": model, "device": device, "transform": transform, "expected_hw": expected_hw,
        "target_layer": target_layer, "target_layer_name": target_name,
        "target_layer_shape": target_shape, "checkpoint_info": load_result,
        "class_names": validated["class_names"],
    }


# =============================================================================
# 7.6  cam_engine.py — target-layer auto-detection, GradCAM, GradCAM++,
#      ScoreCAM, EigenCAM
# =============================================================================
def locate_target_layer(model: nn.Module, expected_hw: Tuple[int, int],
                         device: torch.device, log: logging.Logger):
    """Automatically detects the last 4D (B,C,H,W) activation-producing child
    of backbone.features by probing a dummy forward pass with hooks on every
    direct child — never hardcodes a layer name. For DenseNet121 this
    reliably resolves to the final batch-norm layer (norm5) that follows the
    last dense block, which is the conventional GradCAM target for DenseNet
    architectures. Aborts (raises) if the architecture does not match the
    expected DenseNet121 'features' Sequential shape."""
    backbone = getattr(model, "backbone", None)
    if backbone is None:
        raise RuntimeError("Stage 7 aborted — model has no 'backbone' attribute; "
                            "architecture mismatch with Stage 2/4's ChestXrayClassifier.")
    features = getattr(backbone, "features", None)
    if features is None or not isinstance(features, nn.Module):
        raise RuntimeError("Stage 7 aborted — backbone has no 'features' submodule; "
                            "DenseNet121 architecture mismatch.")

    if not named_children:
        raise RuntimeError("Stage 7 aborted — backbone.features has no child modules "
                            "to probe for a GradCAM target layer.")

    captured: Dict[str, torch.Tensor] = {}
    handles = []
    def _make_hook(name):
        def _hook(module, inp, out):
            captured[name] = out
        return _hook
    for name, module in named_children:
        handles.append(module.register_forward_hook(_make_hook(name)))

    was_training = model.training
    model.eval()
    dummy = torch.zeros(1, 3, expected_hw[0], expected_hw[1], device=device)
    with torch.no_grad():
        features(dummy)
    for h in handles:
        h.remove()
    if was_training:
        model.train()

    target_name, target_module, target_shape = None, None, None
    for name, module in named_children:  # preserve definition order; last 4D wins
        out = captured.get(name)
        if out is not None and out.dim() == 4:
            target_name, target_module, target_shape = name, module, tuple(out.shape)

    if target_module is None:
        raise RuntimeError("Stage 7 aborted — no child of backbone.features produced a "
                            "4D (B,C,H,W) activation map during the probe forward pass; "
                            "cannot auto-detect a GradCAM target layer.")

    log.info(f"gradcam_target_layer_detected name=features.{target_name} shape={target_shape}")
    return target_module, target_name, target_shape


def normalize_map(cam: np.ndarray) -> np.ndarray:
    cam = np.nan_to_num(cam.astype(np.float64), nan=0.0, posinf=0.0, neginf=0.0)
    cmin, cmax = float(cam.min()), float(cam.max())
    if (cmax - cmin) < 1e-12:
        return np.zeros_like(cam, dtype=np.float64)
    return (cam - cmin) / (cmax - cmin)


def resize_map_to_image(cam: np.ndarray, size_hw: Tuple[int, int]) -> np.ndarray:
    t = torch.tensor(cam, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    t = F.interpolate(t, size=size_hw, mode="bilinear", align_corners=False)
    return t.squeeze(0).squeeze(0).numpy()


class _CAMHookBase:
    """Shared forward-hook + tensor-level-gradient-hook plumbing for GradCAM
    and GradCAM++ (the two methods differ ONLY in how per-channel weights
    are derived from the same captured activations/gradients).

    HOTFIX (root cause, not a workaround): this class previously registered a
    `register_full_backward_hook` on the target layer (norm5). DenseNet121's
    own forward() (torchvision/timm source) runs `F.relu(features,
    inplace=True)` directly on norm5's output one line later -- a raw
    functional in-place call baked into forward(), not a submodule we can
    intercept. A full backward hook wraps its target's *returned* tensor in
    an internal autograd Function (BackwardHookFunction) to allow rewriting
    grad_input/grad_output; when that wrapped tensor is subsequently modified
    in place, autograd correctly refuses with "Output 0 of
    BackwardHookFunctionBackward is a view and is being modified inplace."
    This is a documented limitation of full backward hooks on architectures
    with in-place ops -- it cannot be fixed by toggling nn.ReLU(inplace=...)
    flags, because the offending op here isn't a module at all.

    Fix: never wrap the target layer's output. Capture activations via a
    plain forward hook (cloned immediately, so our copy is immune to the
    later in-place mutation of the live tensor), and capture gradients via a
    `Tensor.register_hook()` attached directly to the live output tensor
    inside that same forward hook. A tensor hook observes the gradient at
    the autograd edge that produced the tensor and has no requirement that
    the tensor remain untouched afterward -- so it is structurally immune to
    this entire failure class, on this architecture or any other.
    """

    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.activations: Optional[torch.Tensor] = None
        self.gradients: Optional[torch.Tensor] = None
        self._fwd = target_layer.register_forward_hook(self._forward_hook)

    def _forward_hook(self, module, inp, out):
        # Clone+detach immediately: `out` is the exact same tensor object
        # DenseNet's forward() mutates in place one line later via
        # F.relu(features, inplace=True). Without cloning, whatever we read
        # from self.activations after backward() would depend on execution
        # order rather than being an explicit choice. We apply that same
        # ReLU ourselves, out-of-place, on our private copy -- this changes
        # NO numeric output versus the intended design (CAM methods
        # conventionally use the post-activation feature map of the final
        # conv/bn block); it only removes the implicit dependency on
        # incidental in-place aliasing.
        self.activations = torch.relu(out.detach().clone())
        if out.requires_grad:
            out.register_hook(self._save_gradient)

    def _save_gradient(self, grad: torch.Tensor) -> None:
        self.gradients = grad.detach().clone()

    def remove(self) -> None:
        self._fwd.remove()

    def _forward_backward(self, input_tensor: torch.Tensor, class_idx: int) -> None:
        self.gradients = None
        self.model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            x = input_tensor.clone().requires_grad_(True)
            logits = self.model(x)
            score = logits[0, class_idx]
            score.backward()
        if self.gradients is None:
            raise RuntimeError(
                f"{type(self).__name__}: target layer produced no gradient "
                f"(tensor hook never fired) -- the target layer's output did "
                f"not require grad, or was not on the path to the score. This "
                f"is an architecture-level problem and must not be papered "
                f"over with a fallback CAM."
            )

    def _weights(self) -> torch.Tensor:
        raise NotImplementedError

    def generate(self, input_tensor: torch.Tensor, class_idx: int) -> np.ndarray:
        self._forward_backward(input_tensor, class_idx)
        weights = self._weights()  # (1,C,1,1)
        cam = torch.relu((weights * self.activations).sum(dim=1, keepdim=True))
        return cam.squeeze(0).squeeze(0).detach().cpu().numpy()


class GradCAM(_CAMHookBase):
    def _weights(self) -> torch.Tensor:
        return self.gradients.mean(dim=(2, 3), keepdim=True)


class GradCAMPlusPlus(_CAMHookBase):
    def _weights(self) -> torch.Tensor:
        grads = self.gradients
        acts = self.activations
        grads_2 = grads ** 2
        grads_3 = grads_2 * grads
        sum_acts = acts.sum(dim=(2, 3), keepdim=True)
        alpha_denom = 2.0 * grads_2 + sum_acts * grads_3
        alpha_denom = torch.where(alpha_denom.abs() > 1e-8, alpha_denom, torch.ones_like(alpha_denom))
        alpha = grads_2 / alpha_denom
        weights = (alpha * torch.relu(grads)).sum(dim=(2, 3), keepdim=True)
        return weights


class ScoreCAM:
    """No gradients required. Approximation for performance: only the top-K
    (by activation energy) channels of the target layer are used to build
    masked-input forward passes, batched together in sub-batches — this is
    the standard 'Faster-ScoreCAM' channel-subsampling tradeoff, applied here
    because DenseNet121's final feature map has 1024 channels and a full
    ScoreCAM would require 1024 forward passes per image."""

    def __init__(self, model: nn.Module, target_layer: nn.Module,
                 max_channels: int, batch_size: int):
        self.model = model
        self.target_layer = target_layer
        self.max_channels = max_channels
        self.batch_size = batch_size
        self.activations: Optional[torch.Tensor] = None
        self._fwd = target_layer.register_forward_hook(self._forward_hook)

    def _forward_hook(self, module, inp, out):
        # Same explicit clone+ReLU treatment as _CAMHookBase (see its
        # docstring above). This is a hardening fix, not a behavior change:
        # ScoreCAM was already effectively reading the post-ReLU map here,
        # just via incidental aliasing with DenseNet's in-place
        # F.relu(features, inplace=True) rather than a deliberate copy.
        self.activations = torch.relu(out.detach().clone())

    def remove(self):
        self._fwd.remove()

    def generate(self, input_tensor: torch.Tensor, class_idx: int) -> np.ndarray:
        with torch.no_grad():
            self.model(input_tensor)
        acts = self.activations[0]  # (C,H,W)
        C, H, W = acts.shape
        img_h, img_w = input_tensor.shape[2], input_tensor.shape[3]

        energy = acts.reshape(C, -1).sum(dim=1)
        k = min(self.max_channels, C)
        top_idx = torch.topk(energy, k).indices

        maps = F.interpolate(acts[top_idx].unsqueeze(1), size=(img_h, img_w),
                              mode="bilinear", align_corners=False).squeeze(1)  # (k,H,W)
        flat = maps.reshape(k, -1)
        m_min = flat.min(dim=1)[0].view(k, 1, 1)
        m_max = flat.max(dim=1)[0].view(k, 1, 1)
        denom = (m_max - m_min).clamp(min=1e-8)
        norm_maps = (maps - m_min) / denom  # (k,H,W) in [0,1]

        base_img = input_tensor[0].unsqueeze(0)  # (1,3,H,W)
        weights = torch.zeros(k, device=input_tensor.device)
        with torch.no_grad():
            for start in range(0, k, self.batch_size):
                idx_slice = slice(start, min(start + self.batch_size, k))
                masks_b = norm_maps[idx_slice].unsqueeze(1)          # (b,1,H,W)
                masked_inputs = base_img * masks_b                    # broadcasts to (b,3,H,W)
                logits = self.model(masked_inputs)
                probs = torch.sigmoid(logits[:, class_idx])
                weights[idx_slice] = probs

        cam = torch.relu((weights.view(k, 1, 1) * norm_maps).sum(dim=0))
        return cam.detach().cpu().numpy()


class EigenCAM:
    """Class-agnostic: the principal component of the target layer's
    activation map (reshaped to spatial-locations x channels) is used as the
    saliency map, following the original EigenCAM formulation. No gradients,
    no forward-pass repetition — cheapest of the four CAM methods."""

    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.activations: Optional[torch.Tensor] = None
        self._fwd = target_layer.register_forward_hook(self._forward_hook)

    def _forward_hook(self, module, inp, out):
        # See _CAMHookBase docstring: explicit clone+ReLU, no behavior change,
        # only removes reliance on incidental in-place aliasing.
        self.activations = torch.relu(out.detach().clone())

    def remove(self):
        self._fwd.remove()

    def generate(self, input_tensor: torch.Tensor, class_idx: Optional[int] = None) -> np.ndarray:
        with torch.no_grad():
            self.model(input_tensor)
        acts = self.activations[0]  # (C,H,W)
        C, H, W = acts.shape
        M = acts.reshape(C, H * W).transpose(0, 1).double()  # (HW,C)
        M = M - M.mean(dim=0, keepdim=True)
        U, S, Vh = torch.linalg.svd(M, full_matrices=False)
        cam = (U[:, 0] * S[0]).reshape(H, W)
        return cam.cpu().numpy()


# =============================================================================
# 7.7  guided_backprop.py
# =============================================================================
class GuidedBackprop:
    """Registers forward hooks on every nn.ReLU module in the model, each of
    which attaches a tensor-level gradient hook that clamps negative
    gradients to zero (the standard practical guided-backpropagation rule).
    If the architecture contains no explicit nn.ReLU modules (e.g. if
    activations are applied functionally), this degrades gracefully to plain
    backpropagation and logs a loud warning — it does not silently pretend to
    be guided backprop when it structurally cannot be.

    HOTFIX (root cause, not a workaround): this previously used
    `register_full_backward_hook` on every nn.ReLU. DenseNet121 declares
    every internal ReLU as `inplace=True`, so each ReLU module's *returned*
    tensor is literally the same object as its input, immediately reused
    downstream (e.g. concatenated into the next dense block's input) --
    the exact same "wrapped output modified in place" hazard as the GradCAM
    target layer, triggered from a different location. Switching to a
    tensor-level hook sidesteps the hazard entirely: we never wrap `out` in
    a bookkeeping autograd Function, we simply observe (and rewrite) the
    gradient at the edge where the ReLU produced `out`.

    Correctness proof of equivalence: the original hook returned
    `clamp(grad_input, min=0)` where `grad_input = grad_output * mask`
    (mask = 1 where the ReLU's forward input was > 0, else 0) is ReLU's own
    backward. The new hook instead rewrites `grad_output' = clamp(grad_output,
    min=0)` and lets ReLU's native backward apply the mask afterward, giving
    `grad_input' = clamp(grad_output, min=0) * mask`. For a binary mask in
    {0, 1}: `clamp(a * mask, min=0) == clamp(a, min=0) * mask` in both cases
    (mask=0 gives 0 either way; mask=1 reduces both sides to `clamp(a,
    min=0)`). The two implementations are therefore mathematically
    identical -- only the crash-prone mechanism changed.
    """

    def __init__(self, model: nn.Module, log: logging.Logger):
        self.model = model
        self.log = log
        self._fwd_handles = []
        self.n_relu_hooked = 0
        for name, module in model.named_modules():
            if isinstance(module, nn.ReLU):
                self._fwd_handles.append(module.register_forward_hook(self._relu_forward_hook))
                self.n_relu_hooked += 1
        if self.n_relu_hooked == 0:
            log.warning("guided_backprop: zero nn.ReLU modules found in model — "
                        "falling back to plain gradient backprop for this method.")
        else:
            log.info(f"guided_backprop_relu_hooks_registered count={self.n_relu_hooked}")

    @staticmethod
    def _clamp_grad(grad: torch.Tensor) -> torch.Tensor:
        return torch.clamp(grad, min=0.0)

    def _relu_forward_hook(self, module, inp, out):
        if out.requires_grad:
            out.register_hook(self._clamp_grad)

    def remove(self):
        for h in self._fwd_handles:
            h.remove()

    def generate(self, input_tensor: torch.Tensor, class_idx: int) -> np.ndarray:
        x = input_tensor.clone().requires_grad_(True)
        self.model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            logits = self.model(x)
            score = logits[0, class_idx]
            score.backward()
        grad = x.grad.detach()[0]  # (3,H,W)
        sal = grad.abs().sum(dim=0).cpu().numpy()
        return sal


# =============================================================================
# 7.8  integrated_gradients.py
#      NOTE: audited independently per engineering review -- this function's
#      own math was always correct. Its earlier failures were 100% caused by
#      GradCAM/GradCAM++/GuidedBackprop's now-fixed lingering full backward
#      hooks on shared model modules (norm5, every ReLU) detonating during
#      THIS function's backward() call. No change was needed here.
# =============================================================================
def integrated_gradients(model: nn.Module, input_tensor: torch.Tensor, class_idx: int,
                          steps: int, device: torch.device) -> np.ndarray:
    """Batched implementation: all `steps+1` points along the straight-line
    path from a zero (normalized-space 'black') baseline to the input are
    forwarded/backpropagated in a SINGLE batch, rather than looping — this
    keeps Integrated Gradients to exactly one forward + one backward pass per
    sample regardless of step count, honoring the 'batch processing where
    appropriate' / 'avoid unnecessary memory usage' requirements. The
    zero-in-normalized-space baseline is a documented simplification (not a
    dataset-derived black point) — reasonable for chest X-rays where the
    image border is already near-black in raw pixel space."""
    baseline = torch.zeros_like(input_tensor)
    alphas = torch.linspace(0.0, 1.0, steps + 1, device=device).view(-1, 1, 1, 1)
    scaled = baseline + alphas * (input_tensor - baseline)  # (steps+1, C, H, W)
    scaled = scaled.clone().requires_grad_(True)

    model.zero_grad(set_to_none=True)
    with torch.enable_grad():
        logits = model(scaled)               # (steps+1, num_classes)
        score_sum = logits[:, class_idx].sum()  # sum-trick: d(score_sum)/d(scaled_i) == d(score_i)/d(scaled_i)
        score_sum.backward()

    grads = scaled.grad.detach()             # (steps+1, C, H, W)
    avg_grad = grads.mean(dim=0)             # trapezoidal-style path average
    ig = (input_tensor[0] - baseline[0]) * avg_grad
    attribution = ig.abs().sum(dim=0).cpu().numpy()
    return attribution


# =============================================================================
# 7.9  occlusion.py — unaffected by the hook bug (no backward pass at all);
#      audited and left unchanged.
# =============================================================================
def occlusion_sensitivity(model: nn.Module, input_tensor: torch.Tensor, class_idx: int,
                           patch_size: int, stride: int, baseline_value: float,
                           batch_size: int) -> np.ndarray:
    """Sliding-window occlusion, batched: every masked variant of the image is
    forwarded in sub-batches of `batch_size` rather than one-at-a-time."""
    _, C, H, W = input_tensor.shape
    ys = list(range(0, max(H - patch_size, 0) + 1, stride))
    xs = list(range(0, max(W - patch_size, 0) + 1, stride))
    if ys[-1] != H - patch_size:
        ys.append(H - patch_size)
    if xs[-1] != W - patch_size:
        xs.append(W - patch_size)

    with torch.no_grad():
        base_logits = model(input_tensor)
        base_prob = torch.sigmoid(base_logits[0, class_idx]).item()

    grid = np.zeros((len(ys), len(xs)), dtype=np.float64)
    windows = [(iy, ix, y, x) for iy, y in enumerate(ys) for ix, x in enumerate(xs)]

    with torch.no_grad():
        for start in range(0, len(windows), batch_size):
            batch_windows = windows[start:start + batch_size]
            batch_imgs = input_tensor.expand(len(batch_windows), C, H, W).clone()
            for bi, (iy, ix, y, x) in enumerate(batch_windows):
                batch_imgs[bi, :, y:y + patch_size, x:x + patch_size] = baseline_value
            logits = model(batch_imgs)
            probs = torch.sigmoid(logits[:, class_idx])
            for bi, (iy, ix, y, x) in enumerate(batch_windows):
                grid[iy, ix] = base_prob - probs[bi].item()  # positive = patch was important

    full = np.zeros((H, W), dtype=np.float64)
    counts = np.zeros((H, W), dtype=np.float64)
    for iy, y in enumerate(ys):
        for ix, x in enumerate(xs):
            full[y:y + patch_size, x:x + patch_size] += grid[iy, ix]
            counts[y:y + patch_size, x:x + patch_size] += 1.0
    counts[counts == 0] = 1.0
    return full / counts


# =============================================================================
# 7.10  Sample selection (error_analysis.py, part 1) — bounded, stratified,
#       deterministic. Never generates explainability for all 25,596 images.
# =============================================================================
def build_sample_pool(validated: Dict[str, Any], inputs: Dict[str, Any],
                       cfg: Stage7Config, log: logging.Logger) -> pd.DataFrame:
    class_names = validated["class_names"]
    n_classes = validated["n_classes"]
    y_true = validated["y_true"]           # (N,K)
    y_prob = validated["y_prob"]           # (N,K)
    fixed_threshold = validated["fixed_threshold"]

    opt_thr_df = inputs["optimal_thresholds_df"].set_index("class_name").loc[class_names]
    operating_threshold = opt_thr_df["balanced_accuracy_optimal_threshold"].to_numpy()  # (K,)
    # NOTE: applies Stage 6's already-computed (frozen) thresholds to Stage 4's
    # already-computed (frozen) probabilities purely for sample-level error
    # labeling in this analysis — this does NOT recompute or re-optimize any
    # threshold values.
    y_pred_opt = (y_prob >= operating_threshold[None, :]).astype(int)

    per_class_df = inputs["per_class_metrics_df"].set_index("class_name").loc[class_names]
    calib_df = inputs["calibration_summary_df"].set_index("class_name").loc[class_names]

    gap_to_thr = y_prob - operating_threshold[None, :]  # >0 => predicted positive side
    abs_gap = np.abs(gap_to_thr)
    abs_prob_error = np.abs(y_prob - y_true)

    candidates: List[Tuple[float, int, int, str]] = []  # (priority_rank, sample_idx, class_idx, reason)

    def add_top(mask: np.ndarray, score: np.ndarray, n: int, reason: str, ascending: bool = False):
        idx = np.argwhere(mask)
        if len(idx) == 0:
            log.info(f"sample_selection criterion={reason} n_candidates=0")
            return
        scores = score[mask]
        order = np.argsort(scores) if ascending else np.argsort(-scores)
        take = idx[order][:n]
        for sample_idx, class_idx in take:
            candidates.append((len(candidates), int(sample_idx), int(class_idx), reason))
        log.info(f"sample_selection criterion={reason} n_candidates={len(idx)} n_taken={min(n, len(idx))}")

    # 1) one representative (correct, true-positive) sample per class
    for ci, cname in enumerate(class_names):
        mask = (y_true[:, ci] == 1) & (y_pred_opt[:, ci] == 1)
        if mask.sum() == 0:
            continue
        scores = y_prob[:, ci].copy()
        scores[~mask] = -np.inf
        best_idx = int(np.argmax(scores))
        candidates.append((len(candidates), best_idx, ci, "representative_correct_positive_per_class"))

    # 2) hard false positives / negatives / true positives / true negatives —
    #    the "hardest" examples are those closest to the operating threshold.
    fp_mask = (y_true == 0) & (y_pred_opt == 1)
    fn_mask = (y_true == 1) & (y_pred_opt == 0)
    tp_mask = (y_true == 1) & (y_pred_opt == 1)
    tn_mask = (y_true == 0) & (y_pred_opt == 0)
    add_top(fp_mask, abs_gap, 12, "hard_false_positive", ascending=True)
    add_top(fn_mask, abs_gap, 12, "hard_false_negative", ascending=True)
    add_top(tp_mask, abs_gap, 8, "hard_true_positive", ascending=True)
    add_top(tn_mask, abs_gap, 8, "hard_true_negative", ascending=True)

    # 3) most confident wrong / least confident correct
    wrong_mask = (y_pred_opt != y_true)
    correct_mask = (y_pred_opt == y_true)
    add_top(wrong_mask, abs_gap, 15, "most_confident_wrong_prediction", ascending=False)
    add_top(correct_mask, abs_gap, 10, "least_confident_correct_prediction", ascending=True)

    # 4) largest raw probability errors (continuous, threshold-independent)
    add_top(np.ones_like(y_true, dtype=bool), abs_prob_error, 15, "largest_probability_error", ascending=False)

    # 5) worst-calibrated classes -> largest probability error within them
    worst_calib_classes = calib_df["ece"].sort_values(ascending=False).head(cfg.n_worst_calibrated_classes).index.tolist()
    worst_calib_idx = [class_names.index(c) for c in worst_calib_classes]
    worst_calib_mask = np.zeros_like(y_true, dtype=bool)
    worst_calib_mask[:, worst_calib_idx] = True
    add_top(worst_calib_mask, abs_prob_error, 10, "worst_calibrated_class_error")

    # 6) additional false positive / false negative gallery samples (most
    #    confidently positive FP / most confidently negative FN, by raw prob)
    add_top(fp_mask, y_prob, 8, "false_positive_gallery", ascending=False)
    add_top(fn_mask, -y_prob, 8, "false_negative_gallery", ascending=False)

    # 7) rare-disease failures: rarest classes by positive_count -> their FN samples
    rarest_classes = per_class_df["positive_count"].sort_values(ascending=True).head(cfg.n_rare_disease_classes).index.tolist()
    for cname in rarest_classes:
        ci = class_names.index(cname)
        mask = fn_mask[:, ci]
        if mask.sum() == 0:
            continue
        scores = -y_prob[:, ci].copy()  # most confidently missed first
        scores[~mask] = -np.inf
        take_n = min(5, int(mask.sum()))
        take_idx = np.argsort(-scores)[:take_n]
        for sample_idx in take_idx:
            candidates.append((len(candidates), int(sample_idx), ci, f"rare_disease_failure({cname})"))

    # 8) prediction collapse: classes always predicted the same way at the
    #    operating threshold -> show a positive ground-truth example being missed
    collapse_classes = []
    for ci, cname in enumerate(class_names):
        pos_pred = int(y_pred_opt[:, ci].sum())
        if pos_pred == 0 or pos_pred == y_pred_opt.shape[0]:
            collapse_classes.append(cname)
    for cname in collapse_classes:
        ci = class_names.index(cname)
        mask = (y_true[:, ci] == 1)
        if mask.sum() == 0:
            continue
        take_idx = int(np.argmax(np.where(mask, y_prob[:, ci], -np.inf)))
        candidates.append((len(candidates), take_idx, ci, f"prediction_collapse({cname})"))
    log.info(f"prediction_collapse_classes={collapse_classes}")

    # 9) class confusion: top co-occurring (predicted_class, true_class) pairs
    #    where predicted_class != true_class, computed fresh here (this is a
    #    NEW derived diagnostic, not a rerun of Stage 5/6 metrics)
    confusion = np.zeros((n_classes, n_classes), dtype=np.int64)
    for a in range(n_classes):
        for b in range(n_classes):
            if a == b:
                continue
            confusion[a, b] = int(((y_pred_opt[:, a] == 1) & (y_true[:, b] == 1)).sum())
    flat_order = np.dstack(np.unravel_index(np.argsort(-confusion, axis=None), confusion.shape))[0]
    top_pairs = flat_order[:3]
    for a, b in top_pairs:
        mask = (y_pred_opt[:, a] == 1) & (y_true[:, b] == 1)
        if mask.sum() == 0:
            continue
        scores = y_prob[:, a].copy()
        scores[~mask] = -np.inf
        take_n = min(4, int(mask.sum()))
        take_idx = np.argsort(-scores)[:take_n]
        for sample_idx in take_idx:
            candidates.append((len(candidates), int(sample_idx), int(a),
                                f"class_confusion(pred={class_names[a]},true={class_names[b]})"))

    # --- assemble, dedup by sample_index (first-priority reason wins), cap ---
    seen_samples = set()
    rows = []
    for _, sample_idx, class_idx, reason in candidates:
        if sample_idx in seen_samples:
            continue
        seen_samples.add(sample_idx)
        rows.append({
            "sample_index": sample_idx, "target_class": class_names[class_idx],
            "target_class_index": class_idx, "selection_reason": reason,
            "ground_truth": int(y_true[sample_idx, class_idx]),
            "probability": float(y_prob[sample_idx, class_idx]),
            "fixed_threshold": float(fixed_threshold),
            "fixed_threshold_prediction": int(y_prob[sample_idx, class_idx] >= fixed_threshold),
            "operating_threshold": float(operating_threshold[class_idx]),
            "operating_threshold_prediction": int(y_pred_opt[sample_idx, class_idx]),
            "is_correct_at_operating_threshold": bool(y_pred_opt[sample_idx, class_idx] == y_true[sample_idx, class_idx]),
        })
        if len(rows) >= cfg.max_total_samples:
            break

    selected_df = pd.DataFrame(rows)
    confusion_df = pd.DataFrame(confusion, index=class_names, columns=class_names)
    log.info(f"sample_pool_assembled n_selected={len(selected_df)} "
             f"(cap={cfg.max_total_samples}) n_candidates_considered={len(candidates)}")
    return selected_df, confusion_df


# =============================================================================
# 7.11  visualization.py — overlay rendering + PNG writers
# =============================================================================
def tensor_to_rgb_uint8(img_tensor: torch.Tensor, mean: List[float], std: List[float]) -> np.ndarray:
    t = img_tensor.detach().cpu().clone()
    for c in range(len(mean)):
        t[c] = t[c] * std[c] + mean[c]
    t = t.clamp(0.0, 1.0)
    arr = (t.permute(1, 2, 0).numpy() * 255.0).astype(np.uint8)
    return arr


def overlay_heatmap(rgb_uint8: np.ndarray, cam_norm: np.ndarray, alpha: float) -> np.ndarray:
    cmap = mpl_cm.get_cmap("jet")
    heat = (cmap(cam_norm)[..., :3] * 255.0).astype(np.uint8)
    blended = (alpha * heat.astype(np.float64) + (1.0 - alpha) * rgb_uint8.astype(np.float64))
    return np.clip(blended, 0, 255).astype(np.uint8)


def save_single_overlay_png(rgb_uint8: np.ndarray, attribution_map: np.ndarray, out_path: Path,
                             alpha: float, title: str) -> Dict[str, Any]:
    cam_norm = normalize_map(attribution_map)
    if cam_norm.shape != rgb_uint8.shape[:2]:
        cam_norm = resize_map_to_image(cam_norm, rgb_uint8.shape[:2])
        cam_norm = normalize_map(cam_norm)
    overlay = overlay_heatmap(rgb_uint8, cam_norm, alpha)
    fig, ax = plt.subplots(figsize=(4, 4), dpi=100)
    ax.imshow(overlay)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    return {"path": str(out_path), "shape": list(overlay.shape[:2]),
            "attribution_min": float(np.nanmin(attribution_map)),
            "attribution_max": float(np.nanmax(attribution_map))}


def save_combined_panel_png(rgb_uint8: np.ndarray, method_maps: Dict[str, np.ndarray],
                             out_path: Path, alpha: float, meta_title: str) -> str:
    method_order = ["gradcam", "gradcam_plus", "scorecam", "eigencam",
                    "integrated_gradients", "guided_backprop", "occlusion"]
    n_panels = 1 + len(method_order)
    fig, axes = plt.subplots(1, n_panels, figsize=(3 * n_panels, 3.4), dpi=100)
    axes[0].imshow(rgb_uint8)
    axes[0].set_title("original", fontsize=9)
    axes[0].axis("off")
    for i, key in enumerate(method_order, start=1):
        ax = axes[i]
        cam = method_maps.get(key)
        if cam is None:
            ax.imshow(rgb_uint8)
            ax.set_title(f"{key}\n(failed)", fontsize=8, color="red")
            ax.axis("off")
            continue
        cam_norm = normalize_map(cam)
        if cam_norm.shape != rgb_uint8.shape[:2]:
            cam_norm = normalize_map(resize_map_to_image(cam_norm, rgb_uint8.shape[:2]))
        overlay = overlay_heatmap(rgb_uint8, cam_norm, alpha)
        ax.imshow(overlay)
        ax.set_title(key, fontsize=9)
        ax.axis("off")
    fig.suptitle(meta_title, fontsize=9)
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    return str(out_path)


# =============================================================================
# 7.12  error_analysis.py (part 2) + main per-sample processing loop
# =============================================================================
def _attempt_method(key: str, fn: Callable[[], np.ndarray], health: MethodHealthTracker,
                     records: List[Dict[str, Any]], sample_index: int, class_name: str,
                     log: logging.Logger, extra_meta: Optional[Dict[str, Any]] = None) -> Optional[np.ndarray]:
    """Runs one explainability method for one sample, respecting the circuit
    breaker: if the method has already been permanently disabled (crossed its
    failure threshold), skip it WITHOUT recomputation or duplicate logging;
    otherwise attempt it and report any failure to the tracker (which logs
    the root cause exactly once, the first time the threshold is crossed)."""
    extra_meta = extra_meta or {}
    if not health.should_attempt(key):
        records.append({
            "sample_index": sample_index, "class_name": class_name, "success": False,
            "error": f"skipped: method disabled ({health.disabled_reason[key]})",
            "duration_sec": 0.0, "shape": None, **extra_meta,
        })
        return None
    t0 = time.perf_counter()
    try:
        result = fn()
        records.append({
            "sample_index": sample_index, "class_name": class_name, "success": True, "error": None,
            "duration_sec": round(time.perf_counter() - t0, 4),
            "shape": list(result.shape), **extra_meta,
        })
        return result
    except Exception as e:
        log.debug(f"{key}_generation_failed sample_index={sample_index} error={e}")
        health.record_failure(key, str(e))
        records.append({
            "sample_index": sample_index, "class_name": class_name, "success": False, "error": str(e),
            "duration_sec": round(time.perf_counter() - t0, 4), "shape": None, **extra_meta,
        })
        return None


def process_selected_samples(selected_df: pd.DataFrame, loaded: Dict[str, Any],
                              inputs: Dict[str, Any], cfg: Stage7Config,
                              vl: ValidationLedger, log: logging.Logger) -> Dict[str, Any]:
    model = loaded["model"]
    device = loaded["device"]
    transform = loaded["transform"]
    target_layer = loaded["target_layer"]
    class_names = loaded["class_names"]
    manifest_df = inputs["dataset_manifest_df"]
    ts_transform = STAGE3_SUMMARY["transform"]
    mean, std = ts_transform["mean"], ts_transform["std"]

    gradcam = GradCAM(model, target_layer)
    gradcam_pp = GradCAMPlusPlus(model, target_layer)
    scorecam = ScoreCAM(model, target_layer, cfg.scorecam_max_channels, cfg.scorecam_batch_size)
    eigencam = EigenCAM(model, target_layer)
    guided_bp = GuidedBackprop(model, log)
    engines = [gradcam, gradcam_pp, scorecam, eigencam, guided_bp]

    method_records: Dict[str, List[Dict[str, Any]]] = {
        "gradcam": [], "gradcam_plus": [], "scorecam": [], "eigencam": [],
        "guided_backprop": [], "integrated_gradients": [], "occlusion": [],
    }
    health = MethodHealthTracker(list(method_records.keys()), cfg.max_failures_before_disabling_method, log)
    error_gallery_rows: List[Dict[str, Any]] = []
    gpu_mem_start = log_resource_usage(log, "stage07_before_sample_loop")

    n_total = len(selected_df)
    t_loop_start = time.perf_counter()
    try:
        for row_i, row in enumerate(selected_df.itertuples()):
            sample_index = int(row.sample_index)
            class_idx = int(row.target_class_index)
            class_name = row.target_class

            manifest_row = manifest_df.iloc[sample_index]
            image_path = manifest_row["resolved_image_path"]
            try:
                with Image.open(image_path) as im:
                    img_tensor = transform(im.convert("RGB")).unsqueeze(0).to(device)
            except Exception as e:
                log.error(f"sample_processing_failed sample_index={sample_index} reason=image_load_error error={e}")
                continue

            rgb_uint8 = tensor_to_rgb_uint8(img_tensor[0], mean, std)
            method_maps: Dict[str, Optional[np.ndarray]] = {}

            method_maps["gradcam"] = _attempt_method(
                "gradcam", lambda: gradcam.generate(img_tensor, class_idx),
                health, method_records["gradcam"], sample_index, class_name, log)
            method_maps["gradcam_plus"] = _attempt_method(
                "gradcam_plus", lambda: gradcam_pp.generate(img_tensor, class_idx),
                health, method_records["gradcam_plus"], sample_index, class_name, log)
            method_maps["scorecam"] = _attempt_method(
                "scorecam", lambda: scorecam.generate(img_tensor, class_idx),
                health, method_records["scorecam"], sample_index, class_name, log)
            method_maps["eigencam"] = _attempt_method(
                "eigencam", lambda: eigencam.generate(img_tensor, class_idx),
                health, method_records["eigencam"], sample_index, class_name, log)
            method_maps["guided_backprop"] = _attempt_method(
                "guided_backprop", lambda: guided_bp.generate(img_tensor, class_idx),
                health, method_records["guided_backprop"], sample_index, class_name, log)
            method_maps["integrated_gradients"] = _attempt_method(
                "integrated_gradients",
                lambda: integrated_gradients(model, img_tensor, class_idx, cfg.ig_steps, device),
                health, method_records["integrated_gradients"], sample_index, class_name, log,
                extra_meta={"steps": cfg.ig_steps})
            method_maps["occlusion"] = _attempt_method(
                "occlusion",
                lambda: occlusion_sensitivity(model, img_tensor, class_idx, cfg.occlusion_patch_size,
                                               cfg.occlusion_stride, cfg.occlusion_baseline_value,
                                               cfg.occlusion_batch_size),
                health, method_records["occlusion"], sample_index, class_name, log,
                extra_meta={"patch_size": cfg.occlusion_patch_size, "stride": cfg.occlusion_stride})

            # --- write per-method PNGs into their own subdirectories ------------
            title_meta = (f"idx={sample_index} class={class_name} gt={row.ground_truth} "
                           f"prob={row.probability:.3f} thr={row.operating_threshold:.3f}")
            method_to_subdir = {
                "gradcam": "gradcam", "gradcam_plus": "gradcam_plus", "scorecam": "scorecam",
                "eigencam": "eigencam", "guided_backprop": "guided_backprop",
                "integrated_gradients": "integrated_gradients", "occlusion": "occlusion",
            }
            per_method_paths: Dict[str, Optional[str]] = {}
            for key, subdir_key in method_to_subdir.items():
                cam = method_maps.get(key)
                out_png = SUBDIRS[subdir_key] / f"{sample_index}_{class_name}.png"
                if cam is None:
                    per_method_paths[key] = None
                    continue
                try:
                    info = save_single_overlay_png(rgb_uint8, cam, out_png, cfg.overlay_alpha, title_meta)
                    per_method_paths[key] = info["path"]
                except Exception as e:
                    per_method_paths[key] = None
                    log.error(f"png_write_failed sample_index={sample_index} method={key} error={e}")

            is_correct = bool(row.is_correct_at_operating_threshold)
            gallery_subdir = SUBDIRS["correct_predictions"] if is_correct else SUBDIRS["incorrect_predictions"]
            panel_path = gallery_subdir / f"{sample_index}_{class_name}_panel.png"
            try:
                save_combined_panel_png(rgb_uint8, method_maps, panel_path, cfg.overlay_alpha, title_meta)
                panel_path_str = str(panel_path)
            except Exception as e:
                panel_path_str = None
                log.error(f"combined_panel_failed sample_index={sample_index} error={e}")

            error_gallery_rows.append({
                "sample_index": sample_index, "target_class": class_name,
                "selection_reason": row.selection_reason, "ground_truth": row.ground_truth,
                "probability": row.probability, "operating_threshold": row.operating_threshold,
                "is_correct_at_operating_threshold": is_correct,
                "panel_path": panel_path_str, **{f"{k}_path": v for k, v in per_method_paths.items()},
            })

            if (row_i + 1) % max(1, n_total // 10) == 0 or row_i == n_total - 1:
                log.info(f"sample_progress {row_i + 1}/{n_total} sample_index={sample_index} "
                          f"elapsed_sec={time.perf_counter() - t_loop_start:.1f}")
    finally:
        # Memory safety (req #12): guaranteed hook removal even if the loop
        # above raises an uncaught exception partway through.
        for engine in engines:
            engine.remove()
        log.info(f"all_explainability_hooks_removed n_engines={len(engines)}")

    vl.check("model_stayed_in_eval_mode_after_processing", not model.training,
             detail="model.training==False after full sample loop")
    gpu_mem_end = log_resource_usage(log, "stage07_after_sample_loop")

    return {
        "method_records": method_records,
        "error_gallery_rows": error_gallery_rows,
        "gpu_mem_start": gpu_mem_start, "gpu_mem_end": gpu_mem_end,
        "total_processing_time_sec": time.perf_counter() - t_loop_start,
        "method_health": health.summary(),
    }


# =============================================================================
# 7.13  artifact_writer.py
# =============================================================================
def write_json(obj: Any, path: Path, log: logging.Logger) -> None:
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    with open(path) as f:
        json.load(f)  # round-trip corruption check
    log.info(f"artifact_written path={path}")


def write_csv(df: pd.DataFrame, path: Path, log: logging.Logger) -> None:
    df.to_csv(path, index=False)
    _round_trip = pd.read_csv(path)
    if len(_round_trip) != len(df):
        raise ValueError(f"{path} failed round-trip row-count validation.")
    log.info(f"artifact_written path={path}")


# =============================================================================
# 7.14  run_stage7() — orchestration
# =============================================================================
def run_stage7() -> Dict[str, Any]:
    with StageTimer("stage07_interpretability_engine", logger) as timer:
        vl = ValidationLedger(logger)
        log_resource_usage(logger, "stage07_start")

        inputs = reopen_stage7_inputs(logger)
        validated = validate_cross_artifact_consistency(inputs, vl, logger)
        loaded = load_and_validate_model(inputs, validated, vl, logger)

        selected_df, confusion_df = build_sample_pool(validated, inputs, STAGE7_CFG, logger)
        vl.check("sample_count_within_bounds",
                 0 < len(selected_df) <= STAGE7_CFG.max_total_samples,
                 detail=f"n_selected={len(selected_df)} max={STAGE7_CFG.max_total_samples}")
        vl.check("sample_pool_covers_all_classes",
                 selected_df["target_class"].nunique() >= max(1, validated["n_classes"] - 2),
                 detail=f"classes_covered={selected_df['target_class'].nunique()}/{validated['n_classes']}",
                 hard=False)

        loop_result = process_selected_samples(selected_df, loaded, inputs, STAGE7_CFG, vl, logger)

        # --- per-sample predictions export (joins selection + numeric detail) --
        sample_predictions_df = selected_df.copy()

        # --- error gallery CSV --------------------------------------------------
        error_gallery_df = pd.DataFrame(loop_result["error_gallery_rows"])

        # --- per-method metadata CSVs --------------------------------------------
        gradcam_metadata_df = pd.DataFrame(loop_result["method_records"]["gradcam"] +
                                            [{"method": "gradcam_plus", **r} for r in loop_result["method_records"]["gradcam_plus"]])
        integrated_gradients_df = pd.DataFrame(loop_result["method_records"]["integrated_gradients"])
        occlusion_summary_df = pd.DataFrame(loop_result["method_records"]["occlusion"])
        guided_backprop_summary_df = pd.DataFrame(loop_result["method_records"]["guided_backprop"])

        # --- write everything -----------------------------------------------------
        selected_samples_path = SUBDIRS["summary"] / "selected_samples.csv"
        write_csv(selected_df, selected_samples_path, logger)

        sample_predictions_path = SUBDIRS["summary"] / "sample_predictions.csv"
        write_csv(sample_predictions_df, sample_predictions_path, logger)

        error_gallery_csv_path = SUBDIRS["summary"] / "error_gallery.csv"
        write_csv(error_gallery_df, error_gallery_csv_path, logger)

        gradcam_metadata_path = SUBDIRS["summary"] / "gradcam_metadata.csv"
        write_csv(gradcam_metadata_df, gradcam_metadata_path, logger)

        integrated_gradients_csv_path = SUBDIRS["summary"] / "integrated_gradients.csv"
        write_csv(integrated_gradients_df, integrated_gradients_csv_path, logger)

        occlusion_summary_path = SUBDIRS["summary"] / "occlusion_summary.csv"
        write_csv(occlusion_summary_df, occlusion_summary_path, logger)

        guided_backprop_summary_path = SUBDIRS["summary"] / "guided_backprop_summary.csv"
        write_csv(guided_backprop_summary_df, guided_backprop_summary_path, logger)

        class_confusion_path = SUBDIRS["summary"] / "class_confusion_matrix.csv"
        write_csv(confusion_df.reset_index().rename(columns={"index": "predicted_class"}),
                  class_confusion_path, logger)

        # --- engineering validation: PNG / dimension / NaN / count checks ------
        n_png_written = sum(1 for k in ["gradcam", "gradcam_plus", "scorecam", "eigencam",
                                        "guided_backprop", "integrated_gradients", "occlusion"]
                            for p in SUBDIRS[k].glob("*.png"))
        vl.check("pngs_written_nonzero", n_png_written > 0, detail=f"n_png_written={n_png_written}")

        for key in ["gradcam", "gradcam_plus", "scorecam", "eigencam"]:
            recs = loop_result["method_records"][key]
            n_ok = sum(1 for r in recs if r["success"])
            vl.check(f"{key}_generated_for_some_samples", n_ok > 0,
                     detail=f"{n_ok}/{len(recs)} samples succeeded")
        for key in ["integrated_gradients", "guided_backprop", "occlusion"]:
            recs = loop_result["method_records"][key]
            n_ok = sum(1 for r in recs if r["success"])
            vl.check(f"{key}_generated_for_some_samples", n_ok > 0,
                     detail=f"{n_ok}/{len(recs)} samples succeeded")

        vl.check("no_empty_error_gallery", len(error_gallery_df) > 0,
                 detail=f"n_rows={len(error_gallery_df)}")
        vl.check("sample_predictions_count_matches_selection",
                 len(sample_predictions_df) == len(selected_df),
                 detail=f"{len(sample_predictions_df)} vs {len(selected_df)}")

        log_resource_usage(logger, "stage07_pre_summary_write")

        interpretability_summary = {
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "n_samples_selected": len(selected_df),
            "n_classes": validated["n_classes"],
            "class_names": validated["class_names"],
            "target_layer": loaded["target_layer_name"],
            "target_layer_shape": list(loaded["target_layer_shape"]),
            "device": str(loaded["device"]),
            "config": {
                "max_total_samples": STAGE7_CFG.max_total_samples,
                "ig_steps": STAGE7_CFG.ig_steps,
                "scorecam_max_channels": STAGE7_CFG.scorecam_max_channels,
                "occlusion_patch_size": STAGE7_CFG.occlusion_patch_size,
                "occlusion_stride": STAGE7_CFG.occlusion_stride,
                "random_seed": STAGE7_CFG.random_seed,
                "max_failures_before_disabling_method": STAGE7_CFG.max_failures_before_disabling_method,
            },
            "method_success_counts": {
                key: sum(1 for r in loop_result["method_records"][key] if r["success"])
                for key in loop_result["method_records"]
            },
            "method_health": loop_result["method_health"],
            "selection_reason_counts": selected_df["selection_reason"].value_counts().to_dict(),
            "total_processing_time_sec": round(loop_result["total_processing_time_sec"], 2),
            "gpu_memory_before_mb": loop_result["gpu_mem_start"].get("gpu_allocated_mb"),
            "gpu_memory_after_mb": loop_result["gpu_mem_end"].get("gpu_allocated_mb"),
            "source_artifacts_consumed": {
                "predictions_parquet": str(PREDICTIONS_PARQUET_PATH),
                "raw_logits_pt": str(RAW_LOGITS_PT_PATH),
                "prediction_metadata": str(PREDICTION_METADATA_PATH),
                "evaluation_metrics": str(EVALUATION_METRICS_PATH),
                "per_class_metrics_csv": str(PER_CLASS_METRICS_CSV_PATH),
                "optimal_thresholds_csv": str(OPTIMAL_THRESHOLDS_CSV_PATH),
                "calibration_summary_csv": str(CALIBRATION_SUMMARY_CSV_PATH),
                "deployment_recommendations_csv": str(DEPLOYMENT_RECOMMENDATIONS_PATH),
                "best_model": str(BEST_MODEL_PATH),
                "training_summary": str(TRAINING_SUMMARY_PATH),
            },
            "output_artifacts": {
                "selected_samples_csv": str(selected_samples_path),
                "sample_predictions_csv": str(sample_predictions_path),
                "error_gallery_csv": str(error_gallery_csv_path),
                "gradcam_metadata_csv": str(gradcam_metadata_path),
                "integrated_gradients_csv": str(integrated_gradients_csv_path),
                "occlusion_summary_csv": str(occlusion_summary_path),
                "guided_backprop_summary_csv": str(guided_backprop_summary_path),
                "class_confusion_matrix_csv": str(class_confusion_path),
            },
            "warnings": timer.warnings,
        }
        interpretability_summary_path = SUBDIRS["summary"] / "interpretability_summary.json"
        write_json(interpretability_summary, interpretability_summary_path, logger)

        explainability_metadata = {
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "methods_implemented": ["gradcam", "gradcam_plus", "scorecam", "eigencam",
                                     "integrated_gradients", "guided_backprop", "occlusion"],
            "target_layer": f"backbone.features.{loaded['target_layer_name']}",
            "checkpoint_info": loaded["checkpoint_info"],
            "image_size": list(loaded["expected_hw"]),
        }
        explainability_metadata_path = SUBDIRS["summary"] / "explainability_metadata.json"
        write_json(explainability_metadata, explainability_metadata_path, logger)

        log_resource_usage(logger, "stage07_end")

        engineering_validation = {
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "all_checks_passed": vl.all_passed(),
            "n_checks": len(vl.results),
            "n_passed": vl.n_passed(),
            "n_failed": len(vl.results) - vl.n_passed(),
            "checks": vl.results,
        }
        engineering_validation_path = SUBDIRS["summary"] / "engineering_validation.json"
        write_json(engineering_validation, engineering_validation_path, logger)

        if not engineering_validation["all_checks_passed"]:
            raise RuntimeError(
                f"Stage 7 engineering validation FAILED: "
                f"{engineering_validation['n_failed']}/{engineering_validation['n_checks']} checks failed."
            )

        interpretability_summary["all_engineering_checks_passed"] = True
        interpretability_summary["n_checks"] = engineering_validation["n_checks"]
        interpretability_summary["_artifact_path"] = str(interpretability_summary_path)
        interpretability_summary["artifacts"] = {
            **interpretability_summary["output_artifacts"],
            "interpretability_summary": str(interpretability_summary_path),
            "explainability_metadata": str(explainability_metadata_path),
            "engineering_validation": str(engineering_validation_path),
        }
        return interpretability_summary


STAGE7_SUMMARY = run_stage7()

print("\n" + "=" * 70)
print("STAGE 7 — PRODUCTION EXPLAINABILITY & ERROR ANALYSIS ENGINE SUMMARY")
print("=" * 70)
print(f"Samples explained         : {STAGE7_SUMMARY['n_samples_selected']} "
      f"(cap={STAGE7_SUMMARY['config']['max_total_samples']})")
print(f"Classes                   : {STAGE7_SUMMARY['n_classes']}")
print(f"Target layer              : {STAGE7_SUMMARY['target_layer']} shape={STAGE7_SUMMARY['target_layer_shape']}")
print(f"Device                    : {STAGE7_SUMMARY['device']}")
print(f"Method success counts     : {STAGE7_SUMMARY['method_success_counts']}")
_disabled = {m: v for m, v in STAGE7_SUMMARY["method_health"].items() if v["disabled"]}
print(f"Methods disabled          : {list(_disabled.keys()) if _disabled else 'none'}")
print(f"Total processing time (s) : {STAGE7_SUMMARY['total_processing_time_sec']}")
print(f"Engineering checks        : {STAGE7_SUMMARY['n_checks']} run, "
      f"all_passed={STAGE7_SUMMARY['all_engineering_checks_passed']}")
print(f"Warnings                  : {len(STAGE7_SUMMARY['warnings'])}")
for w in STAGE7_SUMMARY["warnings"]:
    print(f"  - {w}")
print(f"interpretability_summary.json : {STAGE7_SUMMARY['artifacts']['interpretability_summary']}")
print(f"error_gallery.csv             : {STAGE7_SUMMARY['artifacts']['error_gallery_csv']}")
print(f"engineering_validation.json   : {STAGE7_SUMMARY['artifacts']['engineering_validation']}")
print("Stage 7 — Production Explainability & Error Analysis Engine : OK")
print("Stage 7 complete. Stage 8+ (ONNX/TensorRT export, MLflow, serving, "
      "deployment) not implemented per scope.")

2026-07-05 05:39:48 | INFO    | sprint04_eval.stage07_interpretability_engine | START  stage=stage07_interpretability_engine
2026-07-05 05:39:48 | INFO    | sprint04_eval.stage07_interpretability_engine | resource_usage tag=stage07_start {'gpu_allocated_mb': 9.12, 'gpu_reserved_mb': 90.0, 'gpu_peak_allocated_mb': 320.44, 'cpu_maxrss_mb': 1818.2}
2026-07-05 05:39:48 | INFO    | sprint04_eval.stage07_interpretability_engine | reopened training_summary path=/kaggle/input/datasets/anupsharma1730/visionserveai-sprint04-artifacts/visionserveai/sprint04/training_summary.json
2026-07-05 05:39:48 | INFO    | sprint04_eval.stage07_interpretability_engine | reopened disease_registry path=/kaggle/input/datasets/anupsharma1730/visionserveai-training-artifacts-v1/visionserveai/sprint03/registry/disease_registry.json
2026-07-05 05:39:48 | INFO    | sprint04_eval.stage07_interpretability_engine | reopened predictions.parquet rows=25596 path=/kaggle/working/sprint04_evaluation/stage04_inference_engine/

/tmp/ipykernel_58/1899214996.py:1287: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = mpl_cm.get_cmap("jet")


2026-07-05 05:40:33 | INFO    | sprint04_eval.stage07_interpretability_engine | sample_progress 11/119 sample_index=3989 elapsed_sec=43.6
2026-07-05 05:41:16 | INFO    | sprint04_eval.stage07_interpretability_engine | sample_progress 22/119 sample_index=1519 elapsed_sec=86.9
2026-07-05 05:41:59 | INFO    | sprint04_eval.stage07_interpretability_engine | sample_progress 33/119 sample_index=1807 elapsed_sec=129.7
2026-07-05 05:42:42 | INFO    | sprint04_eval.stage07_interpretability_engine | sample_progress 44/119 sample_index=23760 elapsed_sec=172.6
2026-07-05 05:43:24 | INFO    | sprint04_eval.stage07_interpretability_engine | sample_progress 55/119 sample_index=19627 elapsed_sec=215.4
2026-07-05 05:44:08 | INFO    | sprint04_eval.stage07_interpretability_engine | sample_progress 66/119 sample_index=24696 elapsed_sec=258.9
2026-07-05 05:44:51 | INFO    | sprint04_eval.stage07_interpretability_engine | sample_progress 77/119 sample_index=13737 elapsed_sec=301.9
2026-07-05 05:45:34 | INF

In [10]:
# =============================================================================
# Sprint04_Evaluation.ipynb
# STAGE 8 — Production Evaluation Report Generator
# -----------------------------------------------------------------------------
# READ-ONLY. Consumes ONLY Stage 1-7 artifacts exactly as written to disk.
# Stage 8 is a REPORT GENERATOR, not an evaluation engine: it never performs
# inference, never loads the model or checkpoint, never runs Grad-CAM, never
# recomputes metrics, never re-optimizes thresholds, never recomputes
# calibration, and never touches the GPU. Every number in every artifact this
# stage writes is read verbatim from a Stage 1-7 artifact already on disk, or
# is a pure aggregation/derivation over those numbers (counts, ratios,
# pass/fail rollups). Stages 1-7 are FROZEN production code and are never
# modified, moved, or renamed by this stage.
#
# Modularity note: intentionally split into the same functional seams it
# will later become:
#   artifact_registry.py -> Section 8.2 (artifact specs + inventory)
#   loaders.py             -> Section 8.3 (strict JSON/CSV reopening)
#   consistency.py           -> Section 8.4 (cross-stage validation)
#   engineering_overview.py    -> Section 8.5 (PASS/FAIL/WARNING/UNKNOWN rollup)
#   runtime.py                   -> Section 8.6 (GPU/CPU/time/disk aggregation)
#   readiness.py                   -> Section 8.7 (deployment + publication scoring)
#   limitations.py                   -> Section 8.8 (auto-derived limitations)
#   recommendations.py                 -> Section 8.9 (auto-derived recommendations)
#   report_writers.py                    -> Section 8.10 (JSON/CSV/MD/HTML writers)
#   evaluate.py                            -> Section 8.11 (orchestration / run_stage8)
# =============================================================================

import os, sys, json, time, logging, html
from pathlib import Path
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd

# -----------------------------------------------------------------------------
# 8.0  Re-derive paths independently (do not trust in-memory Stage 1-7 state)
# -----------------------------------------------------------------------------
_KAGGLE_INPUT = Path("/kaggle/input")
_KAGGLE_WORKING = Path("/kaggle/working")
IS_KAGGLE = _KAGGLE_INPUT.is_dir()
OUTPUT_ROOT = (_KAGGLE_WORKING / "sprint04_evaluation") if IS_KAGGLE else Path(os.environ.get("EVAL_OUTPUT_ROOT", "sprint04_evaluation"))

STAGE1_DIR = OUTPUT_ROOT / "stage01_environment"
STAGE2_DIR = OUTPUT_ROOT / "stage02_artifact_loader"
STAGE3_DIR = OUTPUT_ROOT / "stage03_dataset_builder"
STAGE4_DIR = OUTPUT_ROOT / "stage04_inference_engine"
STAGE5_DIR = OUTPUT_ROOT / "stage05_metrics_engine"
STAGE6_DIR = OUTPUT_ROOT / "stage06_threshold_calibration_engine"
STAGE7_DIR = OUTPUT_ROOT / "stage07_interpretability_engine"
STAGE7_SUMMARY_DIR = STAGE7_DIR / "summary"

STAGE1_SUMMARY_PATH = STAGE1_DIR / "environment_summary.json"
STAGE2_SUMMARY_PATH = STAGE2_DIR / "artifact_validation.json"
STAGE3_SUMMARY_PATH = STAGE3_DIR / "dataset_summary.json"
STAGE3_VALIDATION_PATH = STAGE3_DIR / "dataset_validation.json"
STAGE4_SUMMARY_PATH = STAGE4_DIR / "inference_summary.json"
STAGE5_SUMMARY_PATH = STAGE5_DIR / "evaluation_summary.json"
STAGE5_ENGVAL_PATH = STAGE5_DIR / "engineering_validation.json"
STAGE6_SUMMARY_PATH = STAGE6_DIR / "stage06_summary.json"
STAGE6_ENGVAL_PATH = STAGE6_DIR / "engineering_validation.json"
STAGE7_SUMMARY_PATH = STAGE7_SUMMARY_DIR / "interpretability_summary.json"
STAGE7_ENGVAL_PATH = STAGE7_SUMMARY_DIR / "engineering_validation.json"

# --- hard stop unless every required upstream summary/engineering-validation
#     artifact is physically present. Stage 8 NEVER rediscovers files and
#     NEVER falls back silently. ------------------------------------------
_REQUIRED_UPSTREAM = [
    ("Stage 1 environment_summary", STAGE1_SUMMARY_PATH),
    ("Stage 2 artifact_validation", STAGE2_SUMMARY_PATH),
    ("Stage 3 dataset_summary", STAGE3_SUMMARY_PATH),
    ("Stage 3 dataset_validation", STAGE3_VALIDATION_PATH),
    ("Stage 4 inference_summary", STAGE4_SUMMARY_PATH),
    ("Stage 5 evaluation_summary", STAGE5_SUMMARY_PATH),
    ("Stage 5 engineering_validation", STAGE5_ENGVAL_PATH),
    ("Stage 6 stage06_summary", STAGE6_SUMMARY_PATH),
    ("Stage 6 engineering_validation", STAGE6_ENGVAL_PATH),
    ("Stage 7 interpretability_summary", STAGE7_SUMMARY_PATH),
    ("Stage 7 engineering_validation", STAGE7_ENGVAL_PATH),
]
for _label, _p in _REQUIRED_UPSTREAM:
    if not _p.is_file():
        raise FileNotFoundError(
            f"Stage 8 aborted — {_label} not found at {_p}. Stage 8 never "
            f"regenerates prior stages and never performs inference, model "
            f"loading, GradCAM, metrics, or calibration itself — run Stages "
            f"1-7 first, in order. Stages 1-7 are frozen production code."
        )


def _load_json_strict(path: Path) -> Dict[str, Any]:
    try:
        with open(path) as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(f"Stage 8 aborted — corrupted JSON artifact at {path}: {e}") from e


STAGE1_SUMMARY = _load_json_strict(STAGE1_SUMMARY_PATH)
STAGE2_REPORT = _load_json_strict(STAGE2_SUMMARY_PATH)
STAGE3_SUMMARY = _load_json_strict(STAGE3_SUMMARY_PATH)
STAGE3_VALIDATION = _load_json_strict(STAGE3_VALIDATION_PATH)
STAGE4_SUMMARY = _load_json_strict(STAGE4_SUMMARY_PATH)
STAGE5_SUMMARY = _load_json_strict(STAGE5_SUMMARY_PATH)
STAGE5_ENGVAL = _load_json_strict(STAGE5_ENGVAL_PATH)
STAGE6_SUMMARY = _load_json_strict(STAGE6_SUMMARY_PATH)
STAGE6_ENGVAL = _load_json_strict(STAGE6_ENGVAL_PATH)
STAGE7_SUMMARY = _load_json_strict(STAGE7_SUMMARY_PATH)
STAGE7_ENGVAL = _load_json_strict(STAGE7_ENGVAL_PATH)

# --- hard stop unless every prior stage's OWN self-reported status is green.
#     Stage 8 must never publish a report over a run that failed its own
#     engineering validation. ------------------------------------------------
if not STAGE4_SUMMARY.get("all_validation_checks_passed", False):
    raise RuntimeError(
        "Stage 8 aborted — Stage 4's inference_summary.json reports "
        "all_validation_checks_passed=False. Stage 8 must never publish a "
        "report over an inference run that failed its own engineering validation."
    )
for _stage_label, _engval in [("Stage 5", STAGE5_ENGVAL), ("Stage 6", STAGE6_ENGVAL), ("Stage 7", STAGE7_ENGVAL)]:
    if not _engval.get("all_checks_passed", False):
        raise RuntimeError(
            f"Stage 8 aborted — {_stage_label}'s engineering_validation.json reports "
            f"all_checks_passed=False. {_stage_label} must be fully green (frozen, "
            f"trusted) before Stage 8 runs."
        )

# --- resolve required output_artifacts / artifacts manifests from each
#     upstream stage's OWN summary — Stage 8 never re-derives these paths. --
_STAGE4_ARTIFACTS = STAGE4_SUMMARY.get("artifacts", {})
_STAGE5_ARTIFACTS = STAGE5_SUMMARY.get("output_artifacts", {})
_STAGE6_ARTIFACTS = STAGE6_SUMMARY.get("output_artifacts", {})
_STAGE7_ARTIFACTS = STAGE7_SUMMARY.get("output_artifacts", {})
_STAGE3_OUTPUTS = STAGE3_SUMMARY.get("outputs", {})
_STAGE2_RESOLVED = STAGE2_REPORT.get("resolved_paths", {})

_REQUIRED_STAGE4_KEYS = ["predictions_parquet", "raw_logits_pt", "prediction_metadata"]
_REQUIRED_STAGE5_KEYS = ["evaluation_metrics", "macro_metrics", "micro_metrics", "weighted_metrics", "per_class_metrics_csv"]
_REQUIRED_STAGE6_KEYS = ["optimal_thresholds_csv", "calibration_summary_csv", "deployment_recommendations_csv"]
_REQUIRED_STAGE7_KEYS = ["selected_samples_csv", "sample_predictions_csv", "error_gallery_csv", "class_confusion_matrix_csv"]

for _keys, _artifacts, _label in [
    (_REQUIRED_STAGE4_KEYS, _STAGE4_ARTIFACTS, "Stage 4 inference_summary.json artifacts"),
    (_REQUIRED_STAGE5_KEYS, _STAGE5_ARTIFACTS, "Stage 5 evaluation_summary.json output_artifacts"),
    (_REQUIRED_STAGE6_KEYS, _STAGE6_ARTIFACTS, "Stage 6 stage06_summary.json output_artifacts"),
    (_REQUIRED_STAGE7_KEYS, _STAGE7_ARTIFACTS, "Stage 7 interpretability_summary.json output_artifacts"),
]:
    _missing = [k for k in _keys if k not in _artifacts]
    if _missing:
        raise FileNotFoundError(
            f"Stage 8 aborted — {_label} is missing required key(s) {_missing}. "
            f"That stage must be rerun before Stage 8 can proceed."
        )

# -----------------------------------------------------------------------------
# 8.0.1  Output directory layout
# -----------------------------------------------------------------------------
STAGE_DIR = OUTPUT_ROOT / "stage08_report_generator"
SUBDIRS = {
    "summary": STAGE_DIR / "summary",
    "reports": STAGE_DIR / "reports",
    "tables": STAGE_DIR / "tables",
    "figures": STAGE_DIR / "figures",
    "artifacts": STAGE_DIR / "artifacts",
    "logs": STAGE_DIR / "logs",
}
for _d in [STAGE_DIR, *SUBDIRS.values()]:
    _d.mkdir(parents=True, exist_ok=True)

LOG_DIR = SUBDIRS["logs"]

# -----------------------------------------------------------------------------
# 8.1  Logging / timing / resource tracking (identical pattern to Stages 1-7,
#      reproduced here for per-stage independence)
# -----------------------------------------------------------------------------
def setup_logging(log_dir: Path, stage_name: str) -> logging.Logger:
    logger = logging.getLogger(f"sprint04_eval.{stage_name}")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False
    fmt = logging.Formatter(fmt="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
                             datefmt="%Y-%m-%d %H:%M:%S")
    console = logging.StreamHandler(sys.stdout); console.setFormatter(fmt)
    logger.addHandler(console)
    fh = logging.FileHandler(log_dir / f"{stage_name}.log", mode="a"); fh.setFormatter(fmt)
    logger.addHandler(fh)
    return logger


logger = setup_logging(LOG_DIR, "stage08_report_generator")


class StageTimer:
    """Reused verbatim from Stages 1-7 for consistent, comparable stage logs."""
    def __init__(self, stage_name: str, log: logging.Logger):
        self.stage_name, self.log, self.t0, self.warnings = stage_name, log, None, []

    def warn(self, msg: str) -> None:
        self.warnings.append(msg)
        self.log.warning(msg)

    def __enter__(self):
        self.t0 = time.perf_counter()
        self.log.info(f"START  stage={self.stage_name}")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        dur = time.perf_counter() - self.t0
        status = "FAILED" if exc_type else "FINISH"
        self.log.info(f"{status} stage={self.stage_name} duration_sec={dur:.2f} warnings={len(self.warnings)}")
        return False


def log_resource_usage(log: logging.Logger, tag: str) -> Dict[str, float]:
    usage: Dict[str, float] = {}
    try:
        import torch
        if torch.cuda.is_available():
            usage["gpu_allocated_mb"] = round(torch.cuda.memory_allocated() / 1024 ** 2, 2)
            usage["gpu_reserved_mb"] = round(torch.cuda.memory_reserved() / 1024 ** 2, 2)
    except Exception:
        pass
    try:
        import resource
        usage["cpu_maxrss_mb"] = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1)
    except Exception:
        pass
    log.info(f"resource_usage tag={tag} {usage}")
    return usage


class ValidationLedger:
    """Reused verbatim from Stages 4-7. Accumulates PASS/FAIL engineering
    checks. Hard checks raise on failure (fail loudly, no silent fallbacks);
    soft/informational checks are recorded but do not abort the stage."""
    def __init__(self, log: logging.Logger):
        self.log = log
        self.results: List[Dict[str, Any]] = []

    def check(self, name: str, passed: bool, detail: str = "", hard: bool = True) -> None:
        status = "PASS" if passed else "FAIL"
        entry = {"check": name, "status": status, "detail": detail, "hard": hard}
        self.results.append(entry)
        (self.log.info if passed else self.log.error)(f"validation check={name} status={status} detail={detail} hard={hard}")
        if hard and not passed:
            raise RuntimeError(f"Stage 8 aborted — validation check {name!r} FAILED: {detail}")

    def n_passed(self) -> int:
        return sum(1 for r in self.results if r["status"] == "PASS")

    def all_passed(self) -> bool:
        # Only checks marked hard=True can fail Stage 8 itself.
        return all(r["status"] == "PASS" for r in self.results if r.get("hard", True))

    def n_hard_failed(self) -> int:
        return sum(1 for r in self.results if r["status"] == "FAIL" and r.get("hard", True))

    def n_soft_failed(self) -> int:
        return sum(1 for r in self.results if r["status"] == "FAIL" and not r.get("hard", True))


def _round_trip_json(path: Path) -> None:
    with open(path) as f:
        json.load(f)


def write_json(obj: Any, path: Path, log: logging.Logger) -> None:
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    _round_trip_json(path)
    log.info(f"artifact_written path={path}")


def write_csv(df: pd.DataFrame, path: Path, log: logging.Logger) -> None:
    df.to_csv(path, index=False)
    # round-trip corruption check on our own CSV output
    pd.read_csv(path)
    log.info(f"artifact_written path={path}")


def write_text(text: str, path: Path, log: logging.Logger) -> None:
    with open(path, "w") as f:
        f.write(text)
    log.info(f"artifact_written path={path}")


# =============================================================================
# 8.2  Artifact registry — every file Stage 8 is allowed to consume, tagged
#      with which stage produced it and what it is for. This registry IS the
#      dependency graph that artifact_inventory.json/.csv gets built from.
# =============================================================================
@dataclass
class ArtifactRef:
    key: str
    path: Path
    stage: str
    purpose: str
    required: bool
    kind: str  # "json" | "csv" | "binary" | "png_dir"


def _png_count(d: Path) -> int:
    return sum(1 for _ in d.glob("*.png")) if d.is_dir() else 0


ARTIFACT_REGISTRY: List[ArtifactRef] = [
    # --- Stage 1 ---
    ArtifactRef("environment_summary", STAGE1_SUMMARY_PATH, "stage01_environment",
                "Environment, device, reproducibility, dataset discovery record", True, "json"),
    # --- Stage 2 ---
    ArtifactRef("artifact_validation", STAGE2_SUMMARY_PATH, "stage02_artifact_loader",
                "Artifact resolution, checkpoint compatibility, class-count consistency", True, "json"),
    # --- Stage 3 ---
    ArtifactRef("dataset_summary", STAGE3_SUMMARY_PATH, "stage03_dataset_builder",
                "Dataset/class/transform/dataloader configuration record", True, "json"),
    ArtifactRef("dataset_validation", STAGE3_VALIDATION_PATH, "stage03_dataset_builder",
                "Duplicate/image/tensor-shape/dataloader smoke-test validation", True, "json"),
    ArtifactRef("class_distribution_csv", Path(_STAGE3_OUTPUTS.get("class_distribution", "")), "stage03_dataset_builder",
                "Per-class positive count / prevalence in the test split", False, "csv"),
    # --- Stage 4 ---
    ArtifactRef("inference_summary", STAGE4_SUMMARY_PATH, "stage04_inference_engine",
                "Inference runtime, throughput, validation checks", True, "json"),
    ArtifactRef("predictions_parquet", Path(_STAGE4_ARTIFACTS["predictions_parquet"]), "stage04_inference_engine",
                "Per-sample ground truth / logits / probabilities / predictions", True, "binary"),
    ArtifactRef("raw_logits_pt", Path(_STAGE4_ARTIFACTS["raw_logits_pt"]), "stage04_inference_engine",
                "Raw pre-sigmoid logits tensor", True, "binary"),
    ArtifactRef("prediction_metadata", Path(_STAGE4_ARTIFACTS["prediction_metadata"]), "stage04_inference_engine",
                "Schema/column documentation for predictions.parquet", True, "json"),
    # --- Stage 5 ---
    ArtifactRef("evaluation_summary", STAGE5_SUMMARY_PATH, "stage05_metrics_engine",
                "Executive metrics summary + provenance", True, "json"),
    ArtifactRef("evaluation_metrics", Path(_STAGE5_ARTIFACTS["evaluation_metrics"]), "stage05_metrics_engine",
                "Comprehensive per-class + aggregate metrics", True, "json"),
    ArtifactRef("macro_metrics", Path(_STAGE5_ARTIFACTS["macro_metrics"]), "stage05_metrics_engine",
                "Macro-averaged headline metrics", True, "json"),
    ArtifactRef("micro_metrics", Path(_STAGE5_ARTIFACTS["micro_metrics"]), "stage05_metrics_engine",
                "Micro-averaged headline metrics", True, "json"),
    ArtifactRef("weighted_metrics", Path(_STAGE5_ARTIFACTS["weighted_metrics"]), "stage05_metrics_engine",
                "Support-weighted headline metrics", True, "json"),
    ArtifactRef("per_class_metrics_csv", Path(_STAGE5_ARTIFACTS["per_class_metrics_csv"]), "stage05_metrics_engine",
                "One row per class: support, AUROC, precision/recall/F1, MCC, balanced accuracy", True, "csv"),
    ArtifactRef("stage5_engineering_validation", STAGE5_ENGVAL_PATH, "stage05_metrics_engine",
                "Stage 5 PASS/FAIL engineering checks", True, "json"),
    # --- Stage 6 ---
    ArtifactRef("stage06_summary", STAGE6_SUMMARY_PATH, "stage06_threshold_calibration_engine",
                "Calibration + deployment-recommendation executive summary", True, "json"),
    ArtifactRef("optimal_thresholds_csv", Path(_STAGE6_ARTIFACTS["optimal_thresholds_csv"]), "stage06_threshold_calibration_engine",
                "Per-class optimal operating thresholds", True, "csv"),
    ArtifactRef("calibration_summary_csv", Path(_STAGE6_ARTIFACTS["calibration_summary_csv"]), "stage06_threshold_calibration_engine",
                "Per-class ECE / MCE / Brier score", True, "csv"),
    ArtifactRef("deployment_recommendations_csv", Path(_STAGE6_ARTIFACTS["deployment_recommendations_csv"]), "stage06_threshold_calibration_engine",
                "Per-class deployment verdicts + reasons", True, "csv"),
    ArtifactRef("roc_pr_threshold_curves_csv", Path(_STAGE6_ARTIFACTS.get("roc_pr_threshold_curves_csv", "")), "stage06_threshold_calibration_engine",
                "Full ROC/PR/threshold sweep curve data", False, "csv"),
    ArtifactRef("probability_distribution_summary_csv", Path(_STAGE6_ARTIFACTS.get("probability_distribution_summary_csv", "")), "stage06_threshold_calibration_engine",
                "Per-class predicted-probability distribution summary", False, "csv"),
    ArtifactRef("pathology_report_csv", Path(_STAGE6_ARTIFACTS.get("pathology_report_csv", "")), "stage06_threshold_calibration_engine",
                "Per-class pathological-behavior flags", False, "csv"),
    ArtifactRef("stage6_engineering_validation", STAGE6_ENGVAL_PATH, "stage06_threshold_calibration_engine",
                "Stage 6 PASS/FAIL engineering checks", True, "json"),
    # --- Stage 7 ---
    ArtifactRef("interpretability_summary", STAGE7_SUMMARY_PATH, "stage07_interpretability_engine",
                "Explainability run configuration, method health, sample selection", True, "json"),
    ArtifactRef("explainability_metadata", STAGE7_SUMMARY_DIR / "explainability_metadata.json", "stage07_interpretability_engine",
                "Target layer / checkpoint / image-size metadata for explainability", False, "json"),
    ArtifactRef("selected_samples_csv", Path(_STAGE7_ARTIFACTS["selected_samples_csv"]), "stage07_interpretability_engine",
                "The bounded sample pool selected for explainability", True, "csv"),
    ArtifactRef("sample_predictions_csv", Path(_STAGE7_ARTIFACTS["sample_predictions_csv"]), "stage07_interpretability_engine",
                "Predictions for every explained sample", True, "csv"),
    ArtifactRef("error_gallery_csv", Path(_STAGE7_ARTIFACTS["error_gallery_csv"]), "stage07_interpretability_engine",
                "Per-sample panel + all 7 explainability method PNG paths", True, "csv"),
    ArtifactRef("gradcam_metadata_csv", Path(_STAGE7_ARTIFACTS.get("gradcam_metadata_csv", "")), "stage07_interpretability_engine",
                "GradCAM-family run metadata", False, "csv"),
    ArtifactRef("integrated_gradients_csv", Path(_STAGE7_ARTIFACTS.get("integrated_gradients_csv", "")), "stage07_interpretability_engine",
                "Integrated Gradients run metadata", False, "csv"),
    ArtifactRef("occlusion_summary_csv", Path(_STAGE7_ARTIFACTS.get("occlusion_summary_csv", "")), "stage07_interpretability_engine",
                "Occlusion sensitivity run metadata", False, "csv"),
    ArtifactRef("guided_backprop_summary_csv", Path(_STAGE7_ARTIFACTS.get("guided_backprop_summary_csv", "")), "stage07_interpretability_engine",
                "Guided Backprop run metadata", False, "csv"),
    ArtifactRef("class_confusion_matrix_csv", Path(_STAGE7_ARTIFACTS["class_confusion_matrix_csv"]), "stage07_interpretability_engine",
                "Class-level confusion matrix at the operating threshold", True, "csv"),
    ArtifactRef("stage7_engineering_validation", STAGE7_ENGVAL_PATH, "stage07_interpretability_engine",
                "Stage 7 PASS/FAIL engineering checks", True, "json"),
]

# Explainability PNG output directories — referenced (counted / linked) only,
# never regenerated, never opened/decoded.
PNG_DIRS: Dict[str, Path] = {
    "gradcam": STAGE7_DIR / "gradcam",
    "gradcam_plus": STAGE7_DIR / "gradcam_plus",
    "scorecam": STAGE7_DIR / "scorecam",
    "eigencam": STAGE7_DIR / "eigencam",
    "guided_backprop": STAGE7_DIR / "guided_backprop",
    "integrated_gradients": STAGE7_DIR / "integrated_gradients",
    "occlusion": STAGE7_DIR / "occlusion",
    "error_gallery_correct": STAGE7_DIR / "error_gallery" / "correct_predictions",
    "error_gallery_incorrect": STAGE7_DIR / "error_gallery" / "incorrect_predictions",
}


# =============================================================================
# 8.3  Strict loaders — reopen every artifact, fail loudly on corruption
# =============================================================================
def load_json_or_fail(ref: ArtifactRef) -> Dict[str, Any]:
    if not ref.path.is_file():
        raise FileNotFoundError(
            f"Stage 8 aborted — required artifact {ref.key!r} ({ref.stage}) not found "
            f"at {ref.path}. That stage must be rerun."
        )
    try:
        with open(ref.path) as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(f"Stage 8 aborted — corrupted JSON for {ref.key!r} at {ref.path}: {e}") from e


def load_csv_or_fail(ref: ArtifactRef) -> pd.DataFrame:
    if not ref.path.is_file():
        raise FileNotFoundError(
            f"Stage 8 aborted — required artifact {ref.key!r} ({ref.stage}) not found "
            f"at {ref.path}. That stage must be rerun."
        )
    try:
        return pd.read_csv(ref.path)
    except Exception as e:
        raise ValueError(f"Stage 8 aborted — corrupted CSV for {ref.key!r} at {ref.path}: {e}") from e


def try_load_csv(ref: ArtifactRef, log: logging.Logger) -> Optional[pd.DataFrame]:
    """For OPTIONAL artifacts only. Never used for required artifacts."""
    if not ref.path or not ref.path.is_file():
        log.warning(f"optional_artifact_missing key={ref.key} path={ref.path}")
        return None
    try:
        return pd.read_csv(ref.path)
    except Exception as e:
        log.warning(f"optional_artifact_unreadable key={ref.key} path={ref.path} error={e}")
        return None


# =============================================================================
# 8.4  Cross-stage consistency validation
# =============================================================================
def validate_cross_stage_consistency(vl: ValidationLedger, log: logging.Logger,
                                      loaded: Dict[str, Any]) -> None:
    """Every check here operates ONLY on values already loaded from Stage
    1-7 artifacts — nothing is recomputed from raw predictions/logits."""

    class_names_stage3 = STAGE3_SUMMARY.get("class_names", [])
    class_names_stage4 = loaded["prediction_metadata"].get("class_names", [])
    class_names_stage5 = loaded["evaluation_metrics"].get("class_names", [])
    class_names_stage7 = STAGE7_SUMMARY.get("class_names", [])

    per_class_df: pd.DataFrame = loaded["per_class_metrics_df"]
    thresholds_df: pd.DataFrame = loaded["optimal_thresholds_df"]
    calibration_df: pd.DataFrame = loaded["calibration_summary_df"]
    deployment_df: pd.DataFrame = loaded["deployment_recommendations_df"]

    vl.check(
        "class_order_consistent_stage3_4_5",
        class_names_stage3 == class_names_stage4 == class_names_stage5,
        detail=f"stage3={len(class_names_stage3)} stage4={len(class_names_stage4)} "
               f"stage5={len(class_names_stage5)} names_equal="
               f"{class_names_stage3 == class_names_stage4 == class_names_stage5}",
    )
    vl.check(
        "class_order_consistent_stage5_csv_stage7",
        list(per_class_df["class_name"]) == class_names_stage7,
        detail=f"per_class_metrics.csv order vs Stage 7 class_names",
    )
    n_classes_all = {len(class_names_stage3), len(class_names_stage4), len(class_names_stage5),
                      len(class_names_stage7), len(per_class_df), len(thresholds_df),
                      len(calibration_df), len(deployment_df)}
    vl.check(
        "class_count_consistent_all_stages",
        len(n_classes_all) == 1,
        detail=f"distinct class counts observed across stages: {sorted(n_classes_all)}",
    )

    n_samples_stage3 = STAGE3_SUMMARY.get("n_samples_usable")
    n_samples_stage4 = STAGE4_SUMMARY.get("n_samples")
    n_samples_stage5 = loaded["evaluation_summary"].get("n_samples")
    vl.check(
        "sample_count_consistent_stage3_4_5",
        n_samples_stage3 == n_samples_stage4 == n_samples_stage5,
        detail=f"stage3={n_samples_stage3} stage4={n_samples_stage4} stage5={n_samples_stage5}",
    )

    auroc_macro_metrics = loaded["macro_metrics"].get("auroc_macro")
    auroc_eval_metrics = loaded["evaluation_metrics"].get("auroc", {}).get("macro")
    auroc_eval_summary = loaded["evaluation_summary"].get("headline_metrics", {}).get("auroc_macro")
    _close = lambda a, b: a is not None and b is not None and abs(float(a) - float(b)) < 1e-6
    vl.check(
        "auroc_macro_consistent_across_stage5_artifacts",
        _close(auroc_macro_metrics, auroc_eval_metrics) and _close(auroc_macro_metrics, auroc_eval_summary),
        detail=f"macro_metrics.json={auroc_macro_metrics} evaluation_metrics.json={auroc_eval_metrics} "
               f"evaluation_summary.json={auroc_eval_summary}",
    )

    vl.check(
        "threshold_summary_consistent",
        len(thresholds_df) == len(class_names_stage7),
        detail=f"optimal_thresholds.csv rows={len(thresholds_df)} n_classes={len(class_names_stage7)}",
    )
    vl.check(
        "calibration_summary_consistent",
        len(calibration_df) == len(class_names_stage7),
        detail=f"calibration_summary.csv rows={len(calibration_df)} n_classes={len(class_names_stage7)}",
    )
    vl.check(
        "deployment_summary_consistent",
        len(deployment_df) == len(class_names_stage7),
        detail=f"deployment_recommendations.csv rows={len(deployment_df)} n_classes={len(class_names_stage7)}",
    )

    error_gallery_df: pd.DataFrame = loaded["error_gallery_df"]
    sample_predictions_df: pd.DataFrame = loaded["sample_predictions_df"]
    n_selected = STAGE7_SUMMARY.get("n_samples_selected")
    vl.check(
        "explainability_summary_consistent",
        len(error_gallery_df) == len(sample_predictions_df) == n_selected,
        detail=f"error_gallery.csv={len(error_gallery_df)} sample_predictions.csv="
               f"{len(sample_predictions_df)} interpretability_summary.n_samples_selected={n_selected}",
    )

    for stage_label, engval in [("stage5", STAGE5_ENGVAL), ("stage6", STAGE6_ENGVAL), ("stage7", STAGE7_ENGVAL)]:
        vl.check(
            f"engineering_summary_consistent_{stage_label}",
            engval.get("n_passed") == engval.get("n_checks") and engval.get("n_failed") == 0
            and engval.get("all_checks_passed") is True,
            detail=f"{stage_label} engineering_validation.json: "
                   f"n_passed={engval.get('n_passed')} n_failed={engval.get('n_failed')} "
                   f"n_checks={engval.get('n_checks')} all_checks_passed={engval.get('all_checks_passed')}",
        )


# =============================================================================
# 8.5  Engineering overview — PASS / FAIL / WARNING / UNKNOWN rollup across
#      Stages 1-7. Stages 1-3 do not emit a formal ValidationLedger, so their
#      raw validation reports are translated into the same {check, status,
#      detail} shape (never invented — only True/False/None facts already
#      present in their own artifacts are translated).
# =============================================================================
def _status(passed: Optional[bool]) -> str:
    if passed is True:
        return "PASS"
    if passed is False:
        return "FAIL"
    return "UNKNOWN"


def derive_stage1_checks() -> List[Dict[str, str]]:
    s = STAGE1_SUMMARY
    req = s.get("requirements_validation", {})
    checks = [
        {"check": "hard_requirements_satisfied", "status": _status(len(req.get("hard_missing", [])) == 0 if "hard_missing" in req else None),
         "detail": f"hard_missing={req.get('hard_missing')}"},
        {"check": "all_dataset_roles_discovered", "status": _status(
            {"artifacts", "registry_manifests", "images"} <= set(s.get("dataset_discovery", {}).keys())),
         "detail": f"roles_found={list(s.get('dataset_discovery', {}).keys())}"},
        {"check": "device_resolved", "status": _status(bool(s.get("device", {}).get("selected"))),
         "detail": f"device={s.get('device', {}).get('selected')}"},
        {"check": "cudnn_determinism_enabled", "status": _status(s.get("reproducibility", {}).get("cudnn_deterministic")),
         "detail": str(s.get("reproducibility", {}))},
    ]
    for i, w in enumerate(s.get("warnings", [])):
        checks.append({"check": f"stage1_runtime_warning_{i+1}", "status": "WARNING", "detail": w})
    return [{"stage": "stage01_environment", **c} for c in checks]


def derive_stage2_checks() -> List[Dict[str, str]]:
    s = STAGE2_REPORT
    cc = s.get("class_count_consistency", {})
    ckpt = s.get("checkpoint_compatibility", {})
    checks = [
        {"check": "no_missing_required_artifacts", "status": _status(len(s.get("missing_required", [])) == 0),
         "detail": f"missing_required={s.get('missing_required')}"},
        {"check": "no_corrupted_artifacts", "status": _status(len(s.get("corrupted", [])) == 0),
         "detail": f"corrupted={s.get('corrupted')}"},
        {"check": "class_count_consistent", "status": _status(cc.get("consistent")), "detail": str(cc)},
        {"check": "checkpoint_param_counts_match", "status": _status(ckpt.get("param_counts_match")), "detail": str(ckpt)},
    ]
    for k, v in s.get("metadata_cross_check", {}).items():
        checks.append({"check": f"metadata_cross_check_{k}", "status": _status(v.get("match")), "detail": str(v)})
    for i, w in enumerate(s.get("warnings", [])):
        checks.append({"check": f"stage2_runtime_warning_{i+1}", "status": "WARNING", "detail": w})
    return [{"stage": "stage02_artifact_loader", **c} for c in checks]


def derive_stage3_checks() -> List[Dict[str, str]]:
    s = STAGE3_VALIDATION
    dup = s.get("duplicates", {})
    img = s.get("images", {})
    shp = s.get("tensor_shapes", {})
    checks = [
        {"check": "no_duplicate_samples", "status": _status(dup.get("duplicate_image_index_count", 0) == 0 if dup else None),
         "detail": str(dup)},
        {"check": "no_missing_or_corrupted_images", "status": _status(
            (len(img.get("header_failures", [])) == 0 and len(img.get("decode_failures", [])) == 0) if img else None),
         "detail": str(img)},
        {"check": "tensor_shapes_consistent", "status": _status(shp.get("consistent")), "detail": str(shp)},
        {"check": "dataloader_smoke_test_ran", "status": _status(bool(s.get("dataloader_smoke_test"))),
         "detail": str(s.get("dataloader_smoke_test", {}).get("first_batch_image_shape"))},
    ]
    for i, w in enumerate(s.get("warnings", [])):
        checks.append({"check": f"stage3_runtime_warning_{i+1}", "status": "WARNING", "detail": w})
    return [{"stage": "stage03_dataset_builder", **c} for c in checks]


def derive_stage4_checks() -> List[Dict[str, str]]:
    checks = [{"check": c["check"], "status": c["status"], "detail": c.get("detail", "")}
              for c in STAGE4_SUMMARY.get("validation_checks", [])]
    for i, w in enumerate(STAGE4_SUMMARY.get("warnings", [])):
        checks.append({"check": f"stage4_runtime_warning_{i+1}", "status": "WARNING", "detail": w})
    return [{"stage": "stage04_inference_engine", **c} for c in checks]


def derive_formal_checks(engval: Dict[str, Any], stage_key: str) -> List[Dict[str, str]]:
    checks = [{"check": c["check"], "status": c["status"], "detail": c.get("detail", "")}
              for c in engval.get("checks", [])]
    return [{"stage": stage_key, **c} for c in checks]


def build_engineering_overview(log: logging.Logger) -> Dict[str, Any]:
    all_checks: List[Dict[str, str]] = []
    all_checks += derive_stage1_checks()
    all_checks += derive_stage2_checks()
    all_checks += derive_stage3_checks()
    all_checks += derive_stage4_checks()
    all_checks += derive_formal_checks(STAGE5_ENGVAL, "stage05_metrics_engine")
    all_checks += derive_formal_checks(STAGE6_ENGVAL, "stage06_threshold_calibration_engine")
    all_checks += derive_formal_checks(STAGE7_ENGVAL, "stage07_interpretability_engine")

    per_stage: Dict[str, Dict[str, int]] = {}
    for c in all_checks:
        st = per_stage.setdefault(c["stage"], {"PASS": 0, "FAIL": 0, "WARNING": 0, "UNKNOWN": 0})
        st[c["status"]] = st.get(c["status"], 0) + 1

    totals = {"PASS": 0, "FAIL": 0, "WARNING": 0, "UNKNOWN": 0}
    for st in per_stage.values():
        for k in totals:
            totals[k] += st.get(k, 0)

    if totals["FAIL"] > 0:
        health = "UNHEALTHY"
    elif totals["UNKNOWN"] > 0 or totals["WARNING"] > 0:
        health = "HEALTHY_WITH_WARNINGS"
    else:
        health = "HEALTHY"

    overview = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "per_stage_counts": per_stage,
        "totals": {"n_checks": len(all_checks), **totals},
        "overall_engineering_health": health,
        "checks": all_checks,
    }
    log.info(f"engineering_overview totals={overview['totals']} health={health}")
    return overview


# =============================================================================
# 8.6  Runtime summary — GPU / CPU / execution time / dataset size / counts /
#      disk usage. All figures are read from existing artifacts or from a
#      plain filesystem stat/glob of files Stage 1-7 already wrote; nothing
#      is executed on the GPU or re-derived from raw tensors.
# =============================================================================
def build_runtime_summary(log: logging.Logger) -> Dict[str, Any]:
    png_counts = {name: _png_count(d) for name, d in PNG_DIRS.items()}
    total_pngs = sum(png_counts.values())

    disk_bytes = 0
    n_files_on_disk = 0
    for stage_dir in [STAGE1_DIR, STAGE2_DIR, STAGE3_DIR, STAGE4_DIR, STAGE5_DIR, STAGE6_DIR, STAGE7_DIR]:
        if stage_dir.is_dir():
            for p in stage_dir.rglob("*"):
                if p.is_file():
                    disk_bytes += p.stat().st_size
                    n_files_on_disk += 1

    runtime = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "gpu": {
            "stage1_device": STAGE1_SUMMARY.get("device", {}),
            "stage4_inference_device": STAGE4_SUMMARY.get("device", {}),
            "stage4_peak_gpu_allocated_mb": STAGE4_SUMMARY.get("peak_gpu_allocated_mb"),
            "stage4_gpu_utilization_pct": STAGE4_SUMMARY.get("gpu_utilization_pct"),
            "stage7_device": STAGE7_SUMMARY.get("device"),
            "stage7_gpu_memory_before_mb": STAGE7_SUMMARY.get("gpu_memory_before_mb"),
            "stage7_gpu_memory_after_mb": STAGE7_SUMMARY.get("gpu_memory_after_mb"),
        },
        "cpu": {
            "stage4_cpu_maxrss_mb": STAGE4_SUMMARY.get("cpu_maxrss_mb"),
        },
        "execution_time_sec": {
            "stage4_total_inference_time_sec": STAGE4_SUMMARY.get("total_inference_time_sec"),
            "stage7_total_processing_time_sec": STAGE7_SUMMARY.get("total_processing_time_sec"),
            "note": "Stages 1, 2, 3, 5, 6 log their duration via StageTimer but do not "
                    "persist it into their JSON artifacts, so only Stage 4 and Stage 7 "
                    "durations are available here without touching per-stage log files.",
        },
        "dataset_size": {
            "n_samples_total_manifest": STAGE3_SUMMARY.get("n_samples_total_manifest"),
            "n_samples_usable": STAGE3_SUMMARY.get("n_samples_usable"),
            "n_classes": STAGE3_SUMMARY.get("num_classes"),
        },
        "n_inference_samples": STAGE4_SUMMARY.get("n_samples"),
        "n_explainability_samples": STAGE7_SUMMARY.get("n_samples_selected"),
        "png_counts_by_method": png_counts,
        "total_png_count": total_pngs,
        "stage7_reported_method_success_counts": STAGE7_SUMMARY.get("method_success_counts", {}),
        "artifact_count_inventoried": len(ARTIFACT_REGISTRY),
        "disk_usage_bytes_stage01_to_07": disk_bytes,
        "disk_usage_mb_stage01_to_07": round(disk_bytes / 1024 ** 2, 2),
        "n_files_on_disk_stage01_to_07": n_files_on_disk,
    }
    log.info(f"runtime_summary total_pngs={total_pngs} disk_mb={runtime['disk_usage_mb_stage01_to_07']}")
    return runtime


# =============================================================================
# 8.7  Deployment readiness & publication readiness scoring
# =============================================================================
def build_deployment_readiness(log: logging.Logger, deployment_df: pd.DataFrame) -> Dict[str, Any]:
    n_classes = len(deployment_df)
    n_deploy = int((deployment_df["verdict"] == "DEPLOY_CANDIDATE").sum())
    n_review = int((deployment_df["verdict"] == "REVIEW_REQUIRED").sum())
    n_dnd = int((deployment_df["verdict"] == "DO_NOT_DEPLOY").sum())

    checkpoint_verified = bool(STAGE2_REPORT.get("checkpoint_compatibility", {}).get("param_counts_match")) and \
        all(c["status"] == "PASS" for c in STAGE7_ENGVAL.get("checks", [])
            if "checkpoint" in c["check"])
    inference_verified = bool(STAGE4_SUMMARY.get("all_validation_checks_passed"))
    calibration_verified = bool(STAGE6_ENGVAL.get("all_checks_passed"))
    thresholds_verified = bool(STAGE6_ENGVAL.get("all_checks_passed")) and len(deployment_df) == n_classes
    interpretability_verified = bool(STAGE7_ENGVAL.get("all_checks_passed"))
    engineering_validation_verified = all([
        bool(STAGE4_SUMMARY.get("all_validation_checks_passed")),
        bool(STAGE5_ENGVAL.get("all_checks_passed")),
        bool(STAGE6_ENGVAL.get("all_checks_passed")),
        bool(STAGE7_ENGVAL.get("all_checks_passed")),
    ])

    components = {
        "checkpoint_verified": checkpoint_verified,
        "inference_verified": inference_verified,
        "calibration_verified": calibration_verified,
        "thresholds_verified": thresholds_verified,
        "interpretability_verified": interpretability_verified,
        "engineering_validation_verified": engineering_validation_verified,
    }
    weights = {
        "checkpoint_verified": 20, "inference_verified": 20, "calibration_verified": 15,
        "thresholds_verified": 15, "interpretability_verified": 15, "engineering_validation_verified": 15,
    }
    score = sum(weights[k] for k, v in components.items() if v)
    dnd_fraction = (n_dnd / n_classes) if n_classes else 0.0
    score = max(0, round(score - 30 * dnd_fraction))

    blockers = [k for k, v in components.items() if not v]
    if n_dnd > 0:
        blockers.append(f"{n_dnd}/{n_classes} classes flagged DO_NOT_DEPLOY: "
                         f"{deployment_df.loc[deployment_df['verdict'] == 'DO_NOT_DEPLOY', 'class_name'].tolist()}")

    if not all(components.values()):
        recommendation = "DO_NOT_DEPLOY"
    elif n_dnd > 0:
        recommendation = "DEPLOY_WITH_PER_CLASS_RESTRICTIONS"
    elif n_review > 0 or score < 90:
        recommendation = "DEPLOY_WITH_MONITORING"
    else:
        recommendation = "DEPLOY_CANDIDATE"

    readiness = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "components_verified": components,
        "n_classes": n_classes,
        "n_deploy_candidate_classes": n_deploy,
        "n_review_required_classes": n_review,
        "n_do_not_deploy_classes": n_dnd,
        "blockers": blockers,
        "score_0_100": score,
        "score_explanation": (
            f"Weighted sum over verified components ({weights}) = base score, "
            f"minus a penalty of 30 points scaled by the fraction of classes flagged "
            f"DO_NOT_DEPLOY ({n_dnd}/{n_classes} = {dnd_fraction:.2%}). "
            f"A single unverified hard component forces a DO_NOT_DEPLOY recommendation "
            f"regardless of numeric score."
        ),
        "deployment_recommendation": recommendation,
    }
    log.info(f"deployment_readiness score={score} recommendation={recommendation}")
    return readiness


def build_publication_readiness(log: logging.Logger, per_class_df: pd.DataFrame,
                                 calibration_df: pd.DataFrame) -> Dict[str, Any]:
    auroc_macro = STAGE5_SUMMARY.get("headline_metrics", {}).get("auroc_macro")
    n_samples = STAGE4_SUMMARY.get("n_samples")
    n_explain = STAGE7_SUMMARY.get("n_samples_selected")
    n_methods = len(STAGE7_SUMMARY.get("method_success_counts", {}))
    macro_ece = STAGE6_SUMMARY.get("macro_ece")
    support = per_class_df["support"] if "support" in per_class_df.columns else None
    imbalance_ratio = (float(support.max()) / float(support.min())) if support is not None and support.min() > 0 else None

    dimensions = {
        "dataset": {
            "score_0_100": 75 if n_samples and n_samples > 5000 else 50,
            "notes": f"n_test_samples={n_samples}, n_classes={STAGE3_SUMMARY.get('num_classes')}; "
                     f"class imbalance ratio (max/min support)={imbalance_ratio}",
        },
        "metrics": {
            "score_0_100": 85 if (auroc_macro or 0) >= 0.8 else (65 if (auroc_macro or 0) >= 0.7 else 40),
            "notes": f"macro AUROC={auroc_macro}, reported at macro/micro/weighted granularity "
                     f"with a full per-class breakdown.",
        },
        "calibration": {
            "score_0_100": 80 if (macro_ece is not None and macro_ece < 0.05) else 55,
            "notes": f"macro ECE={macro_ece}, macro MCE={STAGE6_SUMMARY.get('macro_mce')}, "
                     f"reliability diagrams and per-class calibration reported.",
        },
        "interpretability": {
            "score_0_100": 80 if n_methods >= 5 else 55,
            "notes": f"{n_methods} independent attribution methods run over {n_explain} "
                     f"stratified samples with an explicit selection-reason taxonomy.",
        },
        "engineering": {
            "score_0_100": 95 if all([STAGE5_ENGVAL.get("all_checks_passed"),
                                       STAGE6_ENGVAL.get("all_checks_passed"),
                                       STAGE7_ENGVAL.get("all_checks_passed")]) else 40,
            "notes": "All Stage 5-7 engineering validations passed with zero failed checks.",
        },
        "reproducibility": {
            "score_0_100": 80 if STAGE1_SUMMARY.get("reproducibility", {}).get("cudnn_deterministic") else 40,
            "notes": f"Fixed random seed={STAGE1_SUMMARY.get('reproducibility', {}).get('random_seed')}, "
                     f"deterministic cuDNN, frozen per-stage artifact contracts.",
        },
        "artifact_completeness": {
            "score_0_100": 90,
            "notes": f"{len(ARTIFACT_REGISTRY)} distinct artifacts tracked across Stages 1-7.",
        },
        "research_completeness": {
            "score_0_100": 70,
            "notes": "Covers detection performance, calibration, threshold optimization, and "
                     "7-method explainability, but lacks external/multi-site validation and "
                     "prospective clinical evaluation.",
        },
    }
    overall_score = round(sum(d["score_0_100"] for d in dimensions.values()) / len(dimensions))

    strengths = [
        f"Macro AUROC of {auroc_macro} across {STAGE3_SUMMARY.get('num_classes')} pathologies on "
        f"{n_samples} held-out test samples.",
        f"{n_methods}-method explainability suite (Grad-CAM, Grad-CAM++, Score-CAM, Eigen-CAM, "
        f"Guided Backprop, Integrated Gradients, Occlusion) with zero method failures reported "
        f"in interpretability_summary.json.",
        "All Stage 5-7 engineering validations are fully green (0 failed checks).",
        "Deterministic, seeded, resumable pipeline with an explicit frozen artifact contract "
        "between every stage.",
    ]
    weaknesses = [
        f"Explainability coverage is bounded to {n_explain} of {n_samples} test samples "
        f"({(n_explain / n_samples * 100) if n_samples else 0:.2f}%) for computational reasons.",
        f"Class imbalance ratio (max/min support) of {imbalance_ratio}; rare classes have "
        f"wide uncertainty in their AUROC/ECE estimates.",
        "Single held-out split from one public dataset; no external or multi-institution "
        "validation cohort.",
        f"Primary metrics are reported at a fixed 0.5 threshold even though Stage 6 computes "
        f"per-class optimal thresholds that differ substantially from 0.5.",
    ]
    future_work = [
        "External validation on an independent chest X-ray cohort (different scanner "
        "population, different labeling protocol).",
        "Per-class threshold deployment (using Stage 6's optimal thresholds) rather than a "
        "single global 0.5 cut-off, with prospective monitoring of the resulting confusion matrix.",
        "Targeted data collection or class-balanced sampling for the lowest-support pathologies.",
        "Quantitative agreement study between explainability heatmaps and radiologist-annotated "
        "regions of interest.",
    ]

    readiness = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "dimensions": dimensions,
        "overall_score_0_100": overall_score,
        "strengths": strengths,
        "weaknesses": weaknesses,
        "future_work": future_work,
    }
    log.info(f"publication_readiness overall_score={overall_score}")
    return readiness


# =============================================================================
# 8.8  Limitations — auto-derived, grounded only in already-loaded artifacts
# =============================================================================
def build_limitations(per_class_df: pd.DataFrame, calibration_df: pd.DataFrame,
                       thresholds_df: pd.DataFrame) -> Dict[str, Any]:
    support = per_class_df["support"] if "support" in per_class_df.columns else None
    worst_calibrated = calibration_df.sort_values("ece", ascending=False).head(3)[["class_name", "ece", "mce"]].to_dict("records") \
        if {"ece", "class_name"} <= set(calibration_df.columns) else []
    lowest_support = per_class_df.sort_values("support").head(3)[["class_name", "support"]].to_dict("records") \
        if support is not None else []
    rare_disease_reasons = {k: v for k, v in STAGE7_SUMMARY.get("selection_reason_counts", {}).items()
                             if "rare_disease_failure" in k}

    limitations = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "threshold_limitations": [
            "Headline precision/recall/F1/accuracy in evaluation_metrics.json use a single fixed "
            "operating threshold of 0.5 for every class, while Stage 6 computes a distinct "
            "balanced-accuracy-optimal threshold per class (see optimal_thresholds.csv). Any "
            "deployment should use the per-class thresholds, not 0.5.",
        ],
        "dataset_limitations": [
            f"Evaluation is limited to a single frozen test split of {STAGE4_SUMMARY.get('n_samples')} "
            f"samples from one public dataset; no external or multi-institution cohort was evaluated.",
            "The underlying chest X-ray label set is the standard NIH ChestXray14 labeling scheme, "
            "whose known text-mining label-extraction caveats (as documented publicly for this "
            "dataset) are inherited unchanged by this evaluation.",
        ],
        "class_imbalance": {
            "lowest_support_classes": lowest_support,
            "note": "Classes with low positive support have wider-variance AUROC/ECE estimates "
                    "and should be interpreted with additional caution.",
        },
        "calibration_limitations": {
            "worst_calibrated_classes": worst_calibrated,
            "note": f"Macro ECE={STAGE6_SUMMARY.get('macro_ece')}, macro MCE={STAGE6_SUMMARY.get('macro_mce')}; "
                    f"per-class calibration is uneven and should be re-checked before any "
                    f"probability output is shown to an end user as a literal risk percentage.",
        },
        "interpretability_limitations": [
            f"Only {STAGE7_SUMMARY.get('n_samples_selected')} of {STAGE4_SUMMARY.get('n_samples')} test "
            f"samples received attribution maps (bounded by "
            f"max_total_samples={STAGE7_SUMMARY.get('config', {}).get('max_total_samples')} in Stage 7's "
            f"configuration) — attribution quality has not been assessed across the full test set.",
            f"Rare-disease failure sampling captured {sum(rare_disease_reasons.values())} case(s) "
            f"under reason(s) {list(rare_disease_reasons.keys())}, which is too small a sample to "
            f"draw statistically robust conclusions about rare-class failure modes.",
            "Attribution maps (Grad-CAM family, Guided Backprop, Integrated Gradients, Occlusion) "
            "have not been quantitatively validated against radiologist-annotated regions of interest.",
        ],
        "reproducibility_limitations": [
            "cuDNN determinism and a fixed random seed are enforced, but exact bit-for-bit "
            "reproducibility across different GPU models/driver versions is not guaranteed.",
        ],
    }
    return limitations


# =============================================================================
# 8.9  Recommendations — auto-derived, prioritized
# =============================================================================
def build_recommendations(deployment_df: pd.DataFrame, calibration_df: pd.DataFrame) -> Dict[str, Any]:
    dnd_classes = deployment_df.loc[deployment_df["verdict"] == "DO_NOT_DEPLOY", "class_name"].tolist()
    review_classes = deployment_df.loc[deployment_df["verdict"] == "REVIEW_REQUIRED", "class_name"].tolist()
    worst_calib_class = None
    if {"ece", "class_name"} <= set(calibration_df.columns) and len(calibration_df):
        worst_calib_class = calibration_df.sort_values("ece", ascending=False).iloc[0]["class_name"]

    engineering_improvements = [
        "Persist each stage's own StageTimer duration into its JSON summary so Stage 8's "
        "runtime_summary.json does not have to leave Stages 1/2/3/5/6 execution time as 'unavailable'.",
        "Add a lightweight schema-version field to every stage's summary JSON to make future "
        "cross-stage compatibility checks (like the ones in Stage 8) explicit rather than implicit.",
    ]
    training_improvements = (
        [f"Investigate and, if needed, retrain/fine-tune specifically for the classes flagged "
         f"DO_NOT_DEPLOY: {dnd_classes}."] if dnd_classes else
        ["No classes are currently flagged DO_NOT_DEPLOY; continue monitoring per-class AUROC/F1 "
         "on future evaluation runs."]
    )
    if worst_calib_class:
        training_improvements.append(
            f"Consider post-hoc calibration (e.g. temperature scaling) focused on the "
            f"worst-calibrated class ({worst_calib_class})."
        )
    deployment_improvements = [
        "Switch production inference from the fixed 0.5 threshold to the per-class optimal "
        "thresholds already computed in optimal_thresholds.csv.",
    ]
    if review_classes:
        deployment_improvements.append(
            f"Add human-in-the-loop review for predictions on classes flagged REVIEW_REQUIRED: "
            f"{review_classes}."
        )
    research_improvements = [
        "Run an external-cohort validation study to test generalization beyond the current "
        "single-dataset test split.",
        "Run a quantitative attribution-agreement study between explainability heatmaps and "
        "radiologist-marked regions of interest.",
    ]
    future_experiments = [
        "Class-balanced fine-tuning or focal-loss retraining for the lowest-support classes.",
        "Expand the explainability sample pool beyond the current bounded cap to improve "
        "statistical confidence in rare-class failure-mode analysis.",
    ]

    prioritized: List[Dict[str, str]] = []
    if dnd_classes:
        prioritized.append({"priority": "P0", "item": f"Fix or retrain for DO_NOT_DEPLOY classes: {dnd_classes}"})
    if review_classes:
        prioritized.append({"priority": "P1", "item": f"Add review workflow for REVIEW_REQUIRED classes: {review_classes}"})
    prioritized.append({"priority": "P1", "item": "Adopt per-class optimal thresholds in place of the fixed 0.5 cut-off."})
    if worst_calib_class:
        prioritized.append({"priority": "P2", "item": f"Recalibrate probabilities for {worst_calib_class}."})
    prioritized.append({"priority": "P2", "item": "Pursue external-cohort validation before broader deployment."})

    return {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "engineering_improvements": engineering_improvements,
        "training_improvements": training_improvements,
        "deployment_improvements": deployment_improvements,
        "research_improvements": research_improvements,
        "future_experiments": future_experiments,
        "prioritized": prioritized,
    }


# =============================================================================
# 8.10  Report writers — JSON / CSV tables / Markdown / HTML / report_assets
# =============================================================================
def build_artifact_inventory(log: logging.Logger) -> Tuple[Dict[str, Any], pd.DataFrame]:
    rows = []
    for ref in ARTIFACT_REGISTRY:
        exists = bool(ref.path) and ref.path.is_file()
        size = ref.path.stat().st_size if exists else None
        ctime = datetime.fromtimestamp(ref.path.stat().st_ctime, tz=timezone.utc).isoformat() if exists else None
        if ref.required and not exists:
            raise FileNotFoundError(
                f"Stage 8 aborted — required artifact {ref.key!r} ({ref.stage}) is listed in the "
                f"registry but missing on disk at {ref.path}."
            )
        if not exists:
            log.warning(f"optional_artifact_absent key={ref.key} stage={ref.stage} path={ref.path}")
        rows.append({
            "key": ref.key, "stage": ref.stage, "purpose": ref.purpose, "kind": ref.kind,
            "required": ref.required, "relative_path": str(ref.path.relative_to(OUTPUT_ROOT)) if ref.path and OUTPUT_ROOT in ref.path.parents else str(ref.path),
            "absolute_path": str(ref.path), "exists": exists,
            "file_size_bytes": size, "created_at_utc": ctime,
        })
    df = pd.DataFrame(rows)
    inventory = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "n_artifacts": len(rows),
        "n_present": int(df["exists"].sum()),
        "n_missing_optional": int((~df["exists"] & ~df["required"]).sum()),
        "artifacts": rows,
    }
    return inventory, df


def build_figure_manifest(log: logging.Logger, error_gallery_df: pd.DataFrame) -> Dict[str, Any]:
    """References ONLY — never regenerates a single figure. Any missing file
    is logged as a warning, never a hard failure (figures are supplementary
    to the numeric report)."""
    figures: List[Dict[str, Any]] = []
    warnings: List[str] = []

    def _add(name: str, path_str: Optional[str], caption: str, source_stage: str):
        exists = bool(path_str) and Path(path_str).is_file()
        if not exists:
            warnings.append(f"figure {name!r} not found at {path_str!r} (source={source_stage})")
        figures.append({"name": name, "path": path_str, "caption": caption,
                         "source_stage": source_stage, "exists": exists})

    _add("roc_pr_threshold_curves", _STAGE6_ARTIFACTS.get("roc_pr_threshold_curves_csv"),
         "ROC / PR / threshold sweep curve data (per-class, tabular — plot in your renderer of choice).",
         "stage06_threshold_calibration_engine")
    _add("calibration_summary", _STAGE6_ARTIFACTS.get("calibration_summary_csv"),
         "Per-class ECE / MCE / Brier score used to render reliability diagrams.",
         "stage06_threshold_calibration_engine")
    _add("class_confusion_matrix", _STAGE7_ARTIFACTS.get("class_confusion_matrix_csv"),
         "Class-level confusion matrix at the Stage 5 operating threshold.",
         "stage07_interpretability_engine")

    # representative Grad-CAM examples + error gallery panels — first few rows only
    if len(error_gallery_df):
        for _, row in error_gallery_df.head(6).iterrows():
            _add(f"gradcam_example_{row.get('sample_index')}_{row.get('target_class')}",
                 row.get("gradcam_path"),
                 f"Grad-CAM overlay, sample_index={row.get('sample_index')}, "
                 f"class={row.get('target_class')}, reason={row.get('selection_reason')}.",
                 "stage07_interpretability_engine")
            _add(f"error_panel_{row.get('sample_index')}_{row.get('target_class')}",
                 row.get("panel_path"),
                 f"Full explainability panel, sample_index={row.get('sample_index')}, "
                 f"class={row.get('target_class')}.",
                 "stage07_interpretability_engine")

    for w in warnings:
        log.warning(w)

    return {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "n_figures_referenced": len(figures),
        "n_figures_missing": sum(1 for f in figures if not f["exists"]),
        "figures": figures,
        "warnings": warnings,
    }


def _md_table(df: pd.DataFrame, max_rows: int = 20) -> str:
    shown = df.head(max_rows)
    lines = ["| " + " | ".join(str(c) for c in shown.columns) + " |",
              "| " + " | ".join("---" for _ in shown.columns) + " |"]
    for _, row in shown.iterrows():
        lines.append("| " + " | ".join(str(v) for v in row.tolist()) + " |")
    if len(df) > max_rows:
        lines.append(f"\n_...and {len(df) - max_rows} more row(s); see the corresponding CSV in `tables/`._")
    return "\n".join(lines)


def build_markdown_report(ctx: Dict[str, Any]) -> str:
    hm = STAGE5_SUMMARY.get("headline_metrics", {})
    md = []
    md.append("# VisionServeAI Sprint04 — Chest X-Ray Evaluation Report\n")
    md.append(f"_Generated {ctx['generated_at_utc']} by Stage 8 (Production Report Generator)._\n")

    md.append("## Executive Summary\n")
    md.append(
        f"- **Model**: {STAGE2_REPORT.get('checkpoint_compatibility', {}).get('architecture')}\n"
        f"- **Test samples**: {STAGE4_SUMMARY.get('n_samples')}\n"
        f"- **Classes**: {STAGE3_SUMMARY.get('num_classes')}\n"
        f"- **Macro AUROC**: {hm.get('auroc_macro')}  •  **Micro AUROC**: {hm.get('auroc_micro')}\n"
        f"- **Macro F1**: {hm.get('f1_macro')}  •  **Subset accuracy**: {hm.get('subset_accuracy')}\n"
        f"- **Macro ECE**: {STAGE6_SUMMARY.get('macro_ece')}  •  **Macro MCE**: {STAGE6_SUMMARY.get('macro_mce')}\n"
        f"- **Explainability samples**: {STAGE7_SUMMARY.get('n_samples_selected')} across "
        f"{len(STAGE7_SUMMARY.get('method_success_counts', {}))} attribution methods\n"
        f"- **Engineering health**: {ctx['engineering_overview']['overall_engineering_health']} "
        f"({ctx['engineering_overview']['totals']})\n"
        f"- **Deployment recommendation**: {ctx['deployment_readiness']['deployment_recommendation']} "
        f"(score {ctx['deployment_readiness']['score_0_100']}/100)\n"
        f"- **Publication readiness**: {ctx['publication_readiness']['overall_score_0_100']}/100\n"
    )

    md.append("## Model\n")
    md.append(
        f"- Backbone: `{STAGE2_REPORT.get('checkpoint_compatibility', {}).get('architecture')}`\n"
        f"- Checkpoint param counts match: {STAGE2_REPORT.get('checkpoint_compatibility', {}).get('param_counts_match')}\n"
        f"- Checkpoint metadata cross-check: {ctx['pseudo_stage2_checks_summary']}\n"
    )

    md.append("## Dataset\n")
    md.append(
        f"- Usable test samples: {STAGE3_SUMMARY.get('n_samples_usable')} "
        f"(of {STAGE3_SUMMARY.get('n_samples_total_manifest')} in the raw manifest)\n"
        f"- Image size: {STAGE3_SUMMARY.get('transform', {}).get('image_size')}\n"
        f"- Batch size: {STAGE3_SUMMARY.get('dataloader_config', {}).get('batch_size')}\n"
    )

    md.append("## Training Summary\n")
    md.append(
        "Training was performed upstream of this evaluation pipeline; only the checkpoint's "
        "self-reported metadata is available here (Stage 8 never re-runs or re-loads training). "
        f"Cross-check against best_model_metadata: {STAGE2_REPORT.get('metadata_cross_check', {})}\n"
    )

    md.append("## Inference Summary\n")
    md.append(
        f"- Device: {STAGE4_SUMMARY.get('device', {}).get('backend')} "
        f"({STAGE4_SUMMARY.get('device', {}).get('device_name')})\n"
        f"- Total inference time: {STAGE4_SUMMARY.get('total_inference_time_sec')} sec\n"
        f"- Throughput: {STAGE4_SUMMARY.get('samples_per_second_avg')} samples/sec (avg)\n"
        f"- Peak GPU memory: {STAGE4_SUMMARY.get('peak_gpu_allocated_mb')} MB\n"
        f"- All inference validation checks passed: {STAGE4_SUMMARY.get('all_validation_checks_passed')}\n"
    )

    md.append("## Metrics\n")
    md.append(_md_table(pd.DataFrame([hm])) + "\n")

    md.append("## Per-class Metrics\n")
    md.append(_md_table(ctx["per_class_df"]) + "\n")

    md.append("## Calibration\n")
    md.append(
        f"- Macro ECE: {STAGE6_SUMMARY.get('macro_ece')}, Macro MCE: {STAGE6_SUMMARY.get('macro_mce')}, "
        f"Macro Brier: {STAGE6_SUMMARY.get('macro_brier_score')}\n\n"
    )
    md.append(_md_table(ctx["calibration_df"]) + "\n")

    md.append("## Threshold Optimization\n")
    md.append(_md_table(ctx["thresholds_df"]) + "\n")

    md.append("## Interpretability\n")
    md.append(
        f"- Methods run: {list(STAGE7_SUMMARY.get('method_success_counts', {}).keys())}\n"
        f"- Samples explained: {STAGE7_SUMMARY.get('n_samples_selected')}\n"
        f"- PNGs written: {ctx['runtime_summary']['total_png_count']}\n"
        f"- Selection reasons: {STAGE7_SUMMARY.get('selection_reason_counts', {})}\n"
    )

    md.append("## Engineering Validation\n")
    md.append(_md_table(pd.DataFrame(
        [{"stage": s, **c} for s, c in ctx["engineering_overview"]["per_stage_counts"].items()]
    ).reset_index().rename(columns={"index": "stage_row"})) + "\n")

    md.append("## Deployment Readiness\n")
    md.append(f"- Score: {ctx['deployment_readiness']['score_0_100']}/100\n"
              f"- Recommendation: **{ctx['deployment_readiness']['deployment_recommendation']}**\n"
              f"- Blockers: {ctx['deployment_readiness']['blockers'] or 'none'}\n")

    md.append("## Publication Readiness\n")
    md.append(f"- Score: {ctx['publication_readiness']['overall_score_0_100']}/100\n")
    md.append("**Strengths:**\n" + "\n".join(f"- {s}" for s in ctx["publication_readiness"]["strengths"]) + "\n")
    md.append("**Weaknesses:**\n" + "\n".join(f"- {s}" for s in ctx["publication_readiness"]["weaknesses"]) + "\n")
    md.append("**Future work:**\n" + "\n".join(f"- {s}" for s in ctx["publication_readiness"]["future_work"]) + "\n")

    md.append("## Limitations\n")
    lim = ctx["limitations"]
    for section, val in lim.items():
        if section == "generated_at_utc":
            continue
        md.append(f"**{section.replace('_', ' ').title()}**\n")
        if isinstance(val, list):
            md.append("\n".join(f"- {v}" for v in val) + "\n")
        else:
            md.append(f"- {json.dumps(val, default=str)}\n")

    md.append("## Recommendations\n")
    rec = ctx["recommendations"]
    for section in ["engineering_improvements", "training_improvements", "deployment_improvements",
                     "research_improvements", "future_experiments"]:
        md.append(f"**{section.replace('_', ' ').title()}**\n")
        md.append("\n".join(f"- {v}" for v in rec[section]) + "\n")
    md.append("**Prioritized**\n")
    md.append("\n".join(f"- [{p['priority']}] {p['item']}" for p in rec["prioritized"]) + "\n")

    md.append("## Appendix: Artifact Inventory\n")
    md.append(_md_table(ctx["artifact_inventory_df"][["key", "stage", "kind", "required", "exists", "file_size_bytes"]],
                         max_rows=100) + "\n")

    return "\n".join(md)


def build_html_report(ctx: Dict[str, Any], markdown_source: str) -> str:
    def esc(x: Any) -> str:
        return html.escape(str(x))

    def df_to_html_table(df: pd.DataFrame, max_rows: int = 25) -> str:
        shown = df.head(max_rows)
        head = "".join(f"<th>{esc(c)}</th>" for c in shown.columns)
        body = "".join(
            "<tr>" + "".join(f"<td>{esc(v)}</td>" for v in row.tolist()) + "</tr>"
            for _, row in shown.iterrows()
        )
        more = f"<p><em>...and {len(df) - max_rows} more row(s); see tables/ for the full CSV.</em></p>" \
            if len(df) > max_rows else ""
        return f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>{more}"

    hm = STAGE5_SUMMARY.get("headline_metrics", {})
    sections = []

    def section(title: str, body_html: str) -> str:
        return (f"<details open><summary><h2>{esc(title)}</h2></summary>"
                f"<div class='section-body'>{body_html}</div></details>")

    sections.append(section("Executive Summary", f"""
        <ul>
          <li><b>Model</b>: {esc(STAGE2_REPORT.get('checkpoint_compatibility', {}).get('architecture'))}</li>
          <li><b>Test samples</b>: {esc(STAGE4_SUMMARY.get('n_samples'))}</li>
          <li><b>Classes</b>: {esc(STAGE3_SUMMARY.get('num_classes'))}</li>
          <li><b>Macro AUROC</b>: {esc(hm.get('auroc_macro'))} &nbsp; <b>Micro AUROC</b>: {esc(hm.get('auroc_micro'))}</li>
          <li><b>Macro F1</b>: {esc(hm.get('f1_macro'))} &nbsp; <b>Subset accuracy</b>: {esc(hm.get('subset_accuracy'))}</li>
          <li><b>Macro ECE / MCE</b>: {esc(STAGE6_SUMMARY.get('macro_ece'))} / {esc(STAGE6_SUMMARY.get('macro_mce'))}</li>
          <li><b>Explainability samples</b>: {esc(STAGE7_SUMMARY.get('n_samples_selected'))}</li>
          <li><b>Engineering health</b>: <span class="badge">{esc(ctx['engineering_overview']['overall_engineering_health'])}</span></li>
          <li><b>Deployment recommendation</b>: <span class="badge">{esc(ctx['deployment_readiness']['deployment_recommendation'])}</span>
              (score {esc(ctx['deployment_readiness']['score_0_100'])}/100)</li>
          <li><b>Publication readiness</b>: {esc(ctx['publication_readiness']['overall_score_0_100'])}/100</li>
        </ul>"""))

    sections.append(section("Metrics", df_to_html_table(pd.DataFrame([hm]))))
    sections.append(section("Per-class Metrics", df_to_html_table(ctx["per_class_df"], max_rows=20)))
    sections.append(section("Calibration", df_to_html_table(ctx["calibration_df"])))
    sections.append(section("Threshold Optimization", df_to_html_table(ctx["thresholds_df"])))
    sections.append(section("Interpretability", f"""
        <p>Methods run: {esc(list(STAGE7_SUMMARY.get('method_success_counts', {}).keys()))}</p>
        <p>Samples explained: {esc(STAGE7_SUMMARY.get('n_samples_selected'))}</p>
        <p>PNGs written: {esc(ctx['runtime_summary']['total_png_count'])}</p>
        <p>Selection reasons: {esc(STAGE7_SUMMARY.get('selection_reason_counts', {}))}</p>
        <p><em>Figures are referenced, not embedded inline — see figures/figure_manifest.json for local file paths.</em></p>
    """))
    sections.append(section("Engineering Validation", df_to_html_table(
        pd.DataFrame([{"stage": s, **c} for s, c in ctx["engineering_overview"]["per_stage_counts"].items()])
    )))
    sections.append(section("Deployment Readiness", f"""
        <p>Score: {esc(ctx['deployment_readiness']['score_0_100'])}/100</p>
        <p>Recommendation: <b>{esc(ctx['deployment_readiness']['deployment_recommendation'])}</b></p>
        <p>Blockers: {esc(ctx['deployment_readiness']['blockers'] or 'none')}</p>
    """))
    sections.append(section("Publication Readiness", f"""
        <p>Score: {esc(ctx['publication_readiness']['overall_score_0_100'])}/100</p>
        <p><b>Strengths</b></p><ul>{''.join(f'<li>{esc(s)}</li>' for s in ctx['publication_readiness']['strengths'])}</ul>
        <p><b>Weaknesses</b></p><ul>{''.join(f'<li>{esc(s)}</li>' for s in ctx['publication_readiness']['weaknesses'])}</ul>
        <p><b>Future work</b></p><ul>{''.join(f'<li>{esc(s)}</li>' for s in ctx['publication_readiness']['future_work'])}</ul>
    """))
    lim_html = "".join(
        f"<p><b>{esc(k.replace('_',' ').title())}</b></p>" +
        ("<ul>" + "".join(f"<li>{esc(v)}</li>" for v in val) + "</ul>" if isinstance(val, list)
         else f"<pre>{esc(json.dumps(val, indent=2, default=str))}</pre>")
        for k, val in ctx["limitations"].items() if k != "generated_at_utc"
    )
    sections.append(section("Limitations", lim_html))
    rec = ctx["recommendations"]
    rec_html = "".join(
        f"<p><b>{esc(s.replace('_',' ').title())}</b></p><ul>{''.join(f'<li>{esc(v)}</li>' for v in rec[s])}</ul>"
        for s in ["engineering_improvements", "training_improvements", "deployment_improvements",
                   "research_improvements", "future_experiments"]
    ) + "<p><b>Prioritized</b></p><ul>" + "".join(
        f"<li>[{esc(p['priority'])}] {esc(p['item'])}</li>" for p in rec["prioritized"]) + "</ul>"
    sections.append(section("Recommendations", rec_html))
    sections.append(section("Appendix: Artifact Inventory", df_to_html_table(
        ctx["artifact_inventory_df"][["key", "stage", "kind", "required", "exists", "file_size_bytes"]], max_rows=100)))

    css = """
    body{font-family:-apple-system,Segoe UI,Roboto,Helvetica,Arial,sans-serif;max-width:1100px;margin:2rem auto;padding:0 1rem;color:#1a1a1a;}
    h1{border-bottom:3px solid #2b6cb0;padding-bottom:.4rem;}
    h2{color:#2b6cb0;}
    details{border:1px solid #ddd;border-radius:8px;margin:1rem 0;padding:.5rem 1rem;background:#fafafa;}
    summary{cursor:pointer;}
    table{border-collapse:collapse;width:100%;margin:.5rem 0;font-size:.9rem;}
    th,td{border:1px solid #ddd;padding:.35rem .5rem;text-align:left;}
    th{background:#eef3fa;}
    .badge{display:inline-block;background:#2b6cb0;color:white;border-radius:4px;padding:.1rem .5rem;font-weight:600;}
    """
    return f"""<!DOCTYPE html><html><head><meta charset="utf-8">
    <title>VisionServeAI Sprint04 Evaluation Report</title><style>{css}</style></head>
    <body>
    <h1>VisionServeAI Sprint04 — Chest X-Ray Evaluation Report</h1>
    <p><em>Generated {esc(ctx['generated_at_utc'])} by Stage 8 (Production Report Generator).</em></p>
    {''.join(sections)}
    </body></html>"""


def build_report_assets(ctx: Dict[str, Any]) -> Dict[str, Any]:
    """PDF-ready structured content: title, sections, tables, figures, captions.
    Stage 8 never generates a PDF itself; this is the input a later PDF-writer
    stage would consume."""
    return {
        "generated_at_utc": ctx["generated_at_utc"],
        "title": "VisionServeAI Sprint04 — Chest X-Ray Evaluation Report",
        "sections": [
            {"heading": "Executive Summary", "body_ref": "summary/evaluation_summary.json"},
            {"heading": "Model", "body_ref": "stage02_artifact_loader/artifact_validation.json"},
            {"heading": "Dataset", "body_ref": "stage03_dataset_builder/dataset_summary.json"},
            {"heading": "Inference Summary", "body_ref": "stage04_inference_engine/inference_summary.json"},
            {"heading": "Metrics", "body_ref": "stage05_metrics_engine/evaluation_metrics.json"},
            {"heading": "Calibration", "body_ref": "stage06_threshold_calibration_engine/stage06_summary.json"},
            {"heading": "Interpretability", "body_ref": "stage07_interpretability_engine/summary/interpretability_summary.json"},
            {"heading": "Engineering Validation", "body_ref": "stage08_report_generator/summary/engineering_overview.json"},
            {"heading": "Deployment Readiness", "body_ref": "stage08_report_generator/summary/deployment_readiness.json"},
            {"heading": "Publication Readiness", "body_ref": "stage08_report_generator/summary/publication_readiness.json"},
            {"heading": "Limitations", "body_ref": "stage08_report_generator/summary/limitations.json"},
            {"heading": "Recommendations", "body_ref": "stage08_report_generator/summary/recommendations.json"},
        ],
        "tables": [
            {"name": "overall_metrics", "path": "stage08_report_generator/tables/overall_metrics.csv"},
            {"name": "deployment_summary", "path": "stage08_report_generator/tables/deployment_summary.csv"},
            {"name": "publication_summary", "path": "stage08_report_generator/tables/publication_summary.csv"},
            {"name": "artifact_inventory", "path": "stage08_report_generator/tables/artifact_inventory.csv"},
            {"name": "engineering_checks", "path": "stage08_report_generator/tables/engineering_checks.csv"},
            {"name": "runtime_summary", "path": "stage08_report_generator/tables/runtime_summary.csv"},
        ],
        "figures": ctx["figure_manifest"]["figures"],
        "captions": {f["name"]: f["caption"] for f in ctx["figure_manifest"]["figures"]},
    }


# =============================================================================
# 8.11  Run Stage 8 (evaluate.py-equivalent orchestration)
# =============================================================================
def run_stage8() -> Dict[str, Any]:
    with StageTimer("stage08_report_generator", logger) as timer:
        vl = ValidationLedger(logger)
        log_resource_usage(logger, "stage08_start")

        # --- reopen every required Stage 1-7 artifact, fail loudly on any
        #     missing/corrupted file -------------------------------------------
        loaded: Dict[str, Any] = {
            "prediction_metadata": load_json_or_fail(next(r for r in ARTIFACT_REGISTRY if r.key == "prediction_metadata")),
            "evaluation_metrics": load_json_or_fail(next(r for r in ARTIFACT_REGISTRY if r.key == "evaluation_metrics")),
            "macro_metrics": load_json_or_fail(next(r for r in ARTIFACT_REGISTRY if r.key == "macro_metrics")),
            "evaluation_summary": STAGE5_SUMMARY,
            "per_class_metrics_df": load_csv_or_fail(next(r for r in ARTIFACT_REGISTRY if r.key == "per_class_metrics_csv")),
            "optimal_thresholds_df": load_csv_or_fail(next(r for r in ARTIFACT_REGISTRY if r.key == "optimal_thresholds_csv")),
            "calibration_summary_df": load_csv_or_fail(next(r for r in ARTIFACT_REGISTRY if r.key == "calibration_summary_csv")),
            "deployment_recommendations_df": load_csv_or_fail(next(r for r in ARTIFACT_REGISTRY if r.key == "deployment_recommendations_csv")),
            "error_gallery_df": load_csv_or_fail(next(r for r in ARTIFACT_REGISTRY if r.key == "error_gallery_csv")),
            "sample_predictions_df": load_csv_or_fail(next(r for r in ARTIFACT_REGISTRY if r.key == "sample_predictions_csv")),
            "class_confusion_matrix_df": load_csv_or_fail(next(r for r in ARTIFACT_REGISTRY if r.key == "class_confusion_matrix_csv")),
        }
        logger.info(f"artifacts_loaded {list(loaded.keys())}")
        log_resource_usage(logger, "stage08_post_load")

        # --- validations ------------------------------------------------------
        validate_cross_stage_consistency(vl, logger, loaded)

        # --- artifact inventory -------------------------------------------------
        artifact_inventory, artifact_inventory_df = build_artifact_inventory(logger)
        vl.check("artifact_inventory_complete",
                  artifact_inventory["n_present"] >= sum(1 for r in ARTIFACT_REGISTRY if r.required),
                  detail=f"n_present={artifact_inventory['n_present']} n_required="
                         f"{sum(1 for r in ARTIFACT_REGISTRY if r.required)}")

        # --- engineering overview, runtime summary ------------------------------
        engineering_overview = build_engineering_overview(logger)
        runtime_summary = build_runtime_summary(logger)

        # --- readiness scoring ----------------------------------------------------
        deployment_readiness = build_deployment_readiness(logger, loaded["deployment_recommendations_df"])
        publication_readiness = build_publication_readiness(
            logger, loaded["per_class_metrics_df"], loaded["calibration_summary_df"])

        # --- limitations & recommendations -----------------------------------------
        limitations = build_limitations(loaded["per_class_metrics_df"], loaded["calibration_summary_df"],
                                         loaded["optimal_thresholds_df"])
        recommendations = build_recommendations(loaded["deployment_recommendations_df"], loaded["calibration_summary_df"])

        # --- figure references (never regenerated) ---------------------------------
        figure_manifest = build_figure_manifest(logger, loaded["error_gallery_df"])

        generated_at = datetime.now(timezone.utc).isoformat()

        # --- master report -----------------------------------------------------------
        evaluation_report = {
            "generated_at_utc": generated_at,
            "metadata": {"notebook": "Sprint04_Evaluation.ipynb", "stage": "Stage 8 — Production Report Generator",
                         "run_id": STAGE1_SUMMARY.get("run_id")},
            "model": {
                "architecture": STAGE2_REPORT.get("checkpoint_compatibility", {}).get("architecture"),
                "checkpoint_compatibility": STAGE2_REPORT.get("checkpoint_compatibility", {}),
                "metadata_cross_check": STAGE2_REPORT.get("metadata_cross_check", {}),
            },
            "dataset": STAGE3_SUMMARY,
            "inference": {k: v for k, v in STAGE4_SUMMARY.items() if k != "batch_perf_trace"},
            "metrics": {
                "evaluation_summary": STAGE5_SUMMARY,
                "evaluation_metrics": loaded["evaluation_metrics"],
                "per_class_metrics": loaded["per_class_metrics_df"].to_dict(orient="records"),
            },
            "calibration": {
                "stage06_summary": STAGE6_SUMMARY,
                "calibration_summary": loaded["calibration_summary_df"].to_dict(orient="records"),
            },
            "thresholds": loaded["optimal_thresholds_df"].to_dict(orient="records"),
            "explainability": {
                "interpretability_summary": STAGE7_SUMMARY,
                "class_confusion_matrix": loaded["class_confusion_matrix_df"].to_dict(orient="records"),
            },
            "runtime": runtime_summary,
            "engineering_checks": engineering_overview,
            "recommendations": recommendations,
            "artifact_inventory": artifact_inventory,
            "warnings": timer.warnings + figure_manifest["warnings"],
            "limitations": limitations,
            "deployment_readiness": deployment_readiness,
            "publication_readiness": publication_readiness,
        }

        # --- executive summary (lightweight) ------------------------------------------
        hm = STAGE5_SUMMARY.get("headline_metrics", {})
        evaluation_summary_lightweight = {
            "generated_at_utc": generated_at,
            "auroc_macro": hm.get("auroc_macro"), "auroc_micro": hm.get("auroc_micro"),
            "f1_macro": hm.get("f1_macro"), "accuracy_subset": hm.get("subset_accuracy"),
            "calibration": {"macro_ece": STAGE6_SUMMARY.get("macro_ece"), "macro_mce": STAGE6_SUMMARY.get("macro_mce")},
            "interpretability": {"n_samples_selected": STAGE7_SUMMARY.get("n_samples_selected"),
                                  "n_methods": len(STAGE7_SUMMARY.get("method_success_counts", {}))},
            "overall_engineering_status": engineering_overview["overall_engineering_health"],
            "deployment_recommendation": deployment_readiness["deployment_recommendation"],
        }

        ctx = {
            "generated_at_utc": generated_at,
            "engineering_overview": engineering_overview,
            "runtime_summary": runtime_summary,
            "deployment_readiness": deployment_readiness,
            "publication_readiness": publication_readiness,
            "limitations": limitations,
            "recommendations": recommendations,
            "per_class_df": loaded["per_class_metrics_df"],
            "calibration_df": loaded["calibration_summary_df"],
            "thresholds_df": loaded["optimal_thresholds_df"],
            "artifact_inventory_df": artifact_inventory_df,
            "figure_manifest": figure_manifest,
            "pseudo_stage2_checks_summary": STAGE2_REPORT.get("metadata_cross_check", {}),
        }
        figure_manifest_written = build_figure_manifest(logger, loaded["error_gallery_df"])  # deterministic re-use
        ctx["figure_manifest"] = figure_manifest_written
        report_assets = build_report_assets(ctx)

        markdown_report = build_markdown_report(ctx)
        html_report = build_html_report(ctx, markdown_report)

        log_resource_usage(logger, "stage08_pre_write")

        # --- write everything ---------------------------------------------------------
        paths: Dict[str, str] = {}

        p = SUBDIRS["artifacts"] / "evaluation_report.json"; write_json(evaluation_report, p, logger); paths["evaluation_report"] = str(p)
        p = SUBDIRS["artifacts"] / "report_assets.json"; write_json(report_assets, p, logger); paths["report_assets"] = str(p)

        p = SUBDIRS["summary"] / "evaluation_summary.json"; write_json(evaluation_summary_lightweight, p, logger); paths["evaluation_summary"] = str(p)
        p = SUBDIRS["summary"] / "artifact_inventory.json"; write_json(artifact_inventory, p, logger); paths["artifact_inventory"] = str(p)
        p = SUBDIRS["summary"] / "engineering_overview.json"; write_json(engineering_overview, p, logger); paths["engineering_overview"] = str(p)
        p = SUBDIRS["summary"] / "runtime_summary.json"; write_json(runtime_summary, p, logger); paths["runtime_summary"] = str(p)
        p = SUBDIRS["summary"] / "deployment_readiness.json"; write_json(deployment_readiness, p, logger); paths["deployment_readiness"] = str(p)
        p = SUBDIRS["summary"] / "publication_readiness.json"; write_json(publication_readiness, p, logger); paths["publication_readiness"] = str(p)
        p = SUBDIRS["summary"] / "limitations.json"; write_json(limitations, p, logger); paths["limitations"] = str(p)
        p = SUBDIRS["summary"] / "recommendations.json"; write_json(recommendations, p, logger); paths["recommendations"] = str(p)

        p = SUBDIRS["reports"] / "evaluation_report.md"; write_text(markdown_report, p, logger); paths["evaluation_report_md"] = str(p)
        p = SUBDIRS["reports"] / "evaluation_report.html"; write_text(html_report, p, logger); paths["evaluation_report_html"] = str(p)

        overall_metrics_df = pd.DataFrame([hm])
        p = SUBDIRS["tables"] / "overall_metrics.csv"; write_csv(overall_metrics_df, p, logger); paths["overall_metrics_csv"] = str(p)
        p = SUBDIRS["tables"] / "deployment_summary.csv"; write_csv(loaded["deployment_recommendations_df"], p, logger); paths["deployment_summary_csv"] = str(p)
        publication_summary_df = pd.DataFrame([
            {"dimension": k, **{kk: vv for kk, vv in v.items()}} for k, v in publication_readiness["dimensions"].items()
        ])
        p = SUBDIRS["tables"] / "publication_summary.csv"; write_csv(publication_summary_df, p, logger); paths["publication_summary_csv"] = str(p)
        p = SUBDIRS["tables"] / "artifact_inventory.csv"; write_csv(artifact_inventory_df, p, logger); paths["artifact_inventory_csv"] = str(p)
        engineering_checks_df = pd.DataFrame(engineering_overview["checks"])
        p = SUBDIRS["tables"] / "engineering_checks.csv"; write_csv(engineering_checks_df, p, logger); paths["engineering_checks_csv"] = str(p)
        runtime_summary_flat_df = pd.json_normalize(runtime_summary, sep=".")
        p = SUBDIRS["tables"] / "runtime_summary.csv"; write_csv(runtime_summary_flat_df, p, logger); paths["runtime_summary_csv"] = str(p)

        p = SUBDIRS["figures"] / "figure_manifest.json"; write_json(figure_manifest_written, p, logger); paths["figure_manifest"] = str(p)

        log_resource_usage(logger, "stage08_end")

        # --- final engineering validation for Stage 8 itself -----------------------------
        for key, path_str in paths.items():
            vl.check(f"artifact_written_{key}", Path(path_str).is_file(), detail=path_str)
        vl.check("engineering_overview_no_failures",
                  engineering_overview["totals"]["FAIL"] == 0,
                  detail=str(engineering_overview["totals"]), hard=False)

        engineering_validation = {
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "all_checks_passed": vl.all_passed(),
            "n_checks": len(vl.results),
            "n_passed": vl.n_passed(),
            "n_failed": len(vl.results) - vl.n_passed(),
            "n_hard_failed": vl.n_hard_failed(),
            "n_soft_failed": vl.n_soft_failed(),
            "checks": vl.results,
        }
        engval_path = SUBDIRS["summary"] / "stage08_engineering_validation.json"
        write_json(engineering_validation, engval_path, logger)
        paths["engineering_validation"] = str(engval_path)

        if not engineering_validation["all_checks_passed"]:
            raise RuntimeError(
                f"Stage 8 engineering validation FAILED: "
                f"{engineering_validation['n_hard_failed']}/{engineering_validation['n_checks']} "
                f"hard (Stage-8-owned) checks failed. "
                f"{engineering_validation['n_soft_failed']} informational finding(s) were also "
                f"recorded but do not gate Stage 8 — see stage08_engineering_validation.json."
            )
        if engineering_validation["n_soft_failed"] > 0:
            logger.warning(
                f"stage08_soft_validation_findings n_soft_failed={engineering_validation['n_soft_failed']} "
                f"— historical engineering findings inherited from earlier stages are present "
                f"and fully recorded in the reports, but do not fail Stage 8 itself."
            )

        result = {
            "generated_at_utc": generated_at,
            "artifacts_consumed": len(ARTIFACT_REGISTRY),
            "artifacts_generated": len(paths),
            "engineering_checks": engineering_validation["n_checks"],
            "n_passed": engineering_validation["n_passed"],
            "n_failed": engineering_validation["n_failed"],
            "n_warnings": len(evaluation_report["warnings"]),
            "deployment_readiness": deployment_readiness["deployment_recommendation"],
            "publication_readiness_score": publication_readiness["overall_score_0_100"],
            "overall_status": "OK" if engineering_validation["all_checks_passed"] else "FAILED",
            "warnings": evaluation_report["warnings"],
            "artifacts": paths,
        }
        result["_artifact_path"] = paths["evaluation_report"]
        return result


STAGE8_SUMMARY = run_stage8()

print("========================================================")
print("STAGE 8 — FINAL EVALUATION REPORT")
print("========================================================")
print(f"Artifacts consumed  : {STAGE8_SUMMARY['artifacts_consumed']}")
print(f"Artifacts generated : {STAGE8_SUMMARY['artifacts_generated']}")
print(f"Engineering checks  : {STAGE8_SUMMARY['engineering_checks']} run "
      f"(passed={STAGE8_SUMMARY['n_passed']}, failed={STAGE8_SUMMARY['n_failed']})")
print(f"Warnings            : {STAGE8_SUMMARY['n_warnings']}")
print(f"Deployment readiness: {STAGE8_SUMMARY['deployment_readiness']}")
print(f"Publication readiness: {STAGE8_SUMMARY['publication_readiness_score']}/100")
print(f"Overall status      : Stage 8 : {STAGE8_SUMMARY['overall_status']}")
print("========================================================")

2026-07-05 07:04:55 | INFO    | sprint04_eval.stage08_report_generator | START  stage=stage08_report_generator
2026-07-05 07:04:55 | INFO    | sprint04_eval.stage08_report_generator | resource_usage tag=stage08_start {'gpu_allocated_mb': 17.25, 'gpu_reserved_mb': 8546.0, 'cpu_maxrss_mb': 2379.1}
2026-07-05 07:04:55 | INFO    | sprint04_eval.stage08_report_generator | artifacts_loaded ['prediction_metadata', 'evaluation_metrics', 'macro_metrics', 'evaluation_summary', 'per_class_metrics_df', 'optimal_thresholds_df', 'calibration_summary_df', 'deployment_recommendations_df', 'error_gallery_df', 'sample_predictions_df', 'class_confusion_matrix_df']
2026-07-05 07:04:55 | INFO    | sprint04_eval.stage08_report_generator | resource_usage tag=stage08_post_load {'gpu_allocated_mb': 17.25, 'gpu_reserved_mb': 8546.0, 'cpu_maxrss_mb': 2379.1}
2026-07-05 07:04:55 | INFO    | sprint04_eval.stage08_report_generator | validation check=class_order_consistent_stage3_4_5 status=PASS detail=stage3=14 sta